<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cpp.pt/cap03/cap03_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

# 3 Operações Espaciais: Intensidade, Histograma e Filtragem

Este capítulo aprofunda o processamento de imagens no domínio espacial, partindo da manipulação direta de pixels e histogramas para o realce de contraste, até a aplicação de filtros locais por convolução para suavização, redução de ruído e detecção de bordas. O objetivo é desenvolver a intuição matemática e computacional que sustenta grande parte dos algoritmos modernos de Visão Computacional.

## 3.1 Objetivos

Ao final deste capítulo, você será capaz de:

* **Manipular intensidade e pixels:** Executar operações aritméticas saturadas (`mm::addm`, `mm::subm`) e lógicas bit a bit (`mm::band`, `mm::bor`, `mm::bnot`) para combinação e seleção de regiões de interesse (ROI), e aplicar *alpha blending* (`mm::blend`) para fusão ponderada de imagens;
* **Processar histogramas:** Interpretar o histograma como diagnóstico tonal e aplicar equalização global via CDF (`mm::equalize`); na trilha Python, também a equalização adaptativa (CLAHE) e a especificação de histograma para transferência de perfil tonal entre imagens;
* **Compreender fundamentos espaciais:** Entender vizinhança, *padding* de borda (`mm::pad`) e a diferença entre correlação cruzada (`mm::conv`) e convolução — incluindo por que *kernels* assimétricos como o de Sobel produzem resultados distintos nas duas operações;
* **Aplicar filtragem de suavização:** Usar o filtro de média (`mm::blur`, ou `mm::conv` com *kernel* uniforme) e o filtro Gaussiano (`mm::gaussian`) para redução de ruído, compreendendo a vantagem da ponderação radial e da separabilidade Gaussiana;
* **Aplicar filtragem de realce:** Usar o Laplaciano $w_4$ e $w_8$ (`mm::laplacian`) para realce isotrópico de bordas, o operador de Sobel (`mm::sobel`) para a magnitude do gradiente — e, na trilha Python, a decomposição direcional $G_x$, $G_y$ e ângulo —, e o *Unsharp Masking* (`mm::usm`) para amplificação de alta frequência controlada pelo parâmetro $k$;
* **Utilizar filtros de ordem:** Aplicar o filtro da mediana (`mm::median`) para remoção de ruído sal e pimenta, compreendendo por que sua natureza não linear e a robustez a *outliers* o tornam superior aos filtros lineares nesse cenário;
* **Resolver problemas práticos:** Encadear técnicas em *pipelines* de pré-processamento (equalização → Gaussiano → Canny; com CLAHE no lugar da equalização na trilha Python) e usar as funções da `morph` (`mm::conv`, `mm::histImg`, `mm::equalize`, `mm::drawImgKernel`) para análise e visualização didática de cada etapa.

## 3.2 Operações em Nível de Intensidade

O nível mais elementar de processamento de imagens atua diretamente sobre os valores dos pixels, sem considerar vizinhança. Essas operações — chamadas de **transformações de ponto** (*point operations*) — são as mais rápidas computacionalmente e formam a base para técnicas mais complexas.

Formalmente, uma transformação de ponto pode ser descrita como:

<a id="eq-03-ponto"></a>
$$
g(x,y) = T[f(x,y)] \tag{3.1}
$$


onde $f(x,y)$ é a imagem de entrada, $g(x,y)$ é a saída e $T$ é uma função aplicada a cada pixel individualmente.

### 3.2.1 Preparando o Ambiente Prático

O bloco a seguir carrega a biblioteca `morph` do repositório (o módulo `morph.py` e, na trilha C++, também a `morph.hpp` usada no `#include` das células compiladas).

In [1]:
import os, urllib.request

os.makedirs("tmp/state", exist_ok=True)  # artefatos de build da trilha C++ (.cpp, binário, PNGs)

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

# O kernel é Python mesmo na trilha C++: `mm` (morph.py) é usado pelos
# simuladores, pela exibição das figuras que o binário C++ gera e pelo
# estado mm::Image entre células. cpp=True baixa também a trilha compilada
# (morph.hpp + stb_image*.h), usada no #include das células %%writefile *.cpp.
import config
config.setup(cpp=True)
from morph import mm
import numpy as np

✅ Ambiente pronto. Morph: 1.1.9 | OpenCV: 5.0.0


Como objeto de estudo ao longo deste capítulo, utilizaremos as imagens de vida selvagem apresentadas nas [Figura 3.1](#fig-03-mandrill) e [Figura 3.3](#fig-03-leopardo). A partir delas, exploraremos operações espaciais sobre intensidade, histogramas e filtragem, analisando seus efeitos no realce, na suavização, na redução de ruído e na detecção de bordas, de modo a compreender os fundamentos matemáticos e computacionais do PDI.

In [2]:
%%writefile tmp/fig_03_mandrill.cpp
#define MM_OUT "tmp/fig_03_mandrill.png"
// Compile: g++ -std=c++17 -o programa programa.cpp -I. -lstdc++fs

#include "morph.hpp"
#include <iostream>
#include <string>
#include <filesystem>

int main() {
    // | label: fig-03-mandrill
    // | fig-cap: "*Mandrill* (*Mandrillus sphinx*) fotografado em ambiente natural na África do Sul. Crédito: Carlos Guilherme Rodrigues (CC BY-SA 3.0)."
    // | echo: true

    std::string base    = "https://upload.wikimedia.org/wikipedia/commons";
    std::string arquivo = "Carlos_Guilherme_Rodrigues_%2876515283%29.jpeg";
    std::string url     = base + "/9/9b/" + arquivo;
    std::string caminho = "imagens/mandrill-exif.jpg";

    if (!std::filesystem::exists(caminho)) {
        std::filesystem::create_directories("imagens");
        mm::write(mm::read(url), caminho);
    }

    mm::Image img_color = mm::read(caminho);
    mm::Image img_gray  = mm::gray(img_color);

    mm::show(img_color, MM_OUT);

    
// [pdi:state-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp/state");
mm::write(img_gray, "tmp/state/img_gray_8.png");
// [pdi:state-io:end]
return 0;
}

Overwriting tmp/fig_03_mandrill.cpp


In [3]:
!g++ -I. -std=c++17 tmp/fig_03_mandrill.cpp -o tmp/fig_03_mandrill \
  && ./tmp/fig_03_mandrill \
  && test -f "tmp/fig_03_mandrill.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_mandrill.png"

In [4]:
try:
    mm.show(mm.read("tmp/fig_03_mandrill.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_mandrill.png (ver a versao Python)")

<Figure size 480x450 with 1 Axes>

**Figura 3.1:** *Mandrill* (*Mandrillus sphinx*) fotografado em ambiente natural na África do Sul. Crédito: Carlos Guilherme Rodrigues (CC BY-SA 3.0).


### 3.2.2 Operações Aritméticas

Operações aritméticas entre imagens são amplamente usadas em PDI para combinar, comparar ou realçar informações. A **subtração de imagens** é especialmente poderosa para detectar diferenças entre dois quadros — por exemplo, na remoção de fundo estático em câmeras de vigilância:

<a id="eq-03-subtracao"></a>
$$
g(x,y) = f_1(x,y) - f_2(x,y) \tag{3.2}
$$


A **adição saturada** limita o resultado ao intervalo $[0, 255]$: valores acima de 255 são fixados em 255, evitando o *overflow* silencioso do tipo `uint8` (ex.: $200 + 100 = 44$ em vez de 300). A **subtração saturada** aplica o mesmo princípio pelo lado inferior: valores negativos são fixados em 0.

> ### ⚠️ Saturação e *overflow*
>
> Operações aritméticas em `uint8` sofrem *overflow* silencioso: $200 + 100 = 44$ (não 300). `mm::addm` e `mm::subm` fazem a **saturação automática**, fixando o resultado em $[0, 255]$. O *blending* usa pesos fracionários: `mm::blend` opera internamente em ponto flutuante e só então arredonda e satura para `uint8`.

A [Figura 3.2](#fig-03-aritmetica) demonstra adição de uma constante (clareamento) e subtração de uma constante (escurecimento com saturação em 0).

In [5]:
%%writefile tmp/fig_03_aritmetica.cpp
#define MM_OUT "tmp/fig_03_aritmetica.png"
#include <iostream>
#include <vector>
#include <string>
#include "morph.hpp"
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_8.png");
// [pdi:state-io:end]

    // Variáveis fornecidas automaticamente:
    // img_gray (mm::Image)

    int fundo = 60;

    mm::Image img_add = mm::addm(img_gray, fundo);
    mm::Image img_sub = mm::subm(img_gray, fundo);

    mm::show(
        std::vector<mm::Image>{img_gray, img_add, img_sub},
        MM_OUT,
        std::vector<std::string>{"Original", "addm (+60)", "subm (−60)"},
        3
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray, "tmp/fig_03_aritmetica_0.png");
mm::write(img_add, "tmp/fig_03_aritmetica_1.png");
mm::write(img_sub, "tmp/fig_03_aritmetica_2.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_03_aritmetica.cpp


In [6]:
!g++ -I. -std=c++17 tmp/fig_03_aritmetica.cpp -o tmp/fig_03_aritmetica \
  && ./tmp/fig_03_aritmetica \
  && test -f "tmp/fig_03_aritmetica.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_aritmetica.png"

[1] Original
[2] addm (+60)
[3] subm (−60)


In [7]:
try:
    mm.show(
        [
            mm.read("tmp/fig_03_aritmetica_0.png"),
            mm.read("tmp/fig_03_aritmetica_1.png"),
            mm.read("tmp/fig_03_aritmetica_2.png"),
        ],
        titles=[
            'Original',
            'addm (+60)',
            'subm (−60)',
        ],
        cols=3,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_aritmetica_0.png (ver a versao Python)")

<Figure size 2250x750 with 3 Axes>

**Figura 3.2:** Operações aritméticas saturadas: adição de constante (clareamento) e subtração de constante (escurecimento com saturação em 0).


### 3.2.3 Mistura Ponderada (*Alpha Blending*)

A **mistura ponderada** (*alpha blending*) combina duas imagens utilizando pesos complementares $\alpha$ e $(1-\alpha)$:

<a id="eq-03-blend"></a>
$$
g(x,y) = \alpha\,f_1(x,y) + (1-\alpha)\,f_2(x,y), \quad \alpha \in [0,1] \tag{3.3}
$$


Quando $\alpha = 1$, obtém-se apenas a imagem $f_1$; quando $\alpha = 0$, apenas $f_2$. Valores intermediários produzem uma transição suave entre ambas, sendo amplamente utilizados em composição de imagens, sobreposição de camadas, marcas d'água e efeitos de fusão visual.

Para que a combinação produza um resultado coerente, é necessário alinhar previamente as regiões de interesse. Na [Figura 3.4](#fig-03-blend), recorta-se o rosto do leopardo com `mm::crop(img_leop_gray, 250, H-300, 100, W-200)` e a região facial do mandril com `mm::crop(img_gray, 100, 400, 380, 530)`, de modo que olhos e estrutura facial fiquem aproximadamente alinhados. O recorte do leopardo é então redimensionado (`mm::resize`) para as dimensões do mandril antes da mistura.

`mm::blend` faz a operação em ponto flutuante — evitando *overflow* nas contas com pesos fracionários — e só então arredonda e satura o resultado para `uint8`.

In [8]:
%%writefile tmp/fig_03_leopardo.cpp
#define MM_OUT "tmp/fig_03_leopardo.png"
#include "morph.hpp"
#include <iostream>
#include <filesystem>

int main() {
    //| label: fig-03-leopardo
    //| fig-cap: "Retrato de um leopardo (*Panthera pardus*) em ambiente natural. Crédito: C. Brück (CC BY-SA 4.0)."
    //| echo: true

    const std::string base    = "https://upload.wikimedia.org/wikipedia/commons";
    const std::string arquivo = "Leopard_%28Panthera_pardus%29_portrait.jpg";
    const std::string url     = base + "/9/92/" + arquivo;

    mm::Image img_leop      = mm::read(url);
    mm::Image img_leop_gray = mm::gray(img_leop);

    mm::show(img_leop, MM_OUT);

    
// [pdi:state-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp/state");
mm::write(img_leop, "tmp/state/img_leop_12.png");
mm::write(img_leop_gray, "tmp/state/img_leop_gray_12.png");
// [pdi:state-io:end]
return 0;
}

Overwriting tmp/fig_03_leopardo.cpp


In [9]:
!g++ -I. -std=c++17 tmp/fig_03_leopardo.cpp -o tmp/fig_03_leopardo \
  && ./tmp/fig_03_leopardo \
  && test -f "tmp/fig_03_leopardo.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_leopardo.png"

In [10]:
try:
    mm.show(mm.read("tmp/fig_03_leopardo.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_leopardo.png (ver a versao Python)")

<Figure size 621x662 with 1 Axes>

**Figura 3.3:** Retrato de um leopardo (*Panthera pardus*) em ambiente natural. Crédito: C. Brück (CC BY-SA 4.0).


In [11]:
%%writefile tmp/fig_03_blend.cpp
#define MM_OUT "tmp/fig_03_blend.png"
//| label: fig-03-blend
//| fig-cap: "*Alpha blending* entre recortes alinhados de mandrill e do leopardo (@fig-03-leopardo) para diferentes valores de α. Em α=1 vê-se apenas mandrill; em α=0, apenas o leopardo; valores intermediários fundem os olhares das duas imagens proporcionalmente."
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_8.png");
mm::Image img_leop_gray = mm::_read_state("tmp/state/img_leop_gray_12.png");
// [pdi:state-io:end]

    // recortes alinhados: rosto do leopardo e região facial do mandril
    mm::Image leo      = mm::crop(img_leop_gray, 250, img_leop_gray.h - 300, 100, img_leop_gray.w - 200);
    mm::Image mandrill = mm::crop(img_gray, 100, 400, 380, 530);
    mm::Image leo_r    = mm::resize(leo, mandrill.w, mandrill.h, "bilinear");

    mm::show(
        {mm::blend(mandrill, leo_r, 1.0), mm::blend(mandrill, leo_r, 0.8),
         mm::blend(mandrill, leo_r, 0.6), mm::blend(mandrill, leo_r, 0.4),
         mm::blend(mandrill, leo_r, 0.2), mm::blend(mandrill, leo_r, 0.0)},
        MM_OUT,
        {"α=1.0", "α=0.8", "α=0.6", "α=0.4", "α=0.2", "α=0.0"},
        6
    );

    return 0;
}

Overwriting tmp/fig_03_blend.cpp


In [12]:
!g++ -I. -std=c++17 tmp/fig_03_blend.cpp -o tmp/fig_03_blend \
  && ./tmp/fig_03_blend \
  && test -f "tmp/fig_03_blend.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_blend.png"

[1] α=1.0
[2] α=0.8
[3] α=0.6
[4] α=0.4
[5] α=0.2
[6] α=0.0


In [13]:
try:
    mm.show(mm.read("tmp/fig_03_blend.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_blend.png (ver a versao Python)")

<Figure size 1564x450 with 1 Axes>

**Figura 3.4:** *Alpha blending* entre recortes alinhados de mandrill e do leopardo (@fig-03-leopardo) para diferentes valores de α. Em α=1 vê-se apenas mandrill; em α=0, apenas o leopardo; valores intermediários fundem os olhares das duas imagens proporcionalmente.


### 3.2.4 Operações Lógicas e Máscaras Bit a Bit

As operações lógicas bit a bit (AND, OR e NOT) atuam diretamente sobre os bits de cada pixel e são a base para criação e aplicação de **máscaras** (*masks*) — imagens binárias com apenas 0 (preto) e 255 (branco) usadas para isolar **Regiões de Interesse** (**ROI**).

O comportamento de cada operação decorre da representação binária do 255 (`11111111`) e do 0 (`00000000`):

- **AND** com a máscara: onde $m = 255$, os bits originais são preservados; onde $m = 0$, o pixel é zerado. Resultado: recorte da ROI.
<a id="eq-03-mascara"></a>
$$
g(x,y) = f(x,y) \;\text{AND}\; m(x,y) \tag{3.4}
$$

- **OR** com a máscara: onde $m = 255$, o pixel é forçado a branco; onde $m = 0$, o valor original é mantido. Resultado: iluminação da ROI.
- **NOT** (sem máscara): inverte todos os bits ($g = 255 - f$), produzindo o negativo fotográfico da imagem.

A [Figura 3.5](#fig-03-logica) ilustra as três operações aplicadas à imagem do mandrill com uma máscara circular.

In [14]:
%%writefile tmp/fig_03_logica.cpp
#define MM_OUT "tmp/fig_03_logica.png"
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_8.png");
// [pdi:state-io:end]

    // img_gray já está inicializado

    int h = img_gray.h;
    int w = img_gray.w;

    // Máscara circular preenchida, centrada na imagem (mm.circle desenha o
    // disco; a versão didática, teste de raio pixel a pixel, é mm.circle0).
    mm::Image mask_circ(h, w);
    mask_circ = mm::circle(mask_circ, w / 2, h / 2, std::min(h, w) / 3 - 10, 255, -1);

    // Operações via morph
    mm::Image img_not = mm::bnot(img_gray);            // NOT: negativo fotográfico
    mm::Image img_and = mm::band(img_gray, mask_circ); // preserva apenas a ROI circular
    mm::Image img_or  = mm::bor(img_gray, mask_circ);  // ilumina a região da máscara

    mm::show(
        std::vector<mm::Image>{img_gray, img_and, img_or, img_not},
        MM_OUT,
        std::vector<std::string>{"Original", "AND (ROI circular)", "OR (ilumina ROI)", "NOT (negativo)"},
        4
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray, "tmp/fig_03_logica_0.png");
mm::write(img_and, "tmp/fig_03_logica_1.png");
mm::write(img_or, "tmp/fig_03_logica_2.png");
mm::write(img_not, "tmp/fig_03_logica_3.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_03_logica.cpp


In [15]:
!g++ -I. -std=c++17 tmp/fig_03_logica.cpp -o tmp/fig_03_logica \
  && ./tmp/fig_03_logica \
  && test -f "tmp/fig_03_logica.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_logica.png"

[1] Original
[2] AND (ROI circular)
[3] OR (ilumina ROI)
[4] NOT (negativo)


In [16]:
try:
    mm.show(
        [
            mm.read("tmp/fig_03_logica_0.png"),
            mm.read("tmp/fig_03_logica_1.png"),
            mm.read("tmp/fig_03_logica_2.png"),
            mm.read("tmp/fig_03_logica_3.png"),
        ],
        titles=[
            'Original',
            'AND (ROI circular)',
            'OR (ilumina ROI)',
            'NOT (negativo)',
        ],
        cols=4,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_logica_0.png (ver a versao Python)")

<Figure size 3000x750 with 4 Axes>

**Figura 3.5:** Operações lógicas bit a bit com máscara circular: AND (isolamento da ROI), OR (iluminação da ROI) e NOT (negativo).


## 3.3 Histograma de Imagens

O **histograma** de uma imagem em tons de cinza é uma função discreta que descreve a distribuição de frequências das intensidades:

<a id="eq-03-histograma"></a>
$$
h(r_k) = n_k, \quad k = 0, 1, \ldots, L-1 \tag{3.5}
$$


onde $r_k$ é o $k$-ésimo nível de intensidade, $n_k$ é o número de pixels com essa intensidade e $L$ é o total de níveis (tipicamente 256 para 8 bits). O histograma normalizado estima a probabilidade de cada nível:

<a id="eq-03-hist-norm"></a>
$$
p(r_k) = \frac{n_k}{MN} \tag{3.6}
$$


onde $MN$ é o total de pixels. Por ser uma **estatística global**, o histograma não carrega informação posicional, mas revela características essenciais como brilho médio, contraste e distribuição tonal. Na prática, `mm::hist(img)` retorna o vetor de contagens $h(r_k)$, que serve tanto para visualização (via `mm::histImg`) quanto para cálculos como **função de distribuição acumulada (CDF)** e equalização.

> ### 📝 Interpretação do Histograma
>
> - **Estreito à esquerda:** imagem subexposta (escura).
> - **Estreito à direita:** imagem superexposta (clara).
> - **Concentrado no centro:** baixo contraste.
> - **Distribuído por toda a faixa:** alto contraste, boa utilização dos tons disponíveis.

A [Figura 3.6](#fig-03-histograma) apresenta o histograma da imagem do mandrill, bem como versões escurecida (`mm::subm`) e clareada (`mm::addm`). Observa-se o deslocamento da distribuição de intensidades para a esquerda e para a direita, respectivamente. Note que o intervalo representado no eixo $x$ não corresponde necessariamente a toda a faixa de 0 a 255.

In [17]:
%%writefile tmp/fig_03_histograma.cpp
#define MM_OUT "tmp/fig_03_histograma.png"
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_8.png");
// [pdi:state-io:end]

    // img_gray já está inicializado (inserido automaticamente)

    // Versão escurecida (-80) e clareada (+80) com saturação
    mm::Image img_dark = mm::subm(img_gray, 80);
    mm::Image img_high = mm::addm(img_gray, 80);

    // Exibição das imagens e seus histogramas
    mm::show(
        std::vector<mm::Image>{img_gray, img_dark, img_high,
                               mm::histImg(img_gray), mm::histImg(img_dark), mm::histImg(img_high)},
        MM_OUT,
        std::vector<std::string>{"Original", "Escurecida (-80)", "Clareada (+80)",
                                 "Histograma - original", "Histograma - escurecida", "Histograma - clareada"},
        3
    );

    return 0;
}

Overwriting tmp/fig_03_histograma.cpp


In [18]:
!g++ -I. -std=c++17 tmp/fig_03_histograma.cpp -o tmp/fig_03_histograma \
  && ./tmp/fig_03_histograma \
  && test -f "tmp/fig_03_histograma.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_histograma.png"

[1] Original
[2] Escurecida (-80)
[3] Clareada (+80)
[4] Histograma - original
[5] Histograma - escurecida
[6] Histograma - clareada


In [19]:
try:
    mm.show(mm.read("tmp/fig_03_histograma.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_histograma.png (ver a versao Python)")

<Figure size 784x524 with 1 Axes>

**Figura 3.6:** Histogramas da imagem original, de uma versão escurecida (−80) e de uma clareada (+80). A subtração/adição satura em 0 e 255.


### 3.3.1 Equalização de Histograma

A **equalização de histograma** redistribui as intensidades para que o histograma resultante seja o mais uniforme possível. O mapeamento é dado pela **função de distribuição acumulada (CDF)**:

<a id="eq-03-equalizacao"></a>
$$
s_k = T(r_k) = (L-1)\sum_{j=0}^{k} p(r_j) = \frac{L-1}{MN}\sum_{j=0}^{k} n_j \tag{3.7}
$$


A transformação é monotônica: níveis frequentes recebem intervalos maiores no domínio de saída (maior separação → mais contraste), enquanto níveis raros são comprimidos.

O algoritmo completo, em cinco etapas, é apresentado na [Tabela 3.1](#tbl-03-equalizacao).

<a id="tbl-03-equalizacao"></a>

**Tabela 3.1:** Algoritmo de equalização de histograma.

| Etapa | Operação | Fórmula |
|:-----:|:---------|:--------|
| 1 | **Histograma** | $h[k] \leftarrow$ número de pixels com intensidade $k$, $k=0\ldots L-1$ |
| 2 | **Probabilidade** | $p[k] \leftarrow h[k] / MN$ |
| 3 | **CDF** | $\text{cdf}[k] \leftarrow \sum_{j=0}^{k} p[j]$ (soma acumulada) |
| 4 | ***Look-Up Table* (mapeamento)** | $\text{lut}[k] \leftarrow \text{round}(\text{cdf}[k] \times (L-1))$ |
| 5 | **Aplicação** | $g[i,j] \leftarrow \text{lut}[f[i,j]]$ (para todo pixel) |


Note na [Figura 3.7](#fig-03-equalizacao-didatica) que a equalização **redistribui** os tons existentes para posições mais espaçadas na faixa $[0, L-1]$, mas não cria novos tons — a imagem equalizada continua com exatamente 3 tons distintos, agora em $\{1, 5, 7\}$ em vez de $\{2, 3, 4\}$.

In [20]:
%%writefile tmp/fig_03_equalizacao_didatica.cpp
#define MM_OUT "tmp/fig_03_equalizacao_didatica.png"
//| label: fig-03-equalizacao-didatica
//| fig-cap: "Equalização de histograma numa imagem 5×5 de 3 bits (L=8): tons concentrados em {2,3,4} são redistribuídos pela CDF. *mm::equalize(img, 3)* faz o mapeamento."
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>

int main() {
    mm::Image img5(5, 5);
    // preenche a imagem 5x5 com os valores especificados
    unsigned char vals[5][5] = {
        {3, 4, 2, 3, 4},
        {4, 3, 3, 4, 3},
        {2, 3, 4, 3, 2},
        {3, 4, 3, 2, 3},
        {4, 3, 2, 3, 4}
    };
    for (int y = 0; y < 5; ++y)
        for (int x = 0; x < 5; ++x)
            img5.at(y, x) = vals[y][x];

    mm::Image img5_eq = mm::equalize(img5, 3);   // L = 2^3 = 8

    std::cout << "Imagem original 5x5 (3 bits):\n";
    std::cout << mm::drawImg(img5) << "\n";
    std::cout << "Imagem equalizada 5x5:\n";
    std::cout << mm::drawImg(img5_eq) << "\n";

    mm::show(std::vector<mm::Image>{img5, img5_eq, mm::histImg(img5), mm::histImg(img5_eq)},
             MM_OUT,
             std::vector<std::string>{"Original", "Equalizada", "Histograma - original", "Histograma - equalizada"},
             2);

    return 0;
}

Overwriting tmp/fig_03_equalizacao_didatica.cpp


In [21]:
!g++ -I. -std=c++17 tmp/fig_03_equalizacao_didatica.cpp -o tmp/fig_03_equalizacao_didatica \
  && ./tmp/fig_03_equalizacao_didatica \
  && test -f "tmp/fig_03_equalizacao_didatica.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_equalizacao_didatica.png"

Imagem original 5x5 (3 bits):
3 4 2 3 4 
4 3 3 4 3 
2 3 4 3 2 
3 4 3 2 3 
4 3 2 3 4 

Imagem equalizada 5x5:
5 7 1 5 7 
7 5 5 7 5 
1 5 7 5 1 
5 7 5 1 5 
7 5 1 5 7 

[1] Original
[2] Equalizada
[3] Histograma - original
[4] Histograma - equalizada


In [22]:
try:
    mm.show(mm.read("tmp/fig_03_equalizacao_didatica.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_equalizacao_didatica.png (ver a versao Python)")

<Figure size 524x524 with 1 Axes>

**Figura 3.7:** Equalização de histograma numa imagem 5×5 de 3 bits (L=8): tons concentrados em {2,3,4} são redistribuídos pela CDF. *mm::equalize(img, 3)* faz o mapeamento.


**Limitação:** a equalização global pode super-realçar ruídos e produzir contraste excessivo em regiões homogêneas. O **CLAHE** (*Contrast Limited Adaptive Histogram Equalization*) reduz esse problema ao aplicar a equalização em blocos locais (*tiles*) e limitar a altura dos picos do histograma antes da equalização.

A [Figura 3.8](#fig-03-equalizacao) compara a imagem original, a equalização global via `mm::equalize` e o CLAHE do OpenCV, exibindo também os histogramas resultantes. Diferentemente da equalização global, que utiliza uma única transformação baseada na CDF de toda a imagem, o CLAHE adapta o contraste a cada região, sendo particularmente útil em imagens com iluminação não uniforme.

No exemplo, foi utilizado `clipLimit=2.0` e `tileGridSize=(32,32)`. O parâmetro `clipLimit` define o quanto os picos do histograma local podem crescer antes de serem cortados (*clipped*). No OpenCV, esse valor é um fator relativo: o limite real é aproximadamente calculado como `clipLimit × (número de pixels do bloco / número de níveis de cinza)`. Por exemplo, em um bloco com 4096 pixels e uma imagem de 8 bits (256 níveis de cinza), a frequência média por nível é $4096/256=16$. Assim, `clipLimit=2.0` permite picos de aproximadamente $2\times16=32$ ocorrências antes do corte. As ocorrências excedentes não são descartadas: elas são redistribuídas entre os demais níveis de cinza do histograma, reduzindo a concentração excessiva em poucos níveis e evitando uma amplificação exagerada do contraste local. Valores menores limitam mais o contraste e reduzem a amplificação de ruído, enquanto valores maiores permitem um realce mais intenso, mas podem introduzir artefatos.

- `clipLimit=1.0`: realce suave e conservador;
- `clipLimit=2.0`: bom equilíbrio entre contraste e naturalidade;
- `clipLimit=4.0`: maior destaque para detalhes locais;
- `clipLimit=8.0`: contraste agressivo, com possível amplificação de ruído.

Assim, o CLAHE costuma produzir resultados mais naturais do que a equalização global, especialmente em imagens com sombras, reflexos ou iluminação desigual.

In [23]:
%%writefile tmp/fig_03_equalizacao.cpp
#define MM_OUT "tmp/fig_03_equalizacao.png"
//| label: fig-03-equalizacao
//| fig-cap: "Equalização de histograma global (mm::equalize, via CDF) e os histogramas antes/depois. CLAHE (adaptativa) fica só na trilha Python — não tem equivalente em morph.hpp."
//| echo: true
//| output: true

#include "morph.hpp"
#include <vector>
#include <string>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_8.png");
// [pdi:state-io:end]

    mm::Image img_eq = mm::equalize(img_gray);

    mm::show(
        std::vector<mm::Image>{img_gray, img_eq, mm::histImg(img_gray), mm::histImg(img_eq)},
        MM_OUT,
        std::vector<std::string>{"Original", "mm.equalize (CDF)", "Histograma - original", "Histograma - equalizado"},
        2
    );

    return 0;
}

Overwriting tmp/fig_03_equalizacao.cpp


In [24]:
!g++ -I. -std=c++17 tmp/fig_03_equalizacao.cpp -o tmp/fig_03_equalizacao \
  && ./tmp/fig_03_equalizacao \
  && test -f "tmp/fig_03_equalizacao.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_equalizacao.png"

[1] Original
[2] mm.equalize (CDF)
[3] Histograma - original
[4] Histograma - equalizado


In [25]:
try:
    mm.show(mm.read("tmp/fig_03_equalizacao.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_equalizacao.png (ver a versao Python)")

<Figure size 524x524 with 1 Axes>

**Figura 3.8:** Equalização de histograma global (mm::equalize, via CDF) e os histogramas antes/depois. CLAHE (adaptativa) fica só na trilha Python — não tem equivalente em morph.hpp.


### 3.3.2 Especificação de Histograma

Enquanto a equalização impõe uma distribuição uniforme, a **especificação de histograma** (*histogram matching*) permite que o histograma da imagem de saída siga uma distribuição **arbitrária** — por exemplo, o histograma de outra imagem de referência.

O procedimento envolve três etapas:

1. Calcular a CDF da imagem de entrada: $P_r(r_k)$.
2. Calcular a CDF da imagem de referência: $P_z(z_k)$.
3. Para cada nível $r_k$, encontrar o nível $z$ que minimiza $|P_z(z) - P_r(r_k)|$.

<a id="eq-03-especificacao"></a>
$$
T(r_k) = \arg\min_{z}\,|P_z(z) - P_r(r_k)| \tag{3.8}
$$


Na [Figura 3.9](#fig-03-especificacao), transferimos o perfil tonal do leopardo ([Figura 3.3](#fig-03-leopardo)) para a imagem do mandrill — uma aplicação direta do conceito visto no *blending*: em vez de fundir pixels, aqui fundimos distribuições tonais.

In [26]:
%%writefile tmp/fig_03_especificacao.cpp
#define MM_OUT "tmp/fig_03_especificacao.png"
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_8.png");
mm::Image img_leop_gray = mm::_read_state("tmp/state/img_leop_gray_12.png");
// [pdi:state-io:end]

    // img_gray e img_leop_gray são fornecidos automaticamente

    // Equalização global do mandril
    mm::Image img_eq = mm::equalize(img_gray);

    // Exibição das imagens: original, referência e equalizada
    mm::show(std::vector<mm::Image>{img_gray, img_leop_gray, img_eq},
             MM_OUT,
             std::vector<std::string>{"Mandril (original)", "Leopardo (referencia)", "Mandril equalizado"},
             3);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray, "tmp/fig_03_especificacao_0.png");
mm::write(img_leop_gray, "tmp/fig_03_especificacao_1.png");
mm::write(img_eq, "tmp/fig_03_especificacao_2.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_03_especificacao.cpp


In [27]:
!g++ -I. -std=c++17 tmp/fig_03_especificacao.cpp -o tmp/fig_03_especificacao \
  && ./tmp/fig_03_especificacao \
  && test -f "tmp/fig_03_especificacao.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_especificacao.png"

[1] Mandril (original)
[2] Leopardo (referencia)
[3] Mandril equalizado


In [28]:
try:
    mm.show(
        [
            mm.read("tmp/fig_03_especificacao_0.png"),
            mm.read("tmp/fig_03_especificacao_1.png"),
            mm.read("tmp/fig_03_especificacao_2.png"),
        ],
        titles=[
            'Mandril (original)',
            'Leopardo (referencia)',
            'Mandril equalizado',
        ],
        cols=3,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_especificacao_0.png (ver a versao Python)")

<Figure size 2250x750 with 3 Axes>

**Figura 3.9:** Especificação de histograma (mapear o mandril para o perfil tonal do leopardo) precisa da CDF inversa da referência — fica só na trilha Python. Aqui, a equalização global do mandril, como comparação.


## 3.4 Fundamentos Espaciais: Vizinhança, Convolução e *Kernels*

As operações de filtragem espacial não atuam em um único pixel isolado, mas em uma **vizinhança** ao seu redor. Para isso, utiliza-se uma pequena matriz de coeficientes denominada **kernel** (ou máscara), que percorre toda a imagem por meio de uma **janela deslizante** (*sliding window*).

As janelas mais comuns são 3×3, 5×5 e 7×7. Em uma janela 3×3, por exemplo, o pixel central é processado juntamente com seus oito vizinhos imediatos. Em cada posição da janela, os valores dos pixels são combinados com os coeficientes do *kernel*, produzindo um novo valor para o pixel central.

### 3.4.1 Vizinhança

Considere uma janela 3×3 centrada no pixel $(x,y)$:

<a id="eq-03-box3x3"></a>
$$
\begin{bmatrix}
(x-1,y-1) & (x,y-1) & (x+1,y-1) \\
(x-1,y)   & (x,y)   & (x+1,y)   \\
(x-1,y+1) & (x,y+1) & (x+1,y+1)
\end{bmatrix} \tag{3.9}
$$


De forma geral, uma janela de tamanho $(2a+1)\times(2b+1)$ abrange todos os pixels situados até $a$ posições na horizontal e até $b$ posições na vertical em relação ao pixel central. Assim, uma janela 3×3 corresponde a $a=b=1$, uma janela 5×5 a $a=b=2$, e assim por diante.

Matematicamente, a vizinhança é definida por

<a id="eq-03-vizinhanca"></a>
$$
\mathcal{V}(x,y)=
\{(x+s,\,y+t): -a\le s\le a,\,-b\le t\le b\} \tag{3.10}
$$


### 3.4.2 Tratamento de Bordas

Pixels próximos às bordas possuem parte de sua vizinhança fora da imagem. Para aplicar filtros nessas regiões, é preciso definir como os valores externos serão obtidos. As três estratégias mais comuns (com a constante equivalente do OpenCV entre parênteses) são:

- **Zero-padding** (`BORDER_CONSTANT`): completa a região externa com zeros.
- **Replicação** (`BORDER_REPLICATE`): repete o valor do pixel da borda.
- **Reflexão** (`BORDER_REFLECT_101`): espelha os pixels vizinhos, sem repetir o da borda.

`mm::conv` usa a reflexão por padrão, pois preserva melhor a continuidade dos níveis de cinza e reduz artefatos no tratamento das bordas.

O exemplo a seguir compara as três estratégias com `mm::pad` numa matriz 3×3. Observe como cada uma preenche os pixels externos necessários para aplicar um filtro 3×3 também nos cantos.

In [29]:
%%writefile tmp/mm_out_1.cpp
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
    // Criando a imagem 3x3 diretamente
    mm::Image img(3, 3);
    img.at(0,0) = 1; img.at(0,1) = 2; img.at(0,2) = 3;
    img.at(1,0) = 4; img.at(1,1) = 5; img.at(1,2) = 6;
    img.at(2,0) = 7; img.at(2,1) = 8; img.at(2,2) = 9;

    std::vector<std::string> bordas = {"constant", "replicate", "reflect101"};

    std::cout << "Imagem original:" << std::endl;
    std::cout << mm::drawImg(img) << std::endl;

    for (int k : {3, 5}) {
        int b = k / 2;
        std::cout << "=== Kernel " << k << "x" << k << " (padding b=" << b << ") ===" << std::endl;
        for (const std::string& nome : bordas) {
            std::cout << nome << std::endl;

            mm::Border border;
            if (nome == "constant") {
                border = mm::Border::CONSTANT;
            } else if (nome == "replicate") {
                border = mm::Border::REPLICATE;
            } else {
                border = mm::Border::REFLECT101;
            }

            std::cout << mm::drawImg(mm::pad(img, b, border)) << std::endl;
        }
    }

    return 0;
}

Overwriting tmp/mm_out_1.cpp


In [30]:
!g++ -I. -std=c++17 tmp/mm_out_1.cpp -o tmp/mm_out_1 \
  && ./tmp/mm_out_1

Imagem original:
1 2 3 
4 5 6 
7 8 9 

=== Kernel 3x3 (padding b=1) ===
constant
0 0 0 0 0 
0 1 2 3 0 
0 4 5 6 0 
0 7 8 9 0 
0 0 0 0 0 

replicate
1 1 2 3 3 
1 1 2 3 3 
4 4 5 6 6 
7 7 8 9 9 
7 7 8 9 9 

reflect101
5 4 5 6 5 
2 1 2 3 2 
5 4 5 6 5 
8 7 8 9 8 
5 4 5 6 5 

=== Kernel 5x5 (padding b=2) ===
constant
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
0 0 1 2 3 0 0 
0 0 4 5 6 0 0 
0 0 7 8 9 0 0 
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 

replicate
1 1 1 2 3 3 3 
1 1 1 2 3 3 3 
1 1 1 2 3 3 3 
4 4 4 5 6 6 6 
7 7 7 8 9 9 9 
7 7 7 8 9 9 9 
7 7 7 8 9 9 9 

reflect101
9 8 7 8 9 8 7 
6 5 4 5 6 5 4 
3 2 1 2 3 2 1 
6 5 4 5 6 5 4 
9 8 7 8 9 8 7 
6 5 4 5 6 5 4 
3 2 1 2 3 2 1 



Observe que o resultado de um filtro pode variar significativamente conforme o tratamento adotado para as bordas da imagem.

Na `morph.hpp`, as funções de filtragem (`mm::conv`, `mm::blur`, `mm::gaussian`, `mm::laplacian`, `mm::usm`) aplicam *padding* por **reflexão** (`mm::Border::REFLECT101`) por padrão — o mesmo comportamento do `cv2.filter2D`. A variante didática `mm::conv0` usa `mm::Border::KEEP`: os pixels da borda mantêm o valor original, sem o filtro. Já `mm::sobel` e `mm::prewitt` deixam a borda em zero (calculam apenas o interior).

### 3.4.3 Correlação vs. Convolução

Existem dois mecanismos matematicamente relacionados.

**Correlação cruzada** (*cross-correlation*) — o *kernel* é aplicado diretamente:

<a id="eq-03-correlacao"></a>
$$
g(x,y) = \sum_{s=-a}^{a}\sum_{t=-b}^{b} w(s,t)\,f(x+s,\,y+t) \tag{3.11}
$$


**Convolução bidimensional** — o *kernel* é rotacionado 180° antes da aplicação:

<a id="eq-03-convolucao"></a>
$$
g(x,y) = \sum_{s=-a}^{a}\sum_{t=-b}^{b} w(s,t)\,f(x-s,\,y-t) \tag{3.12}
$$


Para *kernels* simétricos (Gaussiano, Laplaciano, média) as duas operações produzem resultados idênticos. Para *kernels* assimétricos (Sobel, Prewitt) a diferença é significativa, como mostram os exemplos a seguir.

#### 3.4.3.1 Correlação (`mm::conv`)

In [31]:
%%writefile tmp/mm_out_2.cpp
// Compile: g++ -std=c++17 -o programa programa.cpp -I. -lmorph

#include "morph.hpp"
#include <iostream>
#include <vector>

int main() {
    // imagem 4x4 (uint8) e kernel assimétrico 3x3
    mm::Image img(4, 4, 1);
    unsigned char dados[4][4] = {{ 1,  2,  3,  4},
                                 { 5,  6,  7,  8},
                                 { 9, 10, 11, 12},
                                 {13, 14, 15, 16}};
    for (int y = 0; y < 4; ++y)
        for (int x = 0; x < 4; ++x)
            img.at(y, x) = dados[y][x];

    mm::Kernel w{{0,1,2},{0,0,0},{0,0,0}};

    mm::Image corr = mm::conv(img, w, mm::Border::CONSTANT);  // zero fora da imagem

    std::cout << "Imagem original:" << std::endl;
    std::cout << mm::drawImg(img) << std::endl;
    std::cout << "Kernel:" << std::endl;
    // Como a função drawImg espera mm::Image, precisamos converter o kernel
    mm::Image kernel_img(3, 3, 1);
    for (int y = 0; y < 3; ++y)
        for (int x = 0; x < 3; ++x)
            kernel_img.at(y, x, 0) = (unsigned char)w.at(y, x);
    std::cout << mm::drawImg(kernel_img) << std::endl;
    std::cout << "Resultado da correlação:" << std::endl;
    std::cout << mm::drawImg(corr) << std::endl;

    return 0;
}

Overwriting tmp/mm_out_2.cpp


In [32]:
!g++ -I. -std=c++17 tmp/mm_out_2.cpp -o tmp/mm_out_2 \
  && ./tmp/mm_out_2

Imagem original:
 1  2  3  4 
 5  6  7  8 
 9 10 11 12 
13 14 15 16 

Kernel:
0 1 2 
0 0 0 
0 0 0 

Resultado da correlação:
 0  0  0  0 
 5  8 11  4 
17 20 23  8 
29 32 35 12 



`mm::conv` realiza correlação, isto é, aplica o *kernel* exatamente na orientação fornecida.

#### 3.4.3.2 Convolução

In [33]:
%%writefile tmp/mm_out_3.cpp
#include "morph.hpp"
#include <iostream>

int main() {
    // repetidos aqui para a célula ser independente
    mm::Image img(4, 4);
    // Preenchendo a imagem com valores 1..16
    int val = 1;
    for (int y = 0; y < 4; y++) {
        for (int x = 0; x < 4; x++) {
            img.at(y, x) = val++;
        }
    }

    mm::Kernel w{{0, 1, 2},
                 {0, 0, 0},
                 {0, 0, 0}};

    // kernel rotacionado 180° (equivale a np.rot90(w, 2))
    mm::Kernel w_conv{{0, 0, 0},
                      {0, 0, 0},
                      {2, 1, 0}};

    mm::Image conv = mm::conv(img, w_conv, mm::Border::CONSTANT);

    std::cout << "Imagem original:" << "\n";
    std::cout << mm::drawImg(img) << "\n";
    std::cout << "Kernel original:" << "\n";
    std::cout << mm::drawImg(w) << "\n";
    std::cout << "Kernel rotacionado 180°:" << "\n";
    std::cout << mm::drawImg(w_conv) << "\n";
    std::cout << "Resultado da convolução:" << "\n";
    std::cout << mm::drawImg(conv) << "\n";

    return 0;
}

Overwriting tmp/mm_out_3.cpp


In [34]:
!g++ -I. -std=c++17 tmp/mm_out_3.cpp -o tmp/mm_out_3 \
  && ./tmp/mm_out_3

Imagem original:
 1  2  3  4 
 5  6  7  8 
 9 10 11 12 
13 14 15 16 

Kernel original:
   0    1    2 
   0    0    0 
   0    0    0 

Kernel rotacionado 180°:
   0    0    0 
   0    0    0 
   2    1    0 

Resultado da convolução:
 5 16 19 22 
 9 28 31 34 
13 40 43 46 
 0  0  0  0 



A convolução utiliza o *kernel* rotacionado em 180°. Para reproduzir a definição matemática de convolução, rotaciona-se o *kernel* (aqui, `[[0,1,2],[0,0,0],[0,0,0]]` → `[[0,0,0],[0,0,0],[2,1,0]]`) antes de aplicar `mm::conv`.

### 3.4.4 O Papel do *Kernel*

Os coeficientes do *kernel* determinam completamente o efeito produzido pelo filtro, conforme resumido na [Tabela 3.2](#tbl-03-kernels).

<a id="tbl-03-kernels"></a>

**Tabela 3.2:** Interpretação típica dos coeficientes do *kernel*.

| Característica | Efeito típico |
|:---|:---|
| Coeficientes positivos com soma 1 | Suavização (*passa-baixa*) |
| Soma igual a 0, com valores positivos e negativos | Detecção de bordas (*passa-alta*) |
| Coeficiente central positivo dominante e vizinhos negativos | Realce de nitidez |
| Coeficientes assimétricos | Gradiente direcional |


**Exemplos:**

Suavização:
$$
\frac{1}{9}
\begin{bmatrix}
1&1&1\\
1&1&1\\
1&1&1
\end{bmatrix}
$$

Detecção de bordas:
$$
\begin{bmatrix}
-1&-1&-1\\
-1&8&-1\\
-1&-1&-1
\end{bmatrix}
$$

Realce de nitidez:
$$
\begin{bmatrix}
0&-1&0\\
-1&5&-1\\
0&-1&0
\end{bmatrix}
$$

Gradiente direcional (Sobel):
$$
\begin{bmatrix}
-1&0&1\\
-2&0&2\\
-1&0&1
\end{bmatrix}
$$

A [Figura 3.10](#fig-03-convolucao-passo) demonstra o mecanismo passo a passo: para cada posição da janela, multiplica-se cada coeficiente do *kernel* pelo pixel correspondente da vizinhança e somam-se os produtos obtidos. O resultado é exatamente o valor definido pela [Equação 3.11](#eq-03-correlacao) para aquela posição da imagem. Embora as imagens produzidas por `mm::conv0` e `cv2.filter2D` (ou `mm::conv`) sejam visualmente muito semelhantes, a implementação baseada em OpenCV é milhares de vezes mais rápida, como mostrado a seguir.

> ### ⚠️ Desempenho: laços Python vs. operações vetorizadas
>
> A função `mm::conv0` implementa a correlação diretamente em Python por meio de laços aninhados. Embora essa abordagem seja adequada para fins didáticos, ela executa um grande número de operações e torna-se lenta para imagens maiores.
>
> Já `mm::conv` utiliza `cv2.filter2D`, implementado em C++ e otimizado para operações matriciais. No exemplo apresentado, a versão vetorizada foi mais de **3000 vezes mais rápida** que a implementação didática, produzindo um resultado visualmente equivalente.
>
> As diferenças numéricas observadas concentram-se principalmente nas bordas da imagem. Em `mm::conv0`, os pixels da borda permanecem inalterados, enquanto `mm::conv` utiliza uma estratégia de reflexão das bordas (`cv2.BORDER_REFLECT_101`, padrão do `cv2.filter2D`).
>
> Por isso, `mm::conv0` deve ser utilizado para compreender o algoritmo, enquanto `mm::conv` é a opção recomendada para aplicações práticas.

In [35]:
%%writefile tmp/fig_03_convolucao_passo.cpp
#define MM_OUT "tmp/fig_03_convolucao_passo.png"
//| label: fig-03-convolucao-passo
//| fig-cap: "Correlação com *kernel* de média 3×3: versão didática mm::conv0 (bordas preservadas) vs. mm::conv (borda refletida). A diferença se concentra nas bordas."
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_leop_gray = mm::_read_state("tmp/state/img_leop_gray_12.png");
// [pdi:state-io:end]

    mm::Kernel w_mean = mm::Kernel::mean(3);
    mm::Image img_gray = img_leop_gray;

    mm::Image img_conv0 = mm::conv0(img_gray, w_mean);   // laços, bordas preservadas
    mm::Image img_conv = mm::conv(img_gray, w_mean);    // borda refletida

    std::cout << "Correlacao no pixel central [251,251]:" << "\n";
    std::cout << "  original = " << (int)img_gray.at(251, 251) << "\n";
    std::cout << "  conv0    = " << (int)img_conv0.at(251, 251) << "\n";
    std::cout << "  conv     = " << (int)img_conv.at(251, 251) << "\n";

    mm::show({img_gray, img_conv0, img_conv},
             MM_OUT,
             {"Original", "conv0 (laços)", "conv (vetorizado)"}, 3);
    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray, "tmp/fig_03_convolucao_passo_0.png");
mm::write(img_conv0, "tmp/fig_03_convolucao_passo_1.png");
mm::write(img_conv, "tmp/fig_03_convolucao_passo_2.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_03_convolucao_passo.cpp


In [36]:
!g++ -I. -std=c++17 tmp/fig_03_convolucao_passo.cpp -o tmp/fig_03_convolucao_passo \
  && ./tmp/fig_03_convolucao_passo \
  && test -f "tmp/fig_03_convolucao_passo.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_convolucao_passo.png"

Correlacao no pixel central [251,251]:
  original = 104
  conv0    = 102
  conv     = 102
[1] Original
[2] conv0 (laços)
[3] conv (vetorizado)


In [37]:
try:
    mm.show(
        [
            mm.read("tmp/fig_03_convolucao_passo_0.png"),
            mm.read("tmp/fig_03_convolucao_passo_1.png"),
            mm.read("tmp/fig_03_convolucao_passo_2.png"),
        ],
        titles=[
            'Original',
            'conv0 (laços)',
            'conv (vetorizado)',
        ],
        cols=3,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_convolucao_passo_0.png (ver a versao Python)")

<Figure size 2250x750 with 3 Axes>

**Figura 3.10:** Correlação com *kernel* de média 3×3: versão didática mm::conv0 (bordas preservadas) vs. mm::conv (borda refletida). A diferença se concentra nas bordas.


In [38]:
%%writefile tmp/mm_out_4.cpp
#include "morph.hpp"
#include <iostream>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_leop = mm::_read_state("tmp/state/img_leop_12.png");
// [pdi:state-io:end]

    //| echo: false
    // A partir daqui o "sujeito" dos exemplos de filtragem passa a ser o
    // leopardo (mais textura e bordas que o mandril).
    mm::Image img_gray = mm::gray(img_leop);

    
// [pdi:state-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp/state");
mm::write(img_gray, "tmp/state/img_gray_45.png");
// [pdi:state-io:end]
return 0;
}

Overwriting tmp/mm_out_4.cpp


In [39]:
!g++ -I. -std=c++17 tmp/mm_out_4.cpp -o tmp/mm_out_4 \
  && ./tmp/mm_out_4

### 3.4.5 Exemplo Numérico: Correlação Passo a Passo

Para tornar concreto o mecanismo da [Equação 3.11](#eq-03-correlacao), considere o *kernel* de média 3×3 ($a=b=1$, todos os coeficientes $= 1/9 \approx 0{,}111$) aplicado ao *patch* 5×5 extraído da imagem do leopardo. A [Figura 3.11](#fig-03-patch) exibe o *patch* com grade e destaca em amarelo a janela 3×3 centrada no pixel $[1,1]$:

In [40]:
%%writefile tmp/fig_03_patch.cpp
#define MM_OUT "tmp/fig_03_patch.png"
#include "morph.hpp"
#include <iostream>

//| label: fig-03-patch
//| fig-cap: "*Patch* 5×5 extraído da imagem do leopardo (posição [250:255, 250:255]). A janela amarela destaca a vizinhança 3×3 centrada no pixel [1,1] onde a correlação será calculada."
//| echo: true
//| output: true

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_45.png");
// [pdi:state-io:end]

    mm::Image patch = mm::crop(img_gray, 250, 255, 250, 255);
    mm::Kernel B{{1,1,1},{1,1,1},{1,1,1}};

    std::cout << "Patch 5×5 (intensidades):\n";
    std::cout << mm::drawImg(patch) << "\n";

    mm::drawImgKernel(patch, B, 1, 1, MM_OUT, 40);

    return 0;
}

Overwriting tmp/fig_03_patch.cpp


In [41]:
!g++ -I. -std=c++17 tmp/fig_03_patch.cpp -o tmp/fig_03_patch \
  && ./tmp/fig_03_patch \
  && test -f "tmp/fig_03_patch.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_patch.png"

Patch 5×5 (intensidades):
 94  96 103 113 115 
107 104 103 107 114 
 98 102 112 117 116 
 81  91 112 122 118 
 85  91 110 119 121 

Processando pixel (x,y)=(1,1)  |  janela do kernel 3x3
 94  96 103 113 115 
107 104 103 107 114 
 98 102 112 117 116 
 81  91 112 122 118 
 85  91 110 119 121 


In [42]:
try:
    mm.show(mm.read("tmp/fig_03_patch.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_patch.png (ver a versao Python)")

<Figure size 450x450 with 1 Axes>

**Figura 3.11:** *Patch* 5×5 extraído da imagem do leopardo (posição [250:255, 250:255]). A janela amarela destaca a vizinhança 3×3 centrada no pixel [1,1] onde a correlação será calculada.


Para ilustrar o cálculo da correlação, considere o pixel na posição $[1,1]$ do *patch* 5×5 mostrado na [Figura 3.11](#fig-03-patch). Essa posição foi escolhida apenas por conveniência didática, pois possui uma vizinhança 3×3 completa ao seu redor.

Os valores dessa vizinhança correspondem à submatriz superior esquerda do *patch*:

$$
\text{vizinhança} = \begin{bmatrix}
 91 &  95 & 108 \\
106 & 107 & 108 \\
102 & 103 & 107
\end{bmatrix}
$$

Aplicando a [Equação 3.11](#eq-03-correlacao) com o kernel de média:

$$
g[1,1] = \frac{ 91+95+108+106+107+108+102+103+107}{9} = \frac{927}{9} = 103
$$

O resultado (103) é ligeiramente menor que o valor original do pixel central (107), pois a média incorpora vizinhos de menor intensidade, produzindo o efeito de suavização. Na prática, o algoritmo inicia o processamento em $[0,0]$ e repete esse mesmo cálculo para cada posição da imagem, deslocando a janela até cobrir todo o domínio.

## 3.5 Filtragem Espacial de Suavização

Os filtros de suavização (*smoothing filters*) atenuam variações bruscas de intensidade, reduzindo ruído e detalhes de alta frequência. São filtros **passa-baixa** — preservam as componentes de baixa frequência (estruturas grandes) e atenuam as de alta frequência (ruído, bordas).

### 3.5.1 Filtro de Média (*Box Filter*)

O filtro de média utiliza um *kernel* uniforme de tamanho $n \times n$, onde todos os coeficientes valem $1/n^2$:

<a id="eq-03-media"></a>
$$
w_{\text{média}} = \frac{1}{n^2}
\begin{bmatrix}
1 & \cdots & 1 \\
\vdots & \ddots & \vdots \\
1 & \cdots & 1
\end{bmatrix}_{n \times n} \tag{3.13}
$$


Cada pixel de saída é a média aritmética dos $n^2$ pixels de sua vizinhança. Note que a soma dos coeficientes é sempre 1 — o brilho médio da imagem é preservado. *Kernels* maiores produzem suavização mais agressiva, mas borram progressivamente as bordas.

A [Figura 3.12](#fig-03-media) mostra o efeito do filtro de média com *kernels* $3\times3$, $7\times7$ e $15\times15$ sobre um detalhe da imagem do leopardo. Os resultados foram obtidos com `mm::blur`, que implementa o filtro de média por meio da função `cv2.blur`, equivalente à convolução da imagem com um *kernel* uniforme cujos coeficientes são $h(x,y)=1/N^2$; de forma equivalente, o mesmo resultado pode ser obtido com `mm::conv`, calculando ($g=f*h$). À medida que o *kernel* aumenta, mais pixels contribuem para cada valor de saída, intensificando a suavização, reduzindo o ruído e tornando detalhes finos e bordas progressivamente mais borrados.

In [43]:
%%writefile tmp/fig_03_media.cpp
#define MM_OUT "tmp/fig_03_media.png"
// Compile: g++ -std=c++17 -o programa programa.cpp -I. -lstdc++fs

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_45.png");
// [pdi:state-io:end]

    // img_gray já está inicializado (inserido automaticamente)

    // Detalhe da região do olho
    int y0 = 580, y1 = 740, x0 = 680, x1 = 900;

    mm::Image img_gray_crop = mm::crop(img_gray, y0, y1, x0, x1);

    std::vector<int> sizes = {3, 7, 15};
    std::vector<mm::Image> imgs;
    imgs.push_back(img_gray_crop);

    for (int k : sizes) {
        imgs.push_back(mm::blur(img_gray_crop, k)); // ou
        // mm::Image kernel = mm::Kernel::mean(k);
        // imgs.push_back(mm::conv(img_gray_crop, mm::Kernel::mean(k)));
    }

    std::vector<std::string> titles;
    titles.push_back("Original");
    for (int k : sizes) {
        titles.push_back("Média " + std::to_string(k) + "×" + std::to_string(k));
    }

    mm::show(imgs, MM_OUT, titles, 4);

    return 0;
}

Overwriting tmp/fig_03_media.cpp


In [44]:
!g++ -I. -std=c++17 tmp/fig_03_media.cpp -o tmp/fig_03_media \
  && ./tmp/fig_03_media \
  && test -f "tmp/fig_03_media.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_media.png"

[1] Original
[2] Média 3×3
[3] Média 7×7
[4] Média 15×15


In [45]:
try:
    mm.show(mm.read("tmp/fig_03_media.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_media.png (ver a versao Python)")

<Figure size 1044x450 with 1 Axes>

**Figura 3.12:** Filtro de média com *kernels* de tamanho crescente (3×3, 7×7, 15×15). O borramento das bordas aumenta com o tamanho do *kernel*.


### 3.5.2 Filtro Gaussiano

O filtro Gaussiano pesa os pixels da vizinhança de acordo com uma função Gaussiana bidimensional:

<a id="eq-03-gaussiana"></a>
$$
G(s,t) = \frac{1}{2\pi\sigma^2}\,e^{-\frac{s^2+t^2}{2\sigma^2}} \tag{3.14}
$$


onde $\sigma$ é o desvio padrão e controla o raio de influência. Pixels mais próximos do centro têm peso maior; pixels distantes são progressivamente ignorados.

A [Figura 3.13](#fig-03-gauss-kernel) apresenta o *kernel* Gaussiano $5\times5$ gerado para $\sigma=1$. O *kernel* foi construído a partir do produto externo de dois vetores Gaussianos unidimensionais e posteriormente normalizado para que a soma de seus coeficientes seja igual a $1$. Observa-se que os maiores pesos concentram-se no centro da matriz, decrescendo radialmente em direção às bordas. Essa distribuição faz com que os pixels centrais tenham maior influência no resultado da filtragem, contribuindo para uma suavização mais natural e com melhor preservação de bordas do que o filtro de média.

In [46]:
%%writefile tmp/fig_03_gauss_kernel.cpp
#define MM_OUT "tmp/fig_03_gauss_kernel.png"
#include "morph.hpp"
#include <iostream>
#include <iomanip>
#include <string>
#include <vector>

int main() {
    //| label: fig-03-gauss-kernel
    //| fig-cap: "*Kernel* Gaussiano 5×5 (σ=1): resposta ao impulso do filtro — pesos maiores no centro, decrescendo radialmente."
    //| echo: true
    //| output: true

    mm::Kernel w = mm::Kernel::gaussian(5, 1.0);   // mm::Kernel::gaussian(5, 1.0) na morph.hpp

    std::cout << "Kernel Gaussiano 5x5 (s=1), normalizado:\n";
    for (int y = 0; y < 5; y++) {
        std::cout << "  ";
        for (int x = 0; x < 5; x++) {
            if (x > 0) std::cout << "  ";
            std::cout << std::fixed << std::setprecision(4) << w.at(y, x);
        }
        std::cout << "\n";
    }
    std::cout << "Peso central [2,2] = " << std::fixed << std::setprecision(4) << w.at(2, 2) 
              << "   |   canto [0,0] = " << std::fixed << std::setprecision(4) << w.at(0, 0) << "\n";

    // Visualização: resposta do filtro Gaussiano a um impulso central
    mm::Image impulso(5, 5);
    impulso.at(2, 2) = 255;
    mm::drawImgPlt(mm::gaussian(impulso, 5, 1.0), MM_OUT, 40);

    return 0;
}

Overwriting tmp/fig_03_gauss_kernel.cpp


In [47]:
!g++ -I. -std=c++17 tmp/fig_03_gauss_kernel.cpp -o tmp/fig_03_gauss_kernel \
  && ./tmp/fig_03_gauss_kernel \
  && test -f "tmp/fig_03_gauss_kernel.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_gauss_kernel.png"

Kernel Gaussiano 5x5 (s=1), normalizado:
  0.0030  0.0133  0.0219  0.0133  0.0030
  0.0133  0.0596  0.0983  0.0596  0.0133
  0.0219  0.0983  0.1621  0.0983  0.0219
  0.0133  0.0596  0.0983  0.0596  0.0133
  0.0030  0.0133  0.0219  0.0133  0.0030
Peso central [2,2] = 0.1621   |   canto [0,0] = 0.0030
 3  7 11  7  3 
 7 15 25 15  7 
11 25 41 25 11 
 7 15 25 15  7 
 3  7 11  7  3 


In [48]:
try:
    mm.show(mm.read("tmp/fig_03_gauss_kernel.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_gauss_kernel.png (ver a versao Python)")

<Figure size 450x450 with 1 Axes>

**Figura 3.13:** *Kernel* Gaussiano 5×5 (σ=1): resposta ao impulso do filtro — pesos maiores no centro, decrescendo radialmente.


> ### 📝 Vantagem computacional da separabilidade
>
> Considere um *kernel* quadrado de tamanho $n \times n$. Se esse filtro for separável (como o Gaussiano), a convolução 2D pode ser decomposta em duas convoluções 1D: uma horizontal e outra vertical.
>
> Nesse caso, o custo por pixel passa de aproximadamente $O(n^2)$ operações (convolução 2D direta) para $O(2n)$ operações (duas convoluções 1D). Assim, a complexidade é reduzida de forma significativa, tornando o processamento mais eficiente

Comparado ao filtro de média, o Gaussiano:

- **Preserva melhor as bordas** — a ponderação radial suaviza sem criar transições abruptas;
- **Não introduz anéis** (*ringing*) no domínio da frequência, pois a Gaussiana é sua própria transformada de Fourier (Capítulo 5);
- **É controlado por $\sigma$** — aumentar $\sigma$ equivale a aumentar o raio de suavização de forma contínua e previsível.

A [Figura 3.14](#fig-03-gauss) compara os filtros de média e Gaussiano aplicados à imagem do leopardo usando uma janela $9\times9$. O filtro de média foi implementado por convolução com um *kernel* uniforme, em que todos os $81$ pixels da vizinhança possuem o mesmo peso ($1/81$), enquanto o filtro Gaussiano foi obtido com `cv2.GaussianBlur`, utilizando pesos definidos por uma distribuição Gaussiana. Ambos reduzem ruído e suavizam a imagem, porém o filtro Gaussiano preserva melhor as bordas e os detalhes locais, como pode ser observado na região ampliada do olho.

A [Figura 3.14](#fig-03-gauss) compara os filtros de média e Gaussiano aplicados a um detalhe da imagem do leopardo com *kernels* $9\times 9$. O filtro de média foi obtido com `mm::blur`, equivalente à convolução com um *kernel* uniforme cujos coeficientes valem $1/81$, enquanto o filtro Gaussiano foi obtido com `mm::gaussian`, equivalente à convolução com um *kernel* gerado a partir de uma distribuição Gaussiana. Ambos promovem suavização e redução de ruído, porém o filtro Gaussiano atribui maior peso aos pixels centrais da vizinhança, preservando melhor as bordas e os detalhes locais, como pode ser observado na região ampliada do olho.

In [49]:
%%writefile tmp/fig_03_gauss.cpp
#define MM_OUT "tmp/fig_03_gauss.png"
#include "morph.hpp"
#include <vector>
#include <string>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_45.png");
// [pdi:state-io:end]

    // Filtro de média 9×9
    mm::Image img_media9 = mm::blur(img_gray, 9);
    // Filtro Gaussiano 9×9 com σ=0 (sigma automático)
    mm::Image img_gauss9 = mm::gaussian(img_gray, 9, 0);

    // Detalhe da região do olho
    mm::Image img_gray_crop = mm::crop(img_gray, 580, 740, 680, 900);
    mm::Image img_media9_crop = mm::crop(img_media9, 580, 740, 680, 900);
    mm::Image img_gauss9_crop = mm::crop(img_gauss9, 580, 740, 680, 900);

    mm::show(
        std::vector<mm::Image>{img_gray_crop, img_media9_crop, img_gauss9_crop},
        MM_OUT,
        std::vector<std::string>{"Detalhe: Original", "Média 9×9", "Gaussiano 9×9"},
        3
    );

    
// [pdi:state-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp/state");
mm::write(img_gray_crop, "tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray_crop, "tmp/fig_03_gauss_0.png");
mm::write(img_media9_crop, "tmp/fig_03_gauss_1.png");
mm::write(img_gauss9_crop, "tmp/fig_03_gauss_2.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_03_gauss.cpp


In [50]:
!g++ -I. -std=c++17 tmp/fig_03_gauss.cpp -o tmp/fig_03_gauss \
  && ./tmp/fig_03_gauss \
  && test -f "tmp/fig_03_gauss.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_gauss.png"

[1] Detalhe: Original
[2] Média 9×9
[3] Gaussiano 9×9


In [51]:
try:
    mm.show(
        [
            mm.read("tmp/fig_03_gauss_0.png"),
            mm.read("tmp/fig_03_gauss_1.png"),
            mm.read("tmp/fig_03_gauss_2.png"),
        ],
        titles=[
            'Detalhe: Original',
            'Média 9×9',
            'Gaussiano 9×9',
        ],
        cols=3,
        figsize=(12, 8),
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_gauss_0.png (ver a versao Python)")

<Figure size 1800x1200 with 3 Axes>

**Figura 3.14:** Comparação entre filtro de média e Gaussiano (*kernel* 9×9, σ=0). O Gaussiano preserva melhor as bordas, visível no detalhe do rosto.


## 3.6 Filtragem Espacial de Realce

Os filtros de realce (*sharpening filters*) enfatizam transições abruptas de intensidade, aumentando a nitidez e a visibilidade de bordas. São filtros **passa-alta** — amplificam as componentes de alta frequência (bordas, textura) e suprimem as de baixa frequência (regiões uniformes).

A intuição é simples: se subtrairmos de uma imagem sua versão suavizada (que contém apenas as baixas frequências), o que resta são as altas frequências — bordas e detalhes. Somando esse resíduo de volta à imagem original, o contraste local aumenta:

<a id="eq-03-realce-intuitivo"></a>
$$
g = f + k\,(f - f_{\text{suave}}), \quad k > 0 \tag{3.15}
$$


O [Figura 3.15](#fig-03-sim-03-filtragem1d) ilustra esse processo em um sinal 1D sintético com três estruturas distintas: um degrau largo, um pico fino e uma rampa suave. No painel ①, o sinal original $f(x)$; no ②, a versão suavizada $f_{\text{suave}}(x)$ obtida por média móvel — note como o pico fino é atenuado. O painel ③ exibe o resíduo $f - f_{\text{suave}}$, que retém apenas as transições abruptas. Por fim, o painel ④ mostra $g(x)$: o pico, antes atenuado, é restaurado e amplificado em relação ao original. Ajuste $k$ e o tamanho da janela para observar o *trade-off* entre nitidez e amplificação de ruído.

Os filtros de realce formalizam essa ideia diretamente no *kernel*, sem precisar de duas etapas separadas.

In [52]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-03-filtragem1d" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🎮 Simulador: Filtragem Espacial de Realce 1D</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f + k·(f − f_suave)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">

    <!-- Controles -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;flex-wrap:wrap;gap:16px;align-items:flex-end;">
      
      <div style="display:flex;flex-direction:column;gap:4px;flex:1;min-width:140px;">
        <label style="font-size:11px;font-weight:700;color:#b9770e;">Parâmetro k (Realce)</label>
        <input id="sim_cap03_k_slider" type="range" min="0" max="5" step="0.1" value="1.5" style="width:100%;cursor:pointer;">
        <span style="font-size:11px;font-family:monospace;font-weight:700;color:#26241d;">k = <span id="sim_cap03_k_val">1.5</span></span>
      </div>

      <div style="display:flex;flex-direction:column;gap:4px;flex:1;min-width:140px;">
        <label style="font-size:11px;font-weight:700;color:#2980b9;">Janela de Suavização (pts)</label>
        <input id="sim_cap03_win_slider" type="range" min="3" max="31" step="2" value="9" style="width:100%;cursor:pointer;">
        <span style="font-size:11px;font-family:monospace;font-weight:700;color:#26241d;">janela = <span id="sim_cap03_win_val">9</span></span>
      </div>

      <div style="display:flex;flex-direction:column;gap:4px;flex:1;min-width:140px;">
        <label style="font-size:11px;font-weight:700;color:#8e44ad;">Nível de Ruído σ</label>
        <input id="sim_cap03_noise_slider" type="range" min="0" max="0.3" step="0.01" value="0.04" style="width:100%;cursor:pointer;">
        <span style="font-size:11px;font-family:monospace;font-weight:700;color:#26241d;">σ = <span id="sim_cap03_noise_val">0.04</span></span>
      </div>

    </div>

    <!-- Gráficos com Fundos Pastéis -->
    <div style="display:flex;flex-direction:column;gap:6px;">

      <!-- ① Azul Pastéis -->
      <div style="background:#ebf4fd;border:1px solid #d4e5f7;border-radius:8px;padding:8px 10px;">
        <div style="font-size:11px;font-weight:700;color:#1a5fa8;margin-bottom:3px;font-family:monospace;">① f(x) — Sinal Original</div>
        <canvas id="sim_cap03_c1" height="80" style="width:100%;display:block;"></canvas>
      </div>

      <div style="text-align:center;font-size:14px;color:#8a8371;line-height:1;">↓ filtro passa-baixa (média móvel)</div>

      <!-- ② Verde Pastéis -->
      <div style="background:#eaf7f2;border:1px solid #d1efe3;border-radius:8px;padding:8px 10px;">
        <div style="font-size:11px;font-weight:700;color:#0d6b4f;margin-bottom:3px;font-family:monospace;">② f_suave(x) — Pico Atenuado pelo Filtro</div>
        <canvas id="sim_cap03_c2" height="80" style="width:100%;display:block;"></canvas>
      </div>

      <div style="text-align:center;font-size:14px;color:#8a8371;line-height:1;">↓ subtração: f − f_suave</div>

      <!-- ③ Salmão/Rosa Pastéis -->
      <div style="background:#fef0eb;border:1px solid #fcdad0;border-radius:8px;padding:8px 10px;">
        <div style="font-size:11px;font-weight:700;color:#a03010;margin-bottom:3px;font-family:monospace;">③ Resíduo (f − f_suave) — Altas Frequências / Bordas</div>
        <canvas id="sim_cap03_c3" height="80" style="width:100%;display:block;"></canvas>
      </div>

      <div style="text-align:center;font-size:14px;color:#8a8371;line-height:1;">↓ soma: f + k · resíduo</div>

      <!-- ④ Roxo Pastéis -->
      <div style="background:#f2f0fd;border:1px solid #e1dcf9;border-radius:8px;padding:8px 10px;">
        <div style="font-size:11px;font-weight:700;color:#4a3faa;margin-bottom:3px;font-family:monospace;">④ g(x) — Sinal com Pico Realçado</div>
        <canvas id="sim_cap03_c4" height="80" style="width:100%;display:block;"></canvas>
      </div>

    </div>

  </div>
</div>

<script>
(function(){
  function initSimCap03(root){
    if (!root || root.dataset.simCap03Init) return;
    root.dataset.simCap03Init = "1";

    var N = 250;
    var peak_i = 115;

    var slK     = root.querySelector('#sim_cap03_k_slider');
    var slWin   = root.querySelector('#sim_cap03_win_slider');
    var slNoise = root.querySelector('#sim_cap03_noise_slider');

    var valK     = root.querySelector('#sim_cap03_k_val');
    var valWin   = root.querySelector('#sim_cap03_win_val');
    var valNoise = root.querySelector('#sim_cap03_noise_val');

    function make_signal(noise) {
      var seed = 42;
      function rand() { seed = (seed * 9301 + 49297) % 233280; return seed / 233280; }
      function randn() { return Math.sqrt(-2 * Math.log(rand() + 1e-9)) * Math.cos(2 * Math.PI * rand()); }
      var f = [];
      for (var i = 0; i < N; i++) {
        var v = 0.2;
        if (i >= 30  && i <= 80)  v += 0.7;
        if (i >= 110 && i <= 120) v += 1.0;
        if (i >= 150 && i <= 190) v += 0.5 * (i - 150) / 40;
        v += noise * randn();
        f.push(v);
      }
      return f;
    }

    function moving_avg(f, win) {
      var half = Math.floor(win / 2);
      return f.map(function(_, i){
        var s = 0, c = 0;
        for (var j = Math.max(0, i - half); j <= Math.min(f.length - 1, i + half); j++) {
          s += f[j];
          c++;
        }
        return s / c;
      });
    }

    function drawCanvas(canvasId, datasets, annotations) {
      var canvas = root.querySelector('#' + canvasId);
      if (!canvas) return;
      canvas.width = canvas.offsetWidth || 800;
      var ctx = canvas.getContext('2d');
      var W = canvas.width, H = canvas.height;
      ctx.clearRect(0, 0, W, H);

      var padL = 8, padR = 8, padT = 6, padB = 6;
      var allVals = [].concat.apply([], datasets.map(function(d){ return d.data; }));
      var mn = Math.min.apply(null, allVals);
      var mx = Math.max.apply(null, allVals);
      var rng = mx - mn || 1;
      var drawH = H - padT - padB, drawW = W - padL - padR;

      function toY(v){ return H - padB - (v - mn) / rng * drawH; }
      function toX(i){ return padL + i / (N - 1) * drawW; }

      // Faixa de destaque do pico
      var xA = toX(108), xB = toX(122);
      ctx.fillStyle = 'rgba(255, 200, 50, 0.15)';
      ctx.fillRect(xA, padT, xB - xA, drawH);

      // Grade
      ctx.strokeStyle = 'rgba(38, 36, 29, 0.08)';
      ctx.lineWidth = 0.5;
      for (var g = 0; g <= 3; g++) {
        var gy = padT + g / 3 * drawH;
        ctx.beginPath(); ctx.moveTo(padL, gy); ctx.lineTo(W - padR, gy); ctx.stroke();
      }

      // Linha do zero
      if (mn < 0 && mx > 0) {
        ctx.strokeStyle = 'rgba(38, 36, 29, 0.2)';
        ctx.lineWidth = 0.8;
        ctx.setLineDash([4, 3]);
        var y0 = toY(0);
        ctx.beginPath(); ctx.moveTo(padL, y0); ctx.lineTo(W - padR, y0); ctx.stroke();
        ctx.setLineDash([]);
      }

      // Desenho das séries
      datasets.forEach(function(ds){
        ctx.strokeStyle = ds.color;
        ctx.lineWidth   = ds.width || 1.8;
        ctx.globalAlpha = ds.alpha || 1;
        if (ds.dash) ctx.setLineDash(ds.dash); else ctx.setLineDash([]);
        ctx.beginPath();
        ds.data.forEach(function(v, i){
          if (i === 0) ctx.moveTo(toX(i), toY(v)); else ctx.lineTo(toX(i), toY(v));
        });
        ctx.stroke();
        ctx.setLineDash([]);
        ctx.globalAlpha = 1;
      });

      // Anotações
      if (annotations) {
        annotations.forEach(function(an){
          var xi = toX(an.i), yi = toY(an.v);
          ctx.strokeStyle = an.color || '#5e5a4a';
          ctx.lineWidth = 1;
          ctx.setLineDash([3, 3]);
          ctx.beginPath(); ctx.moveTo(xi, padT); ctx.lineTo(xi, H - padB); ctx.stroke();
          ctx.setLineDash([]);

          ctx.fillStyle = an.color || '#5e5a4a';
          ctx.beginPath(); ctx.arc(xi, yi, 4, 0, 2 * Math.PI); ctx.fill();

          ctx.fillStyle = an.color || '#26241d';
          ctx.font = 'bold 10px monospace';
          ctx.fillText(an.label, xi + 6, Math.max(padT + 12, Math.min(H - padB - 4, yi - 6)));
        });
      }
    }

    function update() {
      var k     = parseFloat(slK.value) || 0;
      var win   = parseInt(slWin.value) || 3;
      var noise = parseFloat(slNoise.value) || 0;

      valK.textContent     = k.toFixed(1);
      valWin.textContent   = win;
      valNoise.textContent = noise.toFixed(2);

      var f       = make_signal(noise);
      var f_suave = moving_avg(f, win);
      var residuo = f.map(function(v, i){ return v - f_suave[i]; });
      var g       = f.map(function(v, i){ return v + k * residuo[i]; });

      var pk = peak_i;
      var peakF = f[pk], peakS = f_suave[pk], peakR = residuo[pk], peakG = g[pk];

      // ① Original (Azul)
      drawCanvas('sim_cap03_c1',
        [{ data: f, color: '#1a5fa8' }],
        [{ i: pk, v: peakF, color: '#1a5fa8', label: 'pico' }]
      );

      // ② Suavizado (Verde)
      drawCanvas('sim_cap03_c2',
        [{ data: f, color: '#1a5fa8', width: 1, alpha: 0.35, dash: [4, 3] },
         { data: f_suave, color: '#0d6b4f', width: 2 }],
        [{ i: pk, v: peakS, color: '#0d6b4f', label: 'atenuado' }]
      );

      // ③ Resíduo (Salmão / Laranja)
      drawCanvas('sim_cap03_c3',
        [{ data: residuo, color: '#a03010' }],
        [{ i: pk, v: peakR, color: '#a03010', label: 'borda detectada' }]
      );

      // ④ Realçado (Roxo)
      drawCanvas('sim_cap03_c4',
        [{ data: f, color: '#1a5fa8', width: 1, alpha: 0.35, dash: [4, 3] },
         { data: g, color: '#4a3faa', width: 2.2 }],
        [{ i: pk, v: peakG, color: '#4a3faa', label: 'pico realçado' }]
      );
    }

    [slK, slWin, slNoise].forEach(function(sl){
      sl.addEventListener('input', update);
    });

    update();
  }

  function tryInitSimCap03(){
    var root = document.getElementById('sim-03-filtragem1d');
    if (root) initSimCap03(root); else setTimeout(tryInitSimCap03, 200);
  }
  tryInitSimCap03();
})();
</script>
</div>
""")

**Figura 3.15:** Simulador: Filtragem Espacial de Realce 1D (Unsharp Masking e High-Boost)


<figure id="fig-03-sim-03-filtragem1d">
  <img src="imagens/fig-03-sim-03-filtragem1d.png" alt=" Simulador: Filtragem Espacial de Realce 1D (Unsharp Masking e High-Boost) " style="max-width:80%" />
  <figcaption><strong>Figura 3.15:</strong>  Simulador: Filtragem Espacial de Realce 1D (Unsharp Masking e High-Boost) </figcaption>
</figure>

### 3.6.1 Laplaciano

O Laplaciano é um operador de **segunda derivada** isotrópico, ou seja, responde igualmente a variações em todas as direções, ao contrário de operadores de primeira derivada, como Sobel e Prewitt, que são direcionais:

<a id="eq-03-laplaciano"></a>
$$
\nabla^2 f = \frac{\partial^2 f}{\partial x^2} + \frac{\partial^2 f}{\partial y^2} \tag{3.16}
$$


Uma propriedade importante da segunda derivada é que seu valor é **próximo de zero em regiões uniformes** e elevado nas transições de intensidade. Assim, ao subtrair o Laplaciano da imagem original, reforçam-se bordas e detalhes, aumentando o contraste local:

<a id="eq-03-realce-lap"></a>
$$
g(x,y) = f(x,y) - \nabla^2 f(x,y) \tag{3.17}
$$


Na forma discreta, a segunda derivada em $x$ é aproximada por $f(x+1,y) - 2f(x,y) + f(x-1,y)$, e analogamente em $y$. Somando as duas direções, obtém-se o *kernel* $w_4$ (4-vizinhos) ou $w_8$ (8-vizinhos, incluindo diagonais):

<a id="eq-03-laplaciano-kernel"></a>
$$
w_4 = \begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{bmatrix}, \qquad
w_8 = \begin{bmatrix} 1 & 1 & 1 \\ 1 & -8 & 1 \\ 1 & 1 & 1 \end{bmatrix} \tag{3.18}
$$


> ### 📝 Soma zero e centro negativo
>
> Ambos os *kernels* têm **soma dos coeficientes igual a zero**: em regiões uniformes, a saída é 0 — o Laplaciano não altera o brilho médio, apenas detecta variações. O **centro negativo** indica que o pixel é comparado com seus vizinhos: quanto mais ele se destacar (para cima ou para baixo), maior o valor absoluto do Laplaciano naquele ponto.

No exemplo a seguir, o pixel central $[1,1]=107$ possui vizinhos $\{95, 106, 108, 103\}$. Como esses valores são próximos entre si, a região é quase uniforme e o Laplaciano retorna um valor baixo, produzindo pouco realce. Em regiões de borda, onde há diferenças maiores entre o pixel central e seus vizinhos, o Laplaciano assume valores mais elevados (positivos ou negativos), e a operação de [Equação 3.17](#eq-03-realce-lap) intensifica essas transições.

A [Figura 3.16](#fig-03-laplaciano-patch) ilustra o cálculo do Laplaciano com o *kernel* $w_4$ em uma vizinhança $3\times3$ destacada dentro de um *patch* $5\times5$. O exemplo mostra o valor obtido pelo operador e o correspondente pixel realçado na imagem de saída, que passa de 107 para 123.

In [53]:
%%writefile tmp/fig_03_laplaciano_patch.cpp
#define MM_OUT "tmp/fig_03_laplaciano_patch.png"
//| label: fig-03-laplaciano-patch
//| fig-cap: "*Kernel* Laplaciano w4 sobre o *patch* 5×5: a janela amarela destaca a vizinhança 3×3 onde o operador de segunda derivada é calculado."
//| echo: true
//| output: true

#include <iostream>
#include <vector>
#include "morph.hpp"

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_45.png");
// [pdi:state-io:end]

    // patch = mm.crop(img_gray, 250, 255, 250, 255)
    mm::Image patch = mm::crop(img_gray, 250, 255, 250, 255);

    // B = np.ones((3, 3), dtype=np.uint8)
    mm::Kernel B(3, 3, 1.0);

    // print("Patch 5x5 (intensidades):")
    std::cout << "Patch 5x5 (intensidades):\n";

    // print(mm.drawImg(patch))
    std::cout << mm::drawImg(patch) << "\n";

    // mm.drawImgKernel(patch, B, 1, 1, 40)
    mm::drawImgKernel(patch, B, 1, 1, MM_OUT, 40);

    return 0;
}

Overwriting tmp/fig_03_laplaciano_patch.cpp


In [54]:
!g++ -I. -std=c++17 tmp/fig_03_laplaciano_patch.cpp -o tmp/fig_03_laplaciano_patch \
  && ./tmp/fig_03_laplaciano_patch \
  && test -f "tmp/fig_03_laplaciano_patch.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_laplaciano_patch.png"

Patch 5x5 (intensidades):
 94  96 103 113 115 
107 104 103 107 114 
 98 102 112 117 116 
 81  91 112 122 118 
 85  91 110 119 121 

Processando pixel (x,y)=(1,1)  |  janela do kernel 3x3
 94  96 103 113 115 
107 104 103 107 114 
 98 102 112 117 116 
 81  91 112 122 118 
 85  91 110 119 121 


In [55]:
try:
    mm.show(mm.read("tmp/fig_03_laplaciano_patch.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_laplaciano_patch.png (ver a versao Python)")

<Figure size 450x450 with 1 Axes>

**Figura 3.16:** *Kernel* Laplaciano w4 sobre o *patch* 5×5: a janela amarela destaca a vizinhança 3×3 onde o operador de segunda derivada é calculado.


A [Figura 3.17](#fig-03-laplaciano) compara a aplicação dos *kernels* Laplacianos $w_4$ e $w_8$ em um recorte maior da imagem do leopardo. Para cada caso, são mostradas a resposta bruta do operador, que evidencia as bordas e transições de intensidade, e a imagem obtida após o realce por subtração do Laplaciano. Observa-se que o *kernel* $w_8$, por considerar também os vizinhos diagonais, produz uma resposta mais intensa e detecta variações em mais direções, resultando em um realce ligeiramente mais acentuado.

In [56]:
%%writefile tmp/fig_03_laplaciano.cpp
#define MM_OUT "tmp/fig_03_laplaciano.png"
#include "morph.hpp"
#include <iostream>
#include <vector>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    // As variáveis img_gray_crop já estão inicializadas aqui

    // Kernel Laplaciano w4 (vizinhança 4)
    mm::Kernel w4{{0, 1, 0},
                  {1, -4, 1},
                  {0, 1, 0}};

    // Kernel Laplaciano w8 (vizinhança 8)
    mm::Kernel w8{{1, 1, 1},
                  {1, -8, 1},
                  {1, 1, 1}};

    mm::show(
        std::vector<mm::Image>{
            img_gray_crop,
            mm::laplacian_viz(img_gray_crop, w4),
            mm::laplacian(img_gray_crop, w4),
            img_gray_crop,
            mm::laplacian_viz(img_gray_crop, w8),
            mm::laplacian(img_gray_crop, w8)
        },
        MM_OUT,
        {"Original", "Laplaciano w4", "Realce w4",
         "Original", "Laplaciano w8", "Realce w8"},
        3
    );

    return 0;
}

Overwriting tmp/fig_03_laplaciano.cpp


In [57]:
!g++ -I. -std=c++17 tmp/fig_03_laplaciano.cpp -o tmp/fig_03_laplaciano \
  && ./tmp/fig_03_laplaciano \
  && test -f "tmp/fig_03_laplaciano.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_laplaciano.png"

[1] Original
[2] Laplaciano w4
[3] Realce w4
[4] Original
[5] Laplaciano w8
[6] Realce w8


In [58]:
try:
    mm.show(mm.read("tmp/fig_03_laplaciano.png"), figsize=(14, 8))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_laplaciano.png (ver a versao Python)")

<Figure size 2100x1200 with 1 Axes>

**Figura 3.17:** Laplaciano aplicado à imagem do leopardo: resposta bruta (bordas) com w4 e w8, e imagens realçadas pela subtração do Laplaciano. w8 é mais sensível às diagonais.


### 3.6.2 Operador de Sobel

O operador de Sobel estima as **derivadas parciais de primeira ordem** nas direções horizontal e vertical. Diferente do Laplaciano (segunda derivada), o Sobel é direcional e mais robusto ao ruído, pois cada *kernel* combina uma derivada com uma suavização Gaussiana perpendicular:

<a id="eq-03-sobel"></a>
$$
G_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix} * f, \qquad
G_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{bmatrix} * f \tag{3.19}
$$


$G_x$ detecta bordas **verticais** (variação na direção $x$); $G_y$ detecta bordas **horizontais** (variação na direção $y$). Os pesos $\{1,2,1\}$ na direção perpendicular correspondem à suavização Gaussiana 1D, que reduz a sensibilidade ao ruído.

> ### 📝 Sobel é correlação, não convolução
>
> Os *kernels* de Sobel são **assimétricos** — a rotação de 180° altera o resultado. `cv2.Sobel` implementa correlação cruzada (como `cv2.filter2D`). Para obter a derivada direcional correta, os sinais já estão definidos para correlação: $G_x$ retorna valores positivos onde a intensidade cresce da esquerda para a direita.

A magnitude do **gradiente** combina os dois componentes, representando a força da borda independente de direção:

<a id="eq-03-gradiente"></a>
$$
|\nabla f| = \sqrt{G_x^2 + G_y^2} \tag{3.20}
$$


E a **direção** do gradiente (perpendicular à borda) é:

<a id="eq-03-direcao"></a>
$$
\theta = \arctan\left(\frac{G_y}{G_x}\right) \tag{3.21}
$$


Para ilustrar numericamente, calcula-se $G_x$ e $G_y$ manualmente no pixel central $[1,1]$ do *patch* 5×5:

O valor reduzido de $|{\nabla f}|$ nesse *patch* confirma que a região é quase uniforme, pois o gradiente assume valores elevados apenas onde há mudanças significativas de intensidade. A [Figura 3.18](#fig-03-sobel) aplica o operador de Sobel a um recorte maior da imagem do leopardo. São apresentadas as respostas horizontal ($G_x$) e vertical ($G_y$), obtidas por convolução com os respectivos *kernels* de Sobel, além da magnitude $|{\nabla f}|$, calculada a partir da combinação de ambas. Enquanto $G_x$ destaca bordas verticais e $G_y$ bordas horizontais, a magnitude evidencia bordas em qualquer direção.

In [59]:
%%writefile tmp/fig_03_sobel.cpp
#define MM_OUT "tmp/fig_03_sobel.png"
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    // Operador de Sobel na imagem do leopardo: a magnitude |∇f| combina os
    // gradientes horizontal e vertical, revelando todas as bordas. A
    // decomposição Gx/Gy com sinal fica na trilha Python — mm::sobel devolve
    // a magnitude já com clip.
    mm::Image mag = mm::sobel(img_gray_crop);

    mm::show({img_gray_crop, mag}, MM_OUT,
             {"Original", "Magnitude |grad f| (mm.sobel)"}, 2);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray_crop, "tmp/fig_03_sobel_0.png");
mm::write(mag, "tmp/fig_03_sobel_1.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_03_sobel.cpp


In [60]:
!g++ -I. -std=c++17 tmp/fig_03_sobel.cpp -o tmp/fig_03_sobel \
  && ./tmp/fig_03_sobel \
  && test -f "tmp/fig_03_sobel.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_sobel.png"

[1] Original
[2] Magnitude |grad f| (mm.sobel)


In [61]:
try:
    mm.show(
        [
            mm.read("tmp/fig_03_sobel_0.png"),
            mm.read("tmp/fig_03_sobel_1.png"),
        ],
        titles=[
            'Original',
            'Magnitude |grad f| (mm.sobel)',
        ],
        cols=2,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_sobel_0.png (ver a versao Python)")

<Figure size 1500x750 with 2 Axes>

**Figura 3.18:** Operador de Sobel na imagem do leopardo: a magnitude |∇f| combina os gradientes horizontal e vertical, revelando todas as bordas. A decomposição Gx/Gy com sinal fica na trilha Python — mm::sobel devolve a magnitude já com clip.


### 3.6.3 Operador de Prewitt

O operador de Prewitt é estruturalmente idêntico ao Sobel, mas substitui a ponderação Gaussiana $\{1,2,1\}$ por pesos uniformes $\{1,1,1\}$:

<a id="eq-03-prewitt"></a>
$$
G_x = \begin{bmatrix} -1 & 0 & 1 \\ -1 & 0 & 1 \\ -1 & 0 & 1 \end{bmatrix} * f, \qquad
G_y = \begin{bmatrix} -1 & -1 & -1 \\ 0 & 0 & 0 \\ 1 & 1 & 1 \end{bmatrix} * f \tag{3.22}
$$


A magnitude e a direção do gradiente seguem as mesmas equações do Sobel ([Equação 3.20](#eq-03-gradiente) e [Equação 3.21](#eq-03-direcao)). A diferença prática é que o Prewitt é ligeiramente mais sensível ao ruído — a suavização perpendicular uniforme pondera menos o pixel central da linha — mas computacionalmente mais simples. Em imagens com baixo ruído os resultados são equivalentes.

In [62]:
%%writefile tmp/fig_03_prewitt.cpp
#define MM_OUT "tmp/fig_03_prewitt.png"
// Compile: g++ -std=c++17 -o program program.cpp -I. -L. -lmorph

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    // Pré-condição: img_gray_crop já está inicializado como mm::Image
    // (inserido automaticamente)

    // | label: fig-03-prewitt
    // | fig-cap: "Operador de Prewitt: magnitude do gradiente, comparável ao Sobel mas sem a ponderação central. mm::prewitt devolve |∇f| com clip."
    // | echo: true
    // | output: true

    mm::show(std::vector<mm::Image>{img_gray_crop, mm::prewitt(img_gray_crop)},
             MM_OUT,
             std::vector<std::string>{"Original", "Magnitude |grad f| (mm.prewitt)"},
             2);

    return 0;
}

Overwriting tmp/fig_03_prewitt.cpp


In [63]:
!g++ -I. -std=c++17 tmp/fig_03_prewitt.cpp -o tmp/fig_03_prewitt \
  && ./tmp/fig_03_prewitt \
  && test -f "tmp/fig_03_prewitt.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_prewitt.png"

[1] Original
[2] Magnitude |grad f| (mm.prewitt)


In [64]:
try:
    mm.show(mm.read("tmp/fig_03_prewitt.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_prewitt.png (ver a versao Python)")

<Figure size 524x450 with 1 Axes>

**Figura 3.19:** Operador de Prewitt: magnitude do gradiente, comparável ao Sobel mas sem a ponderação central. mm::prewitt devolve |∇f| com clip.


### 3.6.4 *Unsharp Masking* (USM)

O *Unsharp Masking* é uma técnica clássica de realce de nitidez originária da fotografia analógica, hoje amplamente usada em software de edição de imagens. A ideia central é extrair as **componentes de alta frequência** da imagem (bordas e detalhes) e somá-las de volta à original com um peso $k$:

<a id="tbl-03-usm"></a>

**Tabela 3.3:** Etapas do *Unsharp Masking*.

| Etapa | Operação | Descrição |
|:-----:|:---------|:----------|
| 1 | $\bar{f} = f * G_\sigma$ | Suaviza com Gaussiana — retém baixas frequências |
| 2 | $m = f - \bar{f}$ | Máscara: diferença = altas frequências (bordas) |
| 3 | $g = f + k \cdot m$ | Soma ponderada da máscara à original |


Substituindo a etapa 2 na etapa 3, obtém-se a expressão compacta:

<a id="eq-03-usm"></a>
$$
g = f + k\,(f - f*G_\sigma) = (1+k)\,f - k\,(f*G_\sigma) \tag{3.23}
$$


O parâmetro $k$ controla a intensidade do realce:

- $k = 0$: sem realce ($g = f$);
- $k = 1$: USM clássico — duplica a contribuição das altas frequências;
- $k > 1$: *High Boost Filtering* — amplificação além do dobro, útil para imagens muito borradas.

> ### ⚠️ Amplificação de ruído
>
> O USM não distingue bordas de ruído — ambos são componentes de alta frequência. Para $k$ elevado, o ruído presente na imagem é amplificado junto com as bordas. Por isso, é recomendável aplicar uma leve suavização antes do USM em imagens ruidosas, ou usar $\sigma$ pequeno na Gaussiana.

Para ilustrar as etapas do USM, a [Figura 3.20](#fig-03-usm2) aplica o método a um *patch* $30\times30$ da imagem do leopardo, utilizando $\sigma=1$ e $k=1$. Inicialmente, a imagem é suavizada por um filtro Gaussiano. Em seguida, a máscara de alta frequência é obtida pela diferença entre a imagem original e a suavizada. Por fim, essa máscara é somada à imagem original, reforçando bordas e detalhes. A figura apresenta as três etapas do processo e o resultado final do realce.

In [65]:
%%writefile tmp/fig_03_usm2.cpp
#define MM_OUT "tmp/fig_03_usm2.png"
//| label: fig-03-usm2
//| fig-cap: "Realce por *Unsharp Masking* num *patch* do leopardo: mm::usm faz suavização Gaussiana, subtrai da original (máscara de alta frequência) e reintroduz a máscara realçada."
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    mm::Image patch = mm::crop(img_gray_crop, 35, 65, 45, 75);

    mm::Image p_suave = mm::gaussian(patch, 7, 1.0);   // suavização Gaussiana
    mm::Image p_usm   = mm::usm(patch, 1.0);           // realce completo (k = 1.0)

    mm::show(std::vector<mm::Image>{patch, p_suave, p_usm},
             MM_OUT,
             std::vector<std::string>{"Patch original", "Suavizado (σ=1)", "Realçado USM (k=1)"}, 3);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(patch, "tmp/fig_03_usm2_0.png");
mm::write(p_suave, "tmp/fig_03_usm2_1.png");
mm::write(p_usm, "tmp/fig_03_usm2_2.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_03_usm2.cpp


In [66]:
!g++ -I. -std=c++17 tmp/fig_03_usm2.cpp -o tmp/fig_03_usm2 \
  && ./tmp/fig_03_usm2 \
  && test -f "tmp/fig_03_usm2.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_usm2.png"

[1] Patch original
[2] Suavizado (σ=1)
[3] Realçado USM (k=1)


In [67]:
try:
    mm.show(
        [
            mm.read("tmp/fig_03_usm2_0.png"),
            mm.read("tmp/fig_03_usm2_1.png"),
            mm.read("tmp/fig_03_usm2_2.png"),
        ],
        titles=[
            'Patch original',
            'Suavizado (σ=1)',
            'Realçado USM (k=1)',
        ],
        cols=3,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_usm2_0.png (ver a versao Python)")

<Figure size 2250x750 with 3 Axes>

**Figura 3.20:** Realce por *Unsharp Masking* num *patch* do leopardo: mm::usm faz suavização Gaussiana, subtrai da original (máscara de alta frequência) e reintroduz a máscara realçada.


A [Figura 3.21](#fig-03-usm) aplica o método USM a um recorte maior da imagem do leopardo utilizando $\sigma=1$ e diferentes valores do fator de ganho $k$. Em todos os casos, a máscara de alta frequência é obtida pela diferença entre a imagem original e sua versão suavizada por filtro Gaussiano. O parâmetro $k$ controla a intensidade do realce: valores menores produzem um aumento sutil de nitidez, enquanto valores maiores reforçam progressivamente bordas e detalhes. Observa-se que, para valores elevados de $k$, surgem halos ao redor das bordas e o ruído presente na imagem passa a ser amplificado.

In [68]:
%%writefile tmp/fig_03_usm.cpp
#define MM_OUT "tmp/fig_03_usm.png"
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    // Imagem original e versões com Unsharp Masking com diferentes valores de k
    mm::show(
        {img_gray_crop,
         mm::usm(img_gray_crop, 0.5), mm::usm(img_gray_crop, 1.0),
         mm::usm(img_gray_crop, 3.0), mm::usm(img_gray_crop, 5.0),
         mm::usm(img_gray_crop, 8.0)},
        MM_OUT,
        {"Original", "USM k=0.5", "USM k=1.0", "USM k=3.0", "USM k=5.0", "USM k=8.0"},
        3
    );

    return 0;
}

Overwriting tmp/fig_03_usm.cpp


In [69]:
!g++ -I. -std=c++17 tmp/fig_03_usm.cpp -o tmp/fig_03_usm \
  && ./tmp/fig_03_usm \
  && test -f "tmp/fig_03_usm.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_usm.png"

[1] Original
[2] USM k=0.5
[3] USM k=1.0
[4] USM k=3.0
[5] USM k=5.0
[6] USM k=8.0


In [70]:
try:
    mm.show(mm.read("tmp/fig_03_usm.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_usm.png (ver a versao Python)")

<Figure size 784x524 with 1 Axes>

**Figura 3.21:** *Unsharp Masking* na imagem do leopardo com σ=1 e k de 0.5 a 8.0. Para k>2 surgem halos nas bordas e o ruído de fundo aparece.


### 3.6.5 Detector de Canny

O Canny combina quatro etapas em sequência — suavização Gaussiana, gradiente de Sobel,
supressão de não-máximos e histerese por duplo limiar — para produzir bordas **finas,
binárias e conectadas**. Diferente de Sobel e Prewitt, o resultado não é um mapa de
gradiente contínuo, mas uma máscara onde cada pixel é borda ou não.

O parâmetro central é o par de limiares $(T_{low}, T_{high})$. Pixels com gradiente acima de
$T_{high}$ são bordas certas; abaixo de $T_{low}$, descartados. Os pixels ambíguos —
entre os dois limiares — são decididos por **histerese**: tornam-se borda se estiverem
conectados a uma borda certa, e descartados caso contrário. Isso evita tanto a perda de
trechos fracos de bordas reais quanto a inclusão de ruído isolado. Uma heurística comum
é $T_{high} = 3 \times T_{low}$.

In [71]:
%%writefile tmp/fig_03_canny.cpp
#define MM_OUT "tmp/fig_03_canny.png"
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    // img_gray_crop já está inicializado pela linha inserida automaticamente

    //| label: fig-03-canny
    //| fig-cap: "Detector de Canny com diferentes pares de limiar: limiares baixos capturam mais bordas (inclusive ruído); limiares altos retêm apenas as bordas mais fortes."
    //| echo: true
    //| output: true

    mm::show(
        std::vector<mm::Image>{img_gray_crop,
                 mm::canny(img_gray_crop, 30,  90),
                 mm::canny(img_gray_crop, 60,  180),
                 mm::canny(img_gray_crop, 120, 240)},
        MM_OUT,
        std::vector<std::string>{"Original", "Canny (30/90)", "Canny (60/180)", "Canny (120/240)"},
        4
    );

    return 0;
}

Overwriting tmp/fig_03_canny.cpp


In [72]:
!g++ -I. -std=c++17 tmp/fig_03_canny.cpp -o tmp/fig_03_canny \
  && ./tmp/fig_03_canny \
  && test -f "tmp/fig_03_canny.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_canny.png"

[1] Original
[2] Canny (30/90)
[3] Canny (60/180)
[4] Canny (120/240)


In [73]:
try:
    mm.show(mm.read("tmp/fig_03_canny.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_canny.png (ver a versao Python)")

<Figure size 1044x450 with 1 Axes>

**Figura 3.22:** Detector de Canny com diferentes pares de limiar: limiares baixos capturam mais bordas (inclusive ruído); limiares altos retêm apenas as bordas mais fortes.



> ### 📝 Escolha dos limiares
>
> Uma heurística comum é $T_{high} = 3 \times T_{low}$. Valores típicos dependem do intervalo de gradiente da imagem — `cv2.Canny` aceita valores absolutos em $[0, 255]$. Para imagens com contraste variável, calcular os limiares a partir de percentis da magnitude de Sobel é mais robusto do que valores fixos.

## 3.7 Filtros de Ordem: Filtro da Mediana

Os filtros de ordem (*order-statistic filters*) substituem o pixel central pelo valor de um **percentil** da distribuição de intensidades da vizinhança — ao contrário dos filtros lineares, que calculam combinações ponderadas. O mais importante é o **filtro da mediana**.

### 3.7.1 Ruído Impulsivo: Sal e Pimenta

O ruído **sal e pimenta** (*salt-and-pepper noise*) substitui pixels aleatórios por valores extremos: 0 (pimenta, preto) ou 255 (sal, branco). É comum em transmissão de imagens com erros de bit e em câmeras com sensores defeituosos.

Para entender por que os filtros lineares falham, considere uma vizinhança 3×3 onde um único pixel foi corrompido para 255:

$$
\text{vizinhança} = \begin{bmatrix} 102 & 98 & 105 \\ 100 & \mathbf{255} & 97 \\ 103 & 99 & 101 \end{bmatrix}
$$

<a id="tbl-03-mediana"></a>

**Tabela 3.4:** Média vs. mediana com um pixel corrompido. A mediana ignora o outlier; a média é deslocada ~40 níveis.

| Método | Cálculo | Resultado |
|:-------|:--------|----------:|
| Média | (102+98+...+255+...+101)/9 | ≈ 140 |
| Mediana | {97,98,99,100,**101**,102,103,105,255} | 101 |


> ### ⚠️ Por que filtros de média falham com ruído impulsivo?
>
> A média é sensível a ***outliers*** — um único pixel com valor 255 em uma vizinhança de valor ≈ 100 eleva a saída para ≈ 140, espalhando o ruído pela imagem. A mediana, por ser um **estimador robusto**, seleciona o valor central da distribuição ordenada, descartando naturalmente os extremos sem nenhum ajuste especial.

O exemplo a seguir ilustra o comportamento da média e da mediana na presença de um pixel corrompido por ruído impulsivo. Observa-se que a média é fortemente influenciada pelo valor extremo (255), produzindo uma estimativa distante dos valores predominantes da vizinhança. Já a mediana permanece próxima do valor original da região, evidenciando sua maior robustez a *outliers* e justificando seu uso na remoção de ruído sal e pimenta.

A [Figura 3.23](#fig-03-ruido) apresenta o efeito do ruído sal e pimenta em diferentes densidades. O ruído foi gerado substituindo aleatoriamente uma fração dos pixels por valores mínimos (0, pimenta) e máximos (255, sal). À medida que a densidade aumenta de 2% para 10%, cresce a quantidade de pixels corrompidos, tornando a degradação visual mais evidente e dificultando a percepção de detalhes da imagem.

In [74]:
%%writefile tmp/fig_03_ruido.cpp
#define MM_OUT "tmp/fig_03_ruido.png"
#include "morph.hpp"
#include <iostream>
#include <random>
#include <filesystem>

// Função para adicionar ruído sal e pimenta
mm::Image salt_pepper(const mm::Image& img, double prob) {
    mm::Image out = img;  // cópia da imagem
    int h = out.h;
    int w = out.w;

    std::mt19937 rng(42);  // gerador com semente fixa
    std::uniform_int_distribution<int> dist_y(0, h-1);
    std::uniform_int_distribution<int> dist_x(0, w-1);
    std::uniform_real_distribution<double> dist_prob(0.0, 1.0);

    int n = static_cast<int>(prob * h * w);

    for (int i = 0; i < n; ++i) {
        int y = dist_y(rng);
        int x = dist_x(rng);
        // 50% de chance de ser sal (255) ou pimenta (0)
        out.at(y, x) = (dist_prob(rng) < 0.5) ? 0 : 255;
    }

    return out;
}

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    // As variáveis img_gray_crop já estão inicializadas aqui

    // Aplica ruído sal e pimenta com diferentes densidades
    mm::Image n2  = salt_pepper(img_gray_crop, 0.02);
    mm::Image n5  = salt_pepper(img_gray_crop, 0.05);
    mm::Image n10 = salt_pepper(img_gray_crop, 0.10);

    // Exibe as imagens
    mm::show(std::vector<mm::Image>{img_gray_crop, n2, n5, n10},
             MM_OUT,
             std::vector<std::string>{"Original", "Ruido 2%", "Ruido 5%", "Ruido 10%"},
             4);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray_crop, "tmp/fig_03_ruido_0.png");
mm::write(n2, "tmp/fig_03_ruido_1.png");
mm::write(n5, "tmp/fig_03_ruido_2.png");
mm::write(n10, "tmp/fig_03_ruido_3.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_03_ruido.cpp


In [75]:
!g++ -I. -std=c++17 tmp/fig_03_ruido.cpp -o tmp/fig_03_ruido \
  && ./tmp/fig_03_ruido \
  && test -f "tmp/fig_03_ruido.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_ruido.png"

[1] Original
[2] Ruido 2%
[3] Ruido 5%
[4] Ruido 10%


In [76]:
try:
    mm.show(
        [
            mm.read("tmp/fig_03_ruido_0.png"),
            mm.read("tmp/fig_03_ruido_1.png"),
            mm.read("tmp/fig_03_ruido_2.png"),
            mm.read("tmp/fig_03_ruido_3.png"),
        ],
        titles=[
            'Original',
            'Ruido 2%',
            'Ruido 5%',
            'Ruido 10%',
        ],
        cols=4,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_ruido_0.png (ver a versao Python)")

<Figure size 3000x750 with 4 Axes>

**Figura 3.23:** Ruído sal e pimenta com densidades crescentes (2%, 5%, 10%): metade dos pixels corrompidos vira sal (255), metade pimenta (0).


### 3.7.2 Filtro da Mediana

O filtro da mediana substitui cada pixel pelo **valor mediano** dos pixels da sua vizinhança $n \times n$:

<a id="eq-03-mediana"></a>
$$
g(x,y) = \text{med}_{(s,t) \in \mathcal{V}_{n}} \{f(x+s, y+t)\} \tag{3.24}
$$


O valor mediano é aquele que ocupa a posição central quando os $n^2$ valores da vizinhança são ordenados. Para uma janela $3\times3$ ($n^2=9$ pixels), a mediana é o 5º valor da sequência ordenada.

Para ilustrar, considere o mesmo *patch* 5×5 com um pixel corrompido artificialmente em $[1,1]$:

O exemplo confirma: mesmo com o pixel corrompido a 255, a mediana retorna o valor central correto — o *outlier* ocupa a última posição na ordenação e é descartado naturalmente.

Por ser baseada em ordenação e não em soma, a mediana possui três propriedades fundamentais que a diferenciam dos filtros lineares:

- **Robusta** ao ruído impulsivo — outliers vão para as extremidades da sequência ordenada e não afetam o valor central;
- **Preservadora de bordas** — transições abruptas de intensidade são mantidas, pois a mediana seleciona um valor que já existe na vizinhança, sem criar novos níveis intermediários;
- **Não linear** — não pode ser expressa como convolução, portanto `mm::conv` não se aplica; usa-se `cv2.medianBlur`.

A [Figura 3.24](#fig-03-ruido-filtros) compara diferentes técnicas de remoção de ruído sal e pimenta aplicadas a uma imagem com 10% de pixels corrompidos. Foram avaliados os filtros Gaussiano, Média, Mediana, Bilateral e Morfológico (próximo capítulo), permitindo observar o compromisso entre remoção de ruído e preservação de detalhes. Em geral, os filtros de média e Gaussiano reduzem o ruído, mas tendem a borrar as bordas, enquanto a mediana apresenta melhor desempenho para ruído impulsivo. O filtro bilateral preserva melhor as bordas, e o filtro morfológico remove boa parte dos pixels corrompidos sem degradar excessivamente a estrutura da imagem.

In [77]:
%%writefile tmp/fig_03_ruido_filtros.cpp
#define MM_OUT "tmp/fig_03_ruido_filtros.png"
#include "morph.hpp"
#include <random>
#include <vector>
#include <string>
#include <filesystem>

// Função para adicionar ruído sal e pimenta
mm::Image salt_pepper(const mm::Image& img, double prob) {
    mm::Image out = img;  // cópia
    int h = out.h, w = out.w;
    std::mt19937 rng(42);
    std::uniform_int_distribution<int> dist_y(0, h-1);
    std::uniform_int_distribution<int> dist_x(0, w-1);
    std::uniform_real_distribution<double> dist_prob(0.0, 1.0);

    for (int i = 0; i < static_cast<int>(prob * h * w); i++) {
        int y = dist_y(rng);
        int x = dist_x(rng);
        out.at(y, x) = (dist_prob(rng) < 0.5) ? 0 : 255;
    }
    return out;
}

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    // img_gray_crop já está inicializado

    mm::Image noisy = salt_pepper(img_gray_crop, 0.10);

    mm::Image f_gauss   = mm::gaussian(noisy, 5, 1.0);
    mm::Image f_media   = mm::blur(noisy, 5);
    mm::Image f_median3 = mm::median(noisy, 3);
    mm::Image f_median5 = mm::median(noisy, 5);

    mm::show(
        std::vector<mm::Image>{img_gray_crop, noisy, f_gauss, f_media, f_median3, f_median5},
        MM_OUT,
        std::vector<std::string>{"Original", "Ruido 10%", "Gaussiano 5x5", "Media 5x5",
                                 "Mediana 3x3", "Mediana 5x5"},
        3
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray_crop, "tmp/fig_03_ruido_filtros_0.png");
mm::write(noisy, "tmp/fig_03_ruido_filtros_1.png");
mm::write(f_gauss, "tmp/fig_03_ruido_filtros_2.png");
mm::write(f_media, "tmp/fig_03_ruido_filtros_3.png");
mm::write(f_median3, "tmp/fig_03_ruido_filtros_4.png");
mm::write(f_median5, "tmp/fig_03_ruido_filtros_5.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_03_ruido_filtros.cpp


In [78]:
!g++ -I. -std=c++17 tmp/fig_03_ruido_filtros.cpp -o tmp/fig_03_ruido_filtros \
  && ./tmp/fig_03_ruido_filtros \
  && test -f "tmp/fig_03_ruido_filtros.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_ruido_filtros.png"

[1] Original
[2] Ruido 10%
[3] Gaussiano 5x5
[4] Media 5x5
[5] Mediana 3x3
[6] Mediana 5x5


In [79]:
try:
    mm.show(
        [
            mm.read("tmp/fig_03_ruido_filtros_0.png"),
            mm.read("tmp/fig_03_ruido_filtros_1.png"),
            mm.read("tmp/fig_03_ruido_filtros_2.png"),
            mm.read("tmp/fig_03_ruido_filtros_3.png"),
            mm.read("tmp/fig_03_ruido_filtros_4.png"),
            mm.read("tmp/fig_03_ruido_filtros_5.png"),
        ],
        titles=[
            'Original',
            'Ruido 10%',
            'Gaussiano 5x5',
            'Media 5x5',
            'Mediana 3x3',
            'Mediana 5x5',
        ],
        cols=3,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_ruido_filtros_0.png (ver a versao Python)")

<Figure size 2250x1500 with 6 Axes>

**Figura 3.24:** Filtros para ruído sal e pimenta (10%): Gaussiano, Média e Mediana. Bilateral e morfológico (open+close) ficam só na trilha Python — bilateral não tem equivalente em morph.hpp e morfologia é do próximo capítulo.


## 3.8 Aplicação Prática: Pré-processamento para Segmentação

Na prática, as técnicas deste capítulo raramente são usadas isoladamente. Um ***pipeline* de pré-processamento** típico combina várias etapas em sequência, adaptando-se ao tipo de imagem e à aplicação. A [Figura 3.25](#fig-03-pipeline) ilustra um *pipeline* completo:

1. **Equalização de histograma (CLAHE):** normaliza o contraste independente das condições de iluminação;
2. **Filtro Gaussiano:** suaviza ruído de aquisição sem destruir bordas;
3. **Detecção de bordas (Sobel/Canny):** extrai estruturas relevantes para segmentação.

> ### 📝 Ordem importa
>
> A ordem das operações afeta o resultado final. Em geral: **(1) normalização de intensidade → (2) redução de ruído → (3) realce/segmentação**. Inverter a ordem pode amplificar ruído ou perder bordas antes de detectá-las.

In [80]:
%%writefile tmp/fig_03_pipeline.cpp
#define MM_OUT "tmp/fig_03_pipeline.png"
// Compile: g++ -std=c++17 -o pipeline pipeline.cpp -I/path/to/morph.hpp

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    // img_gray_crop já está inicializado aqui

    mm::Image img_eq = mm::equalize(img_gray_crop);    // Etapa 1: equalização global
    mm::Image img_gauss = mm::gaussian(img_eq, 5, 0);  // Etapa 2: Gaussiano
    mm::Image edges = mm::canny(img_gauss, 50, 150);   // Etapa 3: Canny

    mm::Image edges_direct = mm::canny(img_gray_crop, 50, 150);  // Canny direto, sem pré-processo

    mm::show(
        std::vector<mm::Image>{img_gray_crop, img_eq, img_gauss, edges, edges_direct},
        MM_OUT,
        std::vector<std::string>{"Original", "1. Equalizado", "2. Gaussiano", "3. Canny (pipeline)", "Canny (direto)"},
        5
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray_crop, "tmp/fig_03_pipeline_0.png");
mm::write(img_eq, "tmp/fig_03_pipeline_1.png");
mm::write(img_gauss, "tmp/fig_03_pipeline_2.png");
mm::write(edges, "tmp/fig_03_pipeline_3.png");
mm::write(edges_direct, "tmp/fig_03_pipeline_4.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_03_pipeline.cpp


In [81]:
!g++ -I. -std=c++17 tmp/fig_03_pipeline.cpp -o tmp/fig_03_pipeline \
  && ./tmp/fig_03_pipeline \
  && test -f "tmp/fig_03_pipeline.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_pipeline.png"

[1] Original
[2] 1. Equalizado
[3] 2. Gaussiano
[4] 3. Canny (pipeline)
[5] Canny (direto)


In [82]:
try:
    mm.show(
        [
            mm.read("tmp/fig_03_pipeline_0.png"),
            mm.read("tmp/fig_03_pipeline_1.png"),
            mm.read("tmp/fig_03_pipeline_2.png"),
            mm.read("tmp/fig_03_pipeline_3.png"),
            mm.read("tmp/fig_03_pipeline_4.png"),
        ],
        titles=[
            'Original',
            '1. Equalizado',
            '2. Gaussiano',
            '3. Canny (pipeline)',
            'Canny (direto)',
        ],
        cols=5,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_pipeline_0.png (ver a versao Python)")

<Figure size 3750x750 with 5 Axes>

**Figura 3.25:** *Pipeline* de pré-processamento: equalização → Gaussiano → Canny. A trilha Python usa CLAHE no lugar da equalização global; CLAHE não tem equivalente em morph.hpp.


## 3.9 Resumo

Neste capítulo foram apresentadas as principais técnicas de processamento no domínio espacial, da manipulação direta de pixels até a filtragem por vizinhança:

- **Operações de ponto:** aritméticas saturadas (`mm::addm`, `mm::subm`) e lógicas bit a bit (`mm::band`, `mm::bor`, `mm::bnot`) para recorte de ROI e combinação de imagens; *alpha blending* (`mm::blend`) para fusão ponderada com peso $\alpha \in [0,1]$.
- **Histograma:** função discreta de distribuição de intensidades; visualizado com `mm::histImg` e calculado com `mm::hist`; base para diagnóstico tonal e para as técnicas de equalização e especificação.
- **Equalização:** redistribuição automática das intensidades pela CDF (`mm::equalize`), com variante adaptativa CLAHE para controle local do contraste.
- **Especificação de histograma:** transferência do perfil tonal de uma imagem de referência via mapeamento inverso da CDF — generalização da equalização para distribuições arbitrárias.
- **Correlação e convolução:** mecanismo de janela deslizante implementado em `mm::conv` (`cv2.filter2D`); diferenciados pela rotação de 180° do *kernel* — relevante apenas para kernels assimétricos.
- **Filtros de suavização:** média (*kernel* uniforme, borra bordas proporcionalmente ao tamanho) e Gaussiano (ponderação radial, separável, sem *ringing*, preserva melhor as bordas).
- **Filtros de realce:** Laplaciano ($w_4$/$w_8$, segunda derivada isotrópica), Sobel (gradiente direcional de primeira ordem, com magnitude $|\nabla f|$ e direção $\theta$) e *Unsharp Masking* (amplificação das altas frequências com parâmetro $k$).
- **Filtro da mediana:** não linear, robusto a outliers, preserva bordas — superior aos filtros lineares para ruído sal e pimenta.
- **Pipeline prático:** encadeamento CLAHE → Gaussiano → Canny como estratégia de pré-processamento; `mm::drawImgKernel` para visualização didática da janela deslizante.

O Capítulo 4 abordará a **morfologia matemática** (erosão, dilatação, abertura e fechamento), explorando em profundidade as funções `mm::ero` e `mm::dil` da biblioteca `morph.py`. Em seguida, o Capítulo 5 apresentará o **processamento no domínio da frequência**, com foco na Transformada de Fourier e em técnicas de filtragem espectral.

## 3.10 🤖 Uso do Gemini Notebook como Tutor Complementar

Nesta edição, incentivamos o uso do **Gemini Notebook** como ferramenta complementar de aprendizagem. Essa ferramenta de IA utiliza exclusivamente os documentos fornecidos pelo autor como base de conhecimento, garantindo respostas coerentes com o conteúdo do livro — incluindo as funções da biblioteca `morph.py` e os experimentos realizados neste capítulo.

Para cada capítulo, preparamos um projeto específico na plataforma com o PDF do capítulo, os *notebooks* e materiais auxiliares. Sugerimos explorar especialmente:

- **Guia de Estudo:** resumo estruturado dos conceitos, ideal para revisão antes de provas;
- **Conversa:** tire dúvidas sobre equalização, convolução, filtros e pipelines diretamente com o tutor;
- **Perguntas frequentes:** questões típicas sobre a diferença entre média e mediana, USM, Laplaciano vs. Sobel.

> ### ❗ 🎓 Estude com o Tutor Inteligente
>
> Para interagir com o conteúdo deste capítulo, acesse o *link* a seguir. O ambiente contém materiais didáticos em diferentes formatos, gerados a partir do **PDF** do capítulo. Na plataforma, explore especialmente as opções **Guia de Estudo** e **Conversa** para aprofundar sua compreensão.
>
> [🚀 ACESSAR Gemini Notebook: CAPÍTULO 03](https://notebooklm.google.com/notebook/d6593e26-a008-4d3b-8073-5c9b7d00eacc)
>
> #### 🌐 Idioma e Linguagem de Programação
>
> O projeto deste capítulo no Gemini Notebook foi construído apenas com o texto em **português** e os exemplos de código em **Python**. Se você está estudando pela edição em inglês ou francês, ou acompanhando a trilha em C++, as respostas do tutor podem não corresponder exatamente à versão que você está lendo.
>
> #### ⚠️ Aviso sobre Conteúdo Gerado por IA
>
> A IA é uma poderosa aliada nos estudos, mas o conteúdo gerado pode conter **erros ou imprecisões**. Consulte sempre **livros, artigos científicos e outras fontes acadêmicas confiáveis** para validar as informações. Sempre que possível, execute os exemplos práticos fornecidos neste capítulo para verificar os resultados.

## 3.11 Lista de Exercícios

1. **(10%)** Explique a diferença entre **convolução** e **correlação cruzada**. Para quais tipos de *kernel* os resultados são idênticos? Dê um exemplo de *kernel* assimétrico (como Sobel $G_x$) e mostre numericamente que os resultados diferem aplicando-o ao patch 5×5 do capítulo das duas formas.

2. **(15%)** Considere uma imagem 5×5 com intensidades concentradas entre os níveis 3 e 5 (baixo contraste, 3 bits). Aplique manualmente o algoritmo de equalização da [Tabela 3.1](#tbl-03-equalizacao), preenchendo todas as colunas da tabela ($k$, $h[k]$, $p[k]$, $\text{cdf}[k]$, $\text{lut}[k]$). Verifique o resultado com `mm::equalize`.

3. **(15%)** Usando `mm::conv`, aplique o filtro de média com kernels de tamanho 3×3, 9×9 e 21×21 à imagem do mandrill. Para cada versão, calcule o **PSNR** (*Peak Signal-to-Noise Ratio*) em relação à original:
$$\text{PSNR} = 10\log_{10}\!\left(\frac{255^2}{\text{MSE}}\right), \quad \text{MSE} = \frac{1}{MN}\sum_{i,j}(f-g)^2$$
Plote o PSNR em função do tamanho do kernel e explique o que a queda progressiva indica sobre a relação entre suavização e perda de informação.

4. **(15%)** Usando `add_salt_pepper` com densidade de 5%, aplique e compare: (a) `mm::conv` com média 3×3, (b) `cv2.GaussianBlur` com $\sigma=1$, (c) `cv2.medianBlur` com janela 3×3 e (d) `cv2.medianBlur` com janela 5×5. Exiba as imagens com `mm::show` em grade 2×4 (linha 1: imagens, linha 2: histogramas via `mm::histImg`). Explique por que a mediana supera os filtros lineares usando o argumento da [Tabela 3.4](#tbl-03-mediana).

5. **(15%)** Implemente `mm::conv0` usando apenas operações NumPy vetorizadas — sem laços Python e sem `cv2.filter2D` — com o operador de *stride tricks* (`np.lib.stride_tricks.sliding_window_view`). Compare o resultado e o tempo de execução com `mm::conv0` (laços) e `mm::conv` (cv2) para kernels 3×3 e 15×15 na imagem do mandrill.

6. **(15%)** Aplique o *Unsharp Masking* com $\sigma=1$ e $k \in \{0.5, 1.0, 2.0, 4.0\}$ usando a função `usm` do capítulo. Para cada valor de $k$: (a) calcule a diferença absoluta $|g - f|$, (b) exiba as imagens e as diferenças com `mm::show`, e (c) plote o histograma das diferenças com `mm::histImg`. Identifique a partir de qual $k$ os artefatos (halos e amplificação de ruído) tornam-se visualmente inaceitáveis.

7. **(15%)** Escolha uma imagem de raio-X ou tomografia disponível publicamente (ex.: via `mm::read` de URL) e projete um pipeline de pré-processamento com pelo menos 4 etapas sequenciais, justificando cada escolha com base nos conceitos do capítulo. Exiba com `mm::show` em grade: imagem original, cada etapa intermediária e o resultado final com seus histogramas (`mm::histImg`).

------------------------------------------------------------------------


<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cpp.pt/cap03/cap03.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

## 3.12 💻 **Parte Prática com Exercícios de Programação**


### 🎯 Objetivo deste Caderno

O caderno permite desenvolver, validar, organizar e testar soluções de **Exercícios de Programação (EPs)** em ambientes interativos, como o Colab, com os mesmos casos de teste do Moodle, copiando para lá apenas na hora de registrar a nota oficial.

#### *Download*

Baixe `morph.py` e `testsuite.py` executando a célula abaixo:

In [83]:
import os, urllib.request

os.makedirs("tmp/state", exist_ok=True)  # artefatos de build da trilha C++ (.cpp, binário, PNGs)

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

# O kernel é Python mesmo na trilha C++: `mm` (morph.py) é usado pelos
# simuladores, pela exibição das figuras que o binário C++ gera e pelo
# estado mm::Image entre células. cpp=True baixa também a trilha compilada
# (morph.hpp + stb_image*.h), usada no #include das células %%writefile *.cpp.
import config
config.setup(testsuite=True, cpp=True)
from morph import mm
from testsuite import TestSuite

✅ Ambiente pronto. Morph: 1.1.9 | OpenCV: 5.0.0 | TestSuite: 1.1.2


#### Executando os Testes
Para avaliar os testes, execute `TestSuite("EP03_01.extensão").run()` numa nova célula, trocando a extensão pela da linguagem usada (`.py`, `.java`, `.c`, `.cpp`, `.js` ou `.r`). O sistema baixa os casos de teste do GitHub, executa o programa e calcula a nota automaticamente.

Para testar código Python diretamente, sem salvar arquivo, use `run_code(codigo)` passando o código como *string* numa variável `codigo`:

```python
codigo = """
from morph import mm
# 3 ... seu código aqui ...
"""
TestSuite("EP03_01").run_code(codigo)
```


### 3.0.1 EP03_01 ➕ Adição Saturada de Constante

Em sistemas de vigilância por vídeo, câmeras em ambientes com iluminação variável produzem imagens subexpostas. O ajuste de brilho por **adição saturada de uma constante** é a operação mais simples para correção imediata, sendo aplicada em tempo real nos *chips* de câmeras embarcadas e em *pipelines* de pré-processamento de robôs móveis. 

Ver na [Figura 3.26](#fig-03-sim-ep0301-adicao) uma simulação deste EP.


#### 3.0.1.1 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Constante:** Ler o inteiro $k$ (valor a ser somado).
3. **Dados:** Ler os valores inteiros da matriz original linha a linha.
4. **Mapeamento:** Para cada pixel $p$, calcular o novo valor pela equação:

$$p' = \text{clip}(p + k)$$

5. **Saída:** Exibir a matriz resultante com dimensões $L \times C$.

#### 3.0.1.2 📌 Restrições Computacionais

* **Saturação (*Clipping*):** Os valores devem ser confinados ao intervalo $[0, 255]$:
$$\text{clip}(x) = \max(0, \min(255, x))$$
* **Tipo:** O resultado final deve ser inteiro (sem casas decimais).
* **$k$ pode ser negativo:** valores negativos escurecem a imagem; positivos clareiam.

#### 3.0.1.3 🧠 Fundamentação Teórica

| Parâmetro | Tipo | Impacto Visual |
|-----------|------|----------------|
| **$k > 0$** | Inteiro | Clareia a imagem; pixels próximos de 255 saturam em branco |
| **$k < 0$** | Inteiro | Escurece a imagem; pixels próximos de 0 saturam em preto |
| **$k = 0$** | Inteiro | Imagem inalterada |

#### 3.0.1.4 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $k$.
* Linhas seguintes: Elementos inteiros da matriz original.

**Saída:**

* Matriz transformada em $L$ linhas e $C$ colunas, valores inteiros separados por espaço.

#### 3.0.1.5 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 2<br>3<br>50<br>0 100 200<br>210 240 255 | 50 150 250<br>255 255 255 | Saturação em 255 nos pixels altos |
| 1<br>4<br>-30<br>0 20 200 255 | 0 0 170 225 | Saturação em 0 nos pixels baixos |

In [84]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0301-adicao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">➕ Simulador EP03_01: Adição Saturada de Constante</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">p' = clip(p + k)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Ajuste o valor da constante k para observar o deslocamento de brilho da imagem e o truncamento por saturação no intervalo [0, 255].</p>

    <!-- Controle da Constante k -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#27ae60;">Constante (k)</label>
        <span id="sim_ep0301_vl_k" style="font-family:monospace;font-size:12px;font-weight:700;color:#27ae60;">0</span>
      </div>
      <input type="range" id="sim_ep0301_sl_k" min="-128" max="128" step="1" value="0" style="width:100%;cursor:pointer;">
    </div>

    <!-- Comparativo Lado a Lado: Entrada vs Resultado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Entrada Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Entrada Original (p)</span>
        <div id="sim_ep0301_grid_orig" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0301_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nova Imagem</button>
      </div>

      <!-- Resultado Transformado -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Resultado Transformado (p')</span>
        <div id="sim_ep0301_grid_new" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0301_btnReset" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">↩ Resetar (k = 0)</button>
      </div>

    </div>

    <!-- Mensagem Explicativa Dinâmica -->
    <div id="sim_ep0301_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      Fórmula aplicada: <b>clip(p + (0))</b>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0301(root){
    if (!root || root.dataset.simEp0301Init) return;
    root.dataset.simEp0301Init = "1";

    var slK      = root.querySelector('#sim_ep0301_sl_k');
    var vlK      = root.querySelector('#sim_ep0301_vl_k');
    var gridOrig = root.querySelector('#sim_ep0301_grid_orig');
    var gridNew  = root.querySelector('#sim_ep0301_grid_new');
    var debugDiv = root.querySelector('#sim_ep0301_debug');

    var btnNew   = root.querySelector('#sim_ep0301_btnNew');
    var btnReset = root.querySelector('#sim_ep0301_btnReset');

    var pixels = Array(16).fill(0).map(function(){ return Math.floor(Math.random() * 256); });

    function render() {
      var k = parseInt(slK.value) || 0;
      vlK.textContent = k;
      debugDiv.innerHTML = 'Fórmula aplicada: <b>clip(p + (' + k + '))</b>';

      gridOrig.innerHTML = '';
      gridNew.innerHTML  = '';

      pixels.forEach(function(p) {
        // Célula Original
        var cellO = document.createElement('div');
        var fgColorO = p > 128 ? '#000000' : '#ffffff';
        cellO.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgColorO + ';box-sizing:border-box;';
        cellO.textContent = p;
        gridOrig.appendChild(cellO);

        // Célula Resultado
        var res = Math.max(0, Math.min(255, p + k));
        var fgColorN = res > 128 ? '#000000' : '#ffffff';
        var cellN = document.createElement('div');
        cellN.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + res + ',' + res + ',' + res + ');color:' + fgColorN + ';box-sizing:border-box;';
        cellN.textContent = res;
        gridNew.appendChild(cellN);
      });
    }

    slK.addEventListener('input', render);

    btnNew.addEventListener('click', function() {
      pixels = Array(16).fill(0).map(function(){ return Math.floor(Math.random() * 256); });
      render();
    });

    btnReset.addEventListener('click', function() {
      slK.value = 0;
      render();
    });

    render();
  }

  function tryInitSimEP0301(){
    var root = document.getElementById('sim-ep0301-adicao');
    if (root) initSimEP0301(root); else setTimeout(tryInitSimEP0301, 200);
  }
  tryInitSimEP0301();
})();
</script>
</div>
""")

**Figura 3.26:** Simulador EP03_01: Adição Saturada de Constante (p


<figure id="fig-03-sim-ep0301-adicao">
  <img src="imagens/fig-03-sim-ep0301-adicao.png" alt=" Simulador EP03_01: Adição Saturada de Constante (p' = clip(p + k)) " style="max-width:80%" />
  <figcaption><strong>Figura 3.26:</strong>  Simulador EP03_01: Adição Saturada de Constante (p' = clip(p + k)) </figcaption>
</figure>

In [85]:
%%writefile EP03_01.cpp
// sua solução

Overwriting EP03_01.cpp


In [86]:
TestSuite("EP03_01.cpp").run()


### 3.0.2 EP03_02 🔀 Alpha *Blending* de Duas Imagens

Em medicina nuclear, imagens de diferentes modalidades (tomografia computadorizada e ressonância magnética) são fundidas para auxiliar no diagnóstico. A **mistura ponderada** (*alpha blending*) é a operação fundamental desse processo, permitindo ao radiologista controlar interativamente o peso de cada modalidade na imagem exibida.

Ver na [Figura 3.27](#fig-03-sim-ep0302-blending) uma simulação deste EP.


#### 3.0.2.1 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Parâmetro:** Ler o valor real $\alpha \in [0, 1]$.
3. **Dados:** Ler os valores inteiros da matriz $f_1$ (imagem 1) e em seguida da matriz $f_2$ (imagem 2).
4. **Mapeamento:** Para cada posição $(i, j)$, calcular:

$$g(i,j) = \text{clip}\left(\text{round}\left(\alpha \cdot f_1(i,j) + (1-\alpha) \cdot f_2(i,j)\right)\right)$$

5. **Saída:** Exibir a matriz resultante $L \times C$.

#### 3.0.2.2 📌 Restrições Computacionais

* **Arredondamento:** Aplicar `round` antes da conversão para inteiro.
* **Saturação:** Confinar ao intervalo $[0, 255]$ com $\text{clip}(x) = \max(0, \min(255, x))$.
* **Operação em float:** Realize a operação em ponto flutuante antes de arredondar.

#### 3.0.2.3 🧠 Fundamentação Teórica

| Valor de $\alpha$ | Resultado |
|:-----------------:|:----------|
| $\alpha = 1.0$ | Apenas $f_1$ |
| $\alpha = 0.5$ | Média aritmética de $f_1$ e $f_2$ |
| $\alpha = 0.0$ | Apenas $f_2$ |

#### 3.0.2.4 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Real $\alpha$.
* Linhas seguintes: Elementos de $f_1$ ($L$ linhas com $C$ valores cada).
* Linhas seguintes: Elementos de $f_2$ ($L$ linhas com $C$ valores cada).

**Saída:**

* Matriz resultante $L \times C$.

#### 3.0.2.5 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 1<br>3<br>0.5<br>0 100 200<br>100 200 50 | 50 150 125 | Média entre as duas imagens |
| 1<br>3<br>1.0<br>10 20 30<br>90 80 70 | 10 20 30 | Apenas $f_1$ (alpha=1) |

In [87]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0302-blending" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🔀 Simulador EP03_02: Alpha Blending de Duas Imagens</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = α·f1 + (1−α)·f2</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Ajuste o parâmetro de transparência α para observar a combinação linear ponderada pixel a pixel entre as imagens f1 e f2.</p>

    <!-- Controle do Parâmetro Alpha -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#8e44ad;">α (Alpha — Peso de f1)</label>
        <span id="sim_ep0302_vl_a" style="font-family:monospace;font-size:12px;font-weight:700;color:#8e44ad;">0.50</span>
      </div>
      <input type="range" id="sim_ep0302_sl_a" min="0" max="1" step="0.05" value="0.5" style="width:100%;cursor:pointer;">
      <div style="margin-top:6px;font-size:10px;color:#8a8371;text-align:center;font-family:monospace;">
        α = 0.00 → Apenas f2 &nbsp;|&nbsp; α = 0.50 → Média Ponderada Igual &nbsp;|&nbsp; α = 1.00 → Apenas f1
      </div>
    </div>

    <!-- Comparativo em 3 Colunas: f1 vs f2 vs Resultado g -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(160px, 1fr));gap:12px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem f1 -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Imagem f1</span>
        <div id="sim_ep0302_grid_f1" style="display:grid;grid-template-columns:repeat(4, 38px);gap:3px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0302_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Novas Imagens</button>
      </div>

      <!-- Imagem f2 -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#b9770e;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Imagem f2</span>
        <div id="sim_ep0302_grid_f2" style="display:grid;grid-template-columns:repeat(4, 38px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Resultado g -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#8e44ad;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Resultado g</span>
        <div id="sim_ep0302_grid_g" style="display:grid;grid-template-columns:repeat(4, 38px);gap:3px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0302_btnReset" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">↩ Resetar (α = 0.5)</button>
      </div>

    </div>

    <!-- Mensagem Explicativa Dinâmica -->
    <div id="sim_ep0302_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      Fórmula: <b>clip(round(0.50 · f1 + 0.50 · f2))</b>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0302(root){
    if (!root || root.dataset.simEp0302Init) return;
    root.dataset.simEp0302Init = "1";

    var slA      = root.querySelector('#sim_ep0302_sl_a');
    var vlA      = root.querySelector('#sim_ep0302_vl_a');
    var gF1      = root.querySelector('#sim_ep0302_grid_f1');
    var gF2      = root.querySelector('#sim_ep0302_grid_f2');
    var gG       = root.querySelector('#sim_ep0302_grid_g');
    var debugDiv = root.querySelector('#sim_ep0302_debug');

    var btnNew   = root.querySelector('#sim_ep0302_btnNew');
    var btnReset = root.querySelector('#sim_ep0302_btnReset');

    var px1 = [], px2 = [];

    function generate() {
      px1 = Array.from({length: 16}, function(){ return Math.floor(Math.random() * 256); });
      px2 = Array.from({length: 16}, function(){ return Math.floor(Math.random() * 256); });
    }

    function createCell(val) {
      var c = document.createElement('div');
      var fgColor = val > 128 ? '#000000' : '#ffffff';
      c.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + val + ',' + val + ',' + val + ');color:' + fgColor + ';box-sizing:border-box;';
      c.textContent = val;
      return c;
    }

    function render() {
      var a = parseFloat(slA.value) || 0;
      var a1 = a.toFixed(2);
      var a2 = (1 - a).toFixed(2);

      vlA.textContent = a1;
      debugDiv.innerHTML = 'Fórmula: <b>clip(round(' + a1 + ' · f1 + ' + a2 + ' · f2))</b>';

      gF1.innerHTML = '';
      gF2.innerHTML = '';
      gG.innerHTML  = '';

      for (var i = 0; i < 16; i++) {
        gF1.appendChild(createCell(px1[i]));
        gF2.appendChild(createCell(px2[i]));

        var res = Math.max(0, Math.min(255, Math.round(a * px1[i] + (1 - a) * px2[i])));
        gG.appendChild(createCell(res));
      }
    }

    slA.addEventListener('input', render);

    btnNew.addEventListener('click', function() {
      generate();
      render();
    });

    btnReset.addEventListener('click', function() {
      slA.value = '0.5';
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0302(){
    var root = document.getElementById('sim-ep0302-blending');
    if (root) initSimEP0302(root); else setTimeout(tryInitSimEP0302, 200);
  }
  tryInitSimEP0302();
})();
</script>
</div>
""")

**Figura 3.27:** Simulador EP03_02: Alpha Blending de Duas Imagens (g = α·f1 + (1−α)·f2)


<figure id="fig-03-sim-ep0302-blending">
  <img src="imagens/fig-03-sim-ep0302-blending.png" alt=" Simulador EP03_02: Alpha Blending de Duas Imagens (g = α·f1 + (1−α)·f2) " style="max-width:80%" />
  <figcaption><strong>Figura 3.27:</strong>  Simulador EP03_02: Alpha Blending de Duas Imagens (g = α·f1 + (1−α)·f2) </figcaption>
</figure>

In [88]:
%%writefile EP03_02.cpp
// sua solução

Overwriting EP03_02.cpp


In [89]:
TestSuite("EP03_02.cpp").run()


### 3.0.3 EP03_03 🎭 Inversão de Imagem (Negativo Fotográfico)

Em radiologia, as imagens de raio-X são tradicionalmente visualizadas em negativo: ossos aparecem em preto sobre fundo branco. A operação de **negativo fotográfico** é aplicada rotineiramente em PACS (*Picture Archiving and Communication Systems*) para facilitar a detecção de fraturas e densidades ósseas.

Ver na [Figura 3.28](#fig-03-sim-ep0303-inversao) uma simulação deste EP.


#### 3.0.3.1 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Dados:** Ler os valores inteiros da matriz original.
3. **Mapeamento:** Para cada pixel $p$, calcular o negativo:

$$p' = 255 - p$$

4. **Saída:** Exibir a matriz resultante $L \times C$.

#### 3.0.3.2 📌 Restrições Computacionais

* **Sem *clipping* necessário:** O resultado de $255 - p$ com $p \in [0, 255]$ é sempre $\in [0, 255]$.
* **Tipo inteiro:** Saída deve ser valores inteiros.
* **Equivalência lógica:** A operação é idêntica ao `NOT` bit a bit (`mm::bnot`) em imagens de 8 bits.

#### 3.0.3.3 🧠 Fundamentação Teórica

| Pixel Original $p$ | Pixel Negativo $p'$ | Observação |
|:------------------:|:-------------------:|:----------:|
| 0 (preto) | 255 (branco) | Inversão total |
| 128 (cinza médio) | 127 (cinza médio) | Valor central |
| 255 (branco) | 0 (preto) | Inversão total |

#### 3.0.3.4 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linhas seguintes: Elementos inteiros da matriz original.

**Saída:**

* Matriz negativa em $L$ linhas e $C$ colunas.

#### 3.0.3.5 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 1<br>4<br>0 128 200 255 | 255 127 55 0 | Inversão de cada pixel |
| 2<br>2<br>10 20<br>30 40 | 245 235<br>225 215 | Matriz 2x2 invertida |

In [90]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0303-inversao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🎭 Simulador EP03_03: Negativo Fotográfico (Inversão)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">p' = 255 − p</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Observe a inversão complementar de intensidade: tons escuros tornam-se claros e tons claros tornam-se escuros subtraindo cada píxel do valor máximo de 255.</p>

    <!-- Comparativo Lado a Lado: Entrada vs Negativo -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Entrada Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Entrada Original (p)</span>
        <div id="sim_ep0303_grid_orig" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0303_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nova Imagem</button>
      </div>

      <!-- Negativo (p' = 255 - p) -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Negativo (p' = 255 − p)</span>
        <div id="sim_ep0303_grid_neg" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <div style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid transparent;background:transparent;color:transparent;user-select:none;">&nbsp;</div>
      </div>

    </div>

    <!-- Painel de Informação Explicativo -->
    <div id="sim_ep0303_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      Fórmula aplicada: <b>p' = 255 − p</b>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0303(root){
    if (!root || root.dataset.simEp0303Init) return;
    root.dataset.simEp0303Init = "1";

    var gO = root.querySelector('#sim_ep0303_grid_orig');
    var gN = root.querySelector('#sim_ep0303_grid_neg');
    var btnNew = root.querySelector('#sim_ep0303_btnNew');

    var pixels = [];

    function generate() {
      pixels = Array.from({length: 16}, function(){ return Math.floor(Math.random() * 256); });
    }

    function render() {
      gO.innerHTML = '';
      gN.innerHTML = '';

      pixels.forEach(function(p) {
        // Célula Original
        var cellO = document.createElement('div');
        var fgColorO = p > 128 ? '#000000' : '#ffffff';
        cellO.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgColorO + ';box-sizing:border-box;';
        cellO.textContent = p;
        gO.appendChild(cellO);

        // Célula Negativo
        var r = 255 - p;
        var fgColorN = r > 128 ? '#000000' : '#ffffff';
        var cellN = document.createElement('div');
        cellN.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + r + ',' + r + ',' + r + ');color:' + fgColorN + ';box-sizing:border-box;';
        cellN.textContent = r;
        gN.appendChild(cellN);
      });
    }

    btnNew.addEventListener('click', function() {
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0303(){
    var root = document.getElementById('sim-ep0303-inversao');
    if (root) initSimEP0303(root); else setTimeout(tryInitSimEP0303, 200);
  }
  tryInitSimEP0303();
})();
</script>
""")

**Figura 3.28:** Simulador EP03_03: Inversão de Imagem — Negativo Fotográfico (p


<figure id="fig-03-sim-ep0303-inversao">
  <img src="imagens/fig-03-sim-ep0303-inversao.png" alt=" Simulador EP03_03: Inversão de Imagem — Negativo Fotográfico (p' = 255 − p) " style="max-width:80%" />
  <figcaption><strong>Figura 3.28:</strong>  Simulador EP03_03: Inversão de Imagem — Negativo Fotográfico (p' = 255 − p) </figcaption>
</figure>

In [91]:
%%writefile EP03_03.cpp
// sua solução

Overwriting EP03_03.cpp


In [92]:
TestSuite("EP03_03.cpp").run()


### 3.0.4 EP03_04 📊 Equalização de Histograma (L bits)

Em imagens de satélite de sensoriamento remoto, a variação de iluminação ao longo do dia produz imagens de baixo contraste. A **equalização de histograma** é aplicada automaticamente em satélites como o Landsat para redistribuir os tons, revelando detalhes de vegetação, relevo e zonas urbanas invisíveis na imagem original.

Ver na [Figura 3.29](#fig-03-sim-ep0304-equalizacao) uma simulação deste EP.


#### 3.0.4.1 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas), $C$ (colunas) e $B$ (número de bits, com $L_{\max} = 2^B$).
2. **Dados:** Ler a matriz de pixels $f$ com valores em $[0, 2^B - 1]$.
3. **Histograma:** Calcular $h[k]$ = número de pixels com intensidade $k$, para $k = 0 \ldots 2^B-1$.
4. **Probabilidade:** $p[k] = h[k] / (L \cdot C)$.
5. **CDF:** $\text{cdf}[k] = \sum_{j=0}^{k} p[j]$; função de distribuição acumulada.
6. **LUT:** $\text{lut}[k] = \text{round}\left(\text{cdf}[k] \cdot (2^B - 1)\right)$; *Look-Up Table* (tabela de consulta).
7. **Aplicação:** $g[i,j] = \text{lut}[f[i,j]]$.
8. **Saída:** Exibir a matriz equalizada $L \times C$.

#### 3.0.4.2 📌 Restrições Computacionais

* **Arredondamento:** Usar arredondamento matemático (`round`) na LUT.
* **Bits:** O número de níveis é $2^B$ (ex.: $B=3 \Rightarrow 8$ níveis, $B=8 \Rightarrow 256$ níveis).
* **CDF acumulada:** $\text{cdf}[k] = \sum_{j=0}^{k} p[j]$, com $\text{cdf}[2^B-1] = 1.0$.

#### 3.0.4.3 🧠 Fundamentação Teórica

| Etapa | Operação | Fórmula |
|:-----:|:---------|:--------|
| 1 | Histograma | $h[k] \leftarrow$ nº pixels com intensidade $k$ |
| 2 | Probabilidade | $p[k] = h[k] / (L \cdot C)$ |
| 3 | CDF | $\text{cdf}[k] = \sum_{j=0}^{k} p[j]$ |
| 4 | LUT | $\text{lut}[Look-Up Table (tabela de consulta)k] = \text{round}(\text{cdf}[k] \cdot (2^B-1))$ |
| 5 | Aplicação | $g[i,j] = \text{lut}[f[i,j]]$ |

#### 3.0.4.4 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $B$ (número de bits).
* Linhas seguintes: Elementos inteiros da matriz.

**Saída:**

* Matriz equalizada em $L$ linhas e $C$ colunas.

#### 3.0.4.5 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 5<br>5<br>3<br>3 4 2 3 4<br>4 3 3 4 3<br>2 3 4 3 2<br>3 4 3 2 3<br>4 3 2 3 4 | 5 7 1 5 7<br>7 5 5 7 5<br>1 5 7 5 1<br>5 7 5 1 5<br>7 5 1 5 7 | Exemplo 3 bits do capítulo |
| 1<br>4<br>3<br>0 0 7 7 | 0 0 7 7 | Histograma bimodal extremo |

In [93]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0304-equalizacao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">📊 Simulador EP03_04: Equalização de Histograma</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">lut[k] = round(cdf[k] · (L − 1))</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Escolha a profundidade de bits (B) e gere imagens para analisar o espalhamento dinâmico do histograma e a tabela de remapeamento (LUT) em tempo real.</p>

    <!-- Controle de Bits B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#16a085;">Profundidade de Bits (B)</label>
        <span id="sim_ep0304_vl_bits" style="font-family:monospace;font-size:12px;font-weight:700;color:#16a085;">3 bits → 8 níveis</span>
      </div>
      <input type="range" id="sim_ep0304_sl_bits" min="1" max="8" step="1" value="3" style="width:100%;cursor:pointer;">
      <div style="display:flex;justify-content:space-between;margin-top:6px;font-size:10px;color:#8a8371;font-family:monospace;">
        <span>1 bit (2 níveis)</span>
        <span>4 bits (16 níveis)</span>
        <span>8 bits (256 níveis)</span>
      </div>
    </div>

    <!-- Comparativo Lado a Lado: Entrada vs Equalizada -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:16px;">
      
      <!-- Entrada Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#7f8c8d;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Entrada Original</span>
        <div id="sim_ep0304_grid_orig" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0304_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nova Imagem</button>
      </div>

      <!-- Resultado Equalizado -->
      <div style="background:#fafaf7;border:1px solid #16a085;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#16a085;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Resultado Equalizado</span>
        <div id="sim_ep0304_grid_eq" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0304_btnReset" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">↩ Nova Amostragem</button>
      </div>

    </div>

    <!-- Histogramas Lado a Lado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(240px, 1fr));gap:16px;margin-bottom:16px;">
      
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
        <span style="font-size:10px;font-weight:700;color:#7f8c8d;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;text-align:center;">Histograma Original</span>
        <canvas id="sim_ep0304_hist_orig" style="width:100%;height:80px;display:block;" width="340" height="80"></canvas>
      </div>

      <div style="background:#fafaf7;border:1px solid #16a085;border-radius:12px;padding:12px;">
        <span style="font-size:10px;font-weight:700;color:#16a085;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;text-align:center;">Histograma Equalizado</span>
        <canvas id="sim_ep0304_hist_eq" style="width:100%;height:80px;display:block;" width="340" height="80"></canvas>
      </div>

    </div>

    <!-- Tabela LUT -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:10px 14px;margin-bottom:14px;overflow-x:auto;">
      <span style="font-size:10px;font-weight:700;color:#5e5a4a;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:6px;">LUT (Tabela de Remapeamento k → v)</span>
      <div id="sim_ep0304_lut_table" style="font-family:monospace;font-size:11px;color:#26241d;white-space:nowrap;"></div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0304_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      lut[k] = round(cdf[k] · 7) | B=3, níveis=8
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0304(root){
    if (!root || root.dataset.simEp0304Init) return;
    root.dataset.simEp0304Init = "1";

    var slBits     = root.querySelector('#sim_ep0304_sl_bits');
    var vlBits     = root.querySelector('#sim_ep0304_vl_bits');
    var debugDiv   = root.querySelector('#sim_ep0304_debug');
    var gridOrig   = root.querySelector('#sim_ep0304_grid_orig');
    var gridEq     = root.querySelector('#sim_ep0304_grid_eq');
    var lutTable   = root.querySelector('#sim_ep0304_lut_table');
    var histOrig   = root.querySelector('#sim_ep0304_hist_orig');
    var histEq     = root.querySelector('#sim_ep0304_hist_eq');

    var btnNew     = root.querySelector('#sim_ep0304_btnNew');
    var btnReset   = root.querySelector('#sim_ep0304_btnReset');

    var ROWS = 4, COLS = 4;
    var pixels = [];

    function generatePixels() {
      var b = parseInt(slBits.value) || 3;
      var levels = Math.pow(2, b);
      var lo = Math.floor(levels * 0.2);
      var hi = Math.floor(levels * 0.5);
      pixels = Array.from({ length: ROWS * COLS }, function(){
        return lo + Math.floor(Math.random() * (hi - lo + 1));
      });
    }

    function computeEqualization(pixArr, b) {
      var levels = Math.pow(2, b);
      var N = pixArr.length;
      var h = new Array(levels).fill(0);
      pixArr.forEach(function(p){ h[p]++; });

      var cdf = new Array(levels).fill(0);
      cdf[0] = h[0] / N;
      for (var k = 1; k < levels; k++) {
        cdf[k] = cdf[k - 1] + h[k] / N;
      }

      var lut = cdf.map(function(c){ return Math.round(c * (levels - 1)); });
      var result = pixArr.map(function(p){ return lut[p]; });
      return { h: h, cdf: cdf, lut: lut, result: result };
    }

    function drawHistogram(canvas, counts, levels, color) {
      var ctx = canvas.getContext('2d');
      var W = canvas.width, H = canvas.height;
      ctx.clearRect(0, 0, W, H);

      var maxVal = Math.max.apply(null, counts.concat([1]));
      var barW = W / levels;

      counts.forEach(function(c, i){
        var barH = (c / maxVal) * (H - 6);
        ctx.fillStyle = color;
        ctx.fillRect(i * barW + 1, H - barH, barW - 2, barH);
      });
    }

    function toGray(val, levels) {
      return Math.round((val / (levels - 1)) * 255);
    }

    function render() {
      var b = parseInt(slBits.value) || 3;
      var levels = Math.pow(2, b);
      vlBits.textContent = b + ' bit' + (b > 1 ? 's' : '') + ' → ' + levels + ' níveis';

      var eqData = computeEqualization(pixels, b);
      var hOrig = eqData.h;
      var lut = eqData.lut;
      var result = eqData.result;

      var hEq = new Array(levels).fill(0);
      result.forEach(function(p){ hEq[p]++; });

      lutTable.innerHTML = lut.map(function(v, k){
        return '<span style="display:inline-block;margin-right:10px;color:#8a8371;">' + k + ' → <b style="color:#16a085;">' + v + '</b></span>';
      }).join('');

      debugDiv.innerHTML = '<b>lut[k] = round(cdf[k] · ' + (levels - 1) + ')</b> &nbsp;|&nbsp; B = ' + b + ', níveis = ' + levels;

      gridOrig.innerHTML = '';
      gridEq.innerHTML = '';

      pixels.forEach(function(p, i) {
        var grayO = toGray(p, levels);
        var fgO = grayO > 128 ? '#000000' : '#ffffff';
        var cellO = document.createElement('div');
        cellO.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + grayO + ',' + grayO + ',' + grayO + ');color:' + fgO + ';box-sizing:border-box;';
        cellO.textContent = p;
        gridOrig.appendChild(cellO);

        var r = result[i];
        var grayR = toGray(r, levels);
        var fgR = grayR > 128 ? '#000000' : '#ffffff';
        var cellR = document.createElement('div');
        cellR.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + grayR + ',' + grayR + ',' + grayR + ');color:' + fgR + ';box-sizing:border-box;';
        cellR.textContent = r;
        gridEq.appendChild(cellR);
      });

      drawHistogram(histOrig, hOrig, levels, '#95a5a6');
      drawHistogram(histEq, hEq, levels, '#16a085');
    }

    slBits.addEventListener('input', function(){
      generatePixels();
      render();
    });

    btnNew.addEventListener('click', function(){
      generatePixels();
      render();
    });

    btnReset.addEventListener('click', function(){
      generatePixels();
      render();
    });

    generatePixels();
    render();
  }

  function tryInitSimEP0304(){
    var root = document.getElementById('sim-ep0304-equalizacao');
    if (root) initSimEP0304(root); else setTimeout(tryInitSimEP0304, 200);
  }
  tryInitSimEP0304();
})();
</script>
</div>
""")

**Figura 3.29:** Simulador EP03_04: Equalização de Histograma (Níveis L = 2^B)


<figure id="fig-03-sim-ep0304-equalizacao">
  <img src="imagens/fig-03-sim-ep0304-equalizacao.png" alt=" Simulador EP03_04: Equalização de Histograma (Níveis L = 2^B) " style="max-width:80%" />
  <figcaption><strong>Figura 3.29:</strong>  Simulador EP03_04: Equalização de Histograma (Níveis L = 2^B) </figcaption>
</figure>

In [94]:
%%writefile EP03_04.cpp
// sua solução

Overwriting EP03_04.cpp


In [95]:
TestSuite("EP03_04.cpp").run()


### 3.0.5 EP03_05 🔲 Aplicação de Máscara AND Binária

Em sistemas de inspeção industrial por visão computacional, é necessário isolar regiões de interesse (ROI) em imagens de peças para verificar defeitos de fabricação. A operação **AND bit a bit com uma máscara binária** é o mecanismo fundamental para recortar exatamente a área de inspeção, zerando todos os pixels fora dela.

Ver na [Figura 3.30](#fig-03-sim-ep0305-mascara) uma simulação deste EP.


#### 3.0.5.1 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Dados:** Ler a matriz de pixels $f$ (valores $\in [0, 255]$).
3. **Máscara:** Ler a matriz binária $m$ (valores: apenas 0 ou 255).
4. **Mapeamento:** Para cada pixel $(i,j)$, aplicar o AND bit a bit:

$$
g(i,j) = f(i,j) \;\text{AND}\; m(i,j)
$$

onde $255 =$ `11111111` e $0 =$ `00000000` em binário.

5. **Saída:** Exibir a matriz resultante $L \times C$.

#### 3.0.5.2 📌 Restrições Computacionais

* **AND com 255:** $p \; \text{AND} \; 255 = p$ (todos os bits preservados).
* **AND com 0:** $p \; \text{AND} \; 0 = 0$ (todos os bits zerados).
* **Máscara:** Os únicos valores possíveis na máscara são 0 e 255.
* **Implementação:** Em Python, o AND bit a bit entre inteiros usa o operador `&`.

#### 3.0.5.3 🧠 Fundamentação Teórica

| Pixel $f$ | Máscara $m$ | Resultado $f$ AND $m$ |
|:---------:|:-----------:|:---------------------:|
| qualquer $v$ | 255 (`11111111`) | $v$ (preservado) |
| qualquer $v$ | 0 (`00000000`) | 0 (zerado) |

#### 3.0.5.4 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linhas seguintes: Elementos de $f$ ($L$ linhas).
* Linhas seguintes: Elementos de $m$ ($L$ linhas com valores 0 ou 255).

**Saída:**

* Matriz resultante $L \times C$.

#### 3.0.5.5 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 2<br>3<br>100 150 200<br>50 80 120<br>255 255 0<br>0 255 255 | 100 150 0<br>0 80 120 | Máscara seleciona região |
| 1<br>4<br>10 20 30 40<br>255 0 255 0 | 10 0 30 0 | Alternado preservado/zerado |

In [96]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0305-mascara" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">⬛ Simulador EP03_05: Máscara AND Binária</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f AND m</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Clique nas células da <b>Máscara m</b> para alternar entre passante (255) e bloqueante (0), aplicando a operação lógica pixel a pixel.</p>

    <!-- Três Colunas Principais: Imagem f, Máscara m, Resultado g -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:16px;">
      
      <!-- Imagem f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Imagem f (0–255)</span>
        <div id="sim_ep0305_grid_f" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0305_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nova Imagem</button>
      </div>

      <!-- Máscara m -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Máscara m (Clique para Alternar)</span>
        <div id="sim_ep0305_grid_mask" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0305_btnReset" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">↺ Resetar Máscara</button>
      </div>

      <!-- Resultado g -->
      <div style="background:#fafaf7;border:2px solid #27ae60;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Resultado g = f AND m</span>
        <div id="sim_ep0305_grid_result" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <div style="margin-top:12px;height:26px;display:flex;align-items:center;justify-content:center;">
          <span id="sim_ep0305_pct" style="font-size:11px;font-weight:700;color:#26241d;font-family:monospace;">—</span>
        </div>
      </div>

    </div>

    <!-- Estatísticas de Preservação -->
    <div style="display:grid;grid-template-columns:repeat(3, minmax(0, 1fr));gap:10px;margin-bottom:14px;text-align:center;">
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:8px;padding:8px 10px;font-size:11px;color:#8a8371;">
        <b id="sim_ep0305_stat_preserved" style="font-size:14px;display:block;color:#26241d;font-family:monospace;">—</b>preservados
      </div>
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:8px;padding:8px 10px;font-size:11px;color:#8a8371;">
        <b id="sim_ep0305_stat_zeroed" style="font-size:14px;display:block;color:#26241d;font-family:monospace;">—</b>zerados
      </div>
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:8px;padding:8px 10px;font-size:11px;color:#8a8371;">
        <b id="sim_ep0305_stat_ratio" style="font-size:14px;display:block;color:#26241d;font-family:monospace;">—</b>visível
      </div>
    </div>

    <!-- Legenda e Debug -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Legenda:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:24px;height:24px;background:#ebf4fd;border:1.5px solid #2980b9;border-radius:4px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:700;color:#2980b9;">255</div>
        <span style="font-size:10.5px;color:#5e5a4a;">Passante (preservado)</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:24px;height:24px;background:#26241d;border:1.5px solid #8a8371;border-radius:4px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:700;color:#7ee7c6;">0</div>
        <span style="font-size:10.5px;color:#5e5a4a;">Bloqueante (zerado)</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0305_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      g(i,j) = f(i,j) &amp; m(i,j)
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0305(root){
    if (!root || root.dataset.simEp0305Init) return;
    root.dataset.simEp0305Init = "1";

    var gridF      = root.querySelector('#sim_ep0305_grid_f');
    var gridMask   = root.querySelector('#sim_ep0305_grid_mask');
    var gridResult = root.querySelector('#sim_ep0305_grid_result');
    var debugDiv   = root.querySelector('#sim_ep0305_debug');
    var pctSpan    = root.querySelector('#sim_ep0305_pct');
    var statPres   = root.querySelector('#sim_ep0305_stat_preserved');
    var statZero   = root.querySelector('#sim_ep0305_stat_zeroed');
    var statRatio  = root.querySelector('#sim_ep0305_stat_ratio');

    var btnNew     = root.querySelector('#sim_ep0305_btnNew');
    var btnReset   = root.querySelector('#sim_ep0305_btnReset');

    var N = 16;
    var pixels = [];
    var mask = [];

    function generatePixels() {
      pixels = Array.from({ length: N }, function(){ return Math.floor(Math.random() * 256); });
    }

    function resetMask() {
      mask = Array.from({ length: N }, function(_, i) {
        var r = Math.floor(i / 4), c = i % 4;
        return (r + c) % 2 === 0 ? 255 : 0;
      });
    }

    function render() {
      gridF.innerHTML = '';
      gridMask.innerHTML = '';
      gridResult.innerHTML = '';

      var preserved = 0, zeroed = 0;

      for (var i = 0; i < N; i++) {
        var p = pixels[i];
        var m = mask[i];
        var res = p & m;

        // Célula F
        var cellF = document.createElement('div');
        var fgF = p > 128 ? '#000000' : '#ffffff';
        cellF.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgF + ';box-sizing:border-box;';
        cellF.textContent = p;
        gridF.appendChild(cellF);

        // Célula Máscara M (interativa)
        var cellM = document.createElement('div');
        cellM.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;cursor:pointer;user-select:none;box-sizing:border-box;transition:all 0.15s ease;';
        if (m === 255) {
          cellM.style.background = '#ebf4fd';
          cellM.style.border = '2px solid #2980b9';
          cellM.style.color = '#2980b9';
        } else {
          cellM.style.background = '#26241d';
          cellM.style.border = '2px solid #8a8371';
          cellM.style.color = '#7ee7c6';
        }
        cellM.textContent = m;
        cellM.title = m === 255 ? 'Clique para bloquear (0)' : 'Clique para passar (255)';

        (function(idx){
          cellM.addEventListener('click', function(){
            mask[idx] = mask[idx] === 255 ? 0 : 255;
            render();
          });
        })(i);

        gridMask.appendChild(cellM);

        // Célula Resultado G
        var cellR = document.createElement('div');
        var fgR = res > 128 ? '#000000' : '#ffffff';
        var borderStyle = m === 255 ? '2px solid #27ae60' : '1px solid #e4dcc8';
        cellR.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;user-select:none;background:rgb(' + res + ',' + res + ',' + res + ');color:' + fgR + ';border:' + borderStyle + ';box-sizing:border-box;';
        cellR.textContent = res;
        gridResult.appendChild(cellR);

        if (m === 255) preserved++; else zeroed++;
      }

      var ratioPct = Math.round((preserved / N) * 100);
      statPres.textContent = preserved;
      statZero.textContent = zeroed;
      statRatio.textContent = ratioPct + '%';
      pctSpan.textContent = ratioPct + '% visível';
      debugDiv.innerHTML = 'g(i,j) = f(i,j) &amp; m(i,j) &nbsp;|&nbsp; <b>' + preserved + '</b> preservados · <b>' + zeroed + '</b> zerados';
    }

    btnNew.addEventListener('click', function(){
      generatePixels();
      render();
    });

    btnReset.addEventListener('click', function(){
      resetMask();
      render();
    });

    generatePixels();
    resetMask();
    render();
  }

  function tryInitSimEP0305(){
    var root = document.getElementById('sim-ep0305-mascara');
    if (root) initSimEP0305(root); else setTimeout(tryInitSimEP0305, 200);
  }
  tryInitSimEP0305();
})();
</script>
</div>
""")

**Figura 3.30:** Simulador EP03_05: Aplicação de Máscara AND Binária


<figure id="fig-03-sim-ep0305-mascara">
  <img src="imagens/fig-03-sim-ep0305-mascara.png" alt=" Simulador EP03_05: Aplicação de Máscara AND Binária " style="max-width:80%" />
  <figcaption><strong>Figura 3.30:</strong>  Simulador EP03_05: Aplicação de Máscara AND Binária </figcaption>
</figure>

In [97]:
%%writefile EP03_05.cpp
// sua solução

Overwriting EP03_05.cpp


In [98]:
TestSuite("EP03_05.cpp").run()


### 3.0.6 EP03_06 🌫️ Filtro de Média com *Kernel* N×N


Em câmeras de veículos autônomos, imagens capturadas sob chuva ou névoa apresentam ruído gaussiano. O **filtro de média** é amplamente utilizado para sua redução em tempo real, sendo implementado diretamente no **ISP** (*Image Signal Processor*) de sensores **CMOS** (*Complementary Metal-Oxide-Semiconductor*).

Os sensores CMOS são os sensores de imagem usados na maioria das câmeras modernas (*smartphones*, *webcams*, câmeras automotivas etc.). Eles convertem a luz em sinais elétricos, e o ISP processa esses sinais em tempo real — aplicando operações como redução de ruído, balanço de branco e outros ajustes de imagem.

Ver na [Figura 3.31](#fig-03-sim-ep0306-media) uma simulação deste EP.


#### 3.0.6.1 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas), $C$ (colunas) e $N$ (tamanho do *kernel*, sempre ímpar).
2. **Dados:** Ler a matriz de pixels $f$.
3. **Filtro de Média:** Para cada pixel $(i,j)$ **interno** (sem bordas), calcular:

$$g(i,j) = \text{round}\left(\frac{1}{N^2} \sum_{s=-(r)}^{r} \sum_{t=-(r)}^{r} f(i+s,\, j+t)\right), \quad r = \lfloor N/2 \rfloor$$

4. **Tratamento de Borda:** Pixels na borda (onde a janela $N \times N$ ultrapassa os limites) devem ser **copiados diretamente** do original sem modificação.
5. **Saída:** Exibir a matriz resultante $L \times C$.

#### 3.0.6.2 📌 Restrições Computacionais

* **Raio:** $r = \lfloor N/2 \rfloor$ (metade do *kernel*, inteiro).
* **Pixels internos:** $(i,j)$ com $r \le i < L-r$ e $r \le j < C-r$.
* **Arredondamento:** Usar arredondamento matemático antes de converter para inteiro.
* **Sem *clipping*:** A média de valores $\in [0,255]$ permanece em $[0,255]$.

#### 3.0.6.3 🧠 Fundamentação Teórica

| Tamanho $N$ | Coeficiente | Pixels na janela | Efeito |
|:-----------:|:-----------:|:----------------:|:------:|
| 3 | $1/9 \approx 0.111$ | 9 | Suave |
| 5 | $1/25 = 0.04$ | 25 | Médio |
| 7 | $1/49 \approx 0.020$ | 49 | Forte |

#### 3.0.6.4 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $N$ (ímpar, $N \ge 3$).
* Linhas seguintes: Elementos da matriz original.

**Saída:**

* Matriz filtrada $L \times C$.

#### 3.0.6.5 📌 Exemplos
| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>3<br>10 20 30<br>40 50 60<br>70 80 90 | 10 20 30<br>40 50 60<br>70 80 90 | Apenas borda (3×3 = borda total) |
| 5<br>5<br>3<br>0 0 0 0 0<br>0 0 0 0 0<br>0 0 100 0 0<br>0 0 0 0 0<br>0 0 0 0 0 | 0 0 0 0 0<br>0 11 11 11 0<br>0 11 11 11 0<br>0 11 11 11 0<br>0 0 0 0 0 | Pixel isolado: todos os 9 pixels internos cuja janela 3×3 inclui o valor 100 recebem round(100/9)=11 |

In [99]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0306-media" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🔲 Simulador EP03_06: Filtro de Média com Kernel N×N</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = Média(Vizinhos)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Selecione o tamanho do kernel e passe o mouse sobre os pixels do resultado para inspecionar a vizinhança e o cálculo da média aritmética.</p>

    <!-- Barra de Controles / Seleção de Kernel -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;align-items:center;gap:10px;flex-wrap:wrap;justify-content:space-between;">
      <div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap;">
        <span style="font-size:11px;font-weight:700;color:#26241d;">Tamanho do kernel:</span>
        <button id="sim_ep0306_btn_k3" style="padding:5px 12px;font-size:11px;font-weight:700;border:1px solid #2980b9;background:#ebf4fd;color:#2980b9;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">3 × 3 (9 vizinhos)</button>
        <button id="sim_ep0306_btn_k5" style="padding:5px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">5 × 5 (25 vizinhos)</button>
      </div>
      <button id="sim_ep0306_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nova Imagem</button>
    </div>

    <!-- Comparativo Lado a Lado: Original vs Resultado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem Original (7x7) -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Imagem Original f (7×7)</span>
        <span style="font-size:10px;color:#8a8371;display:block;margin-bottom:10px;">Com ruído sal e pimenta</span>
        <div id="sim_ep0306_grid_f" style="display:grid;gap:4px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Resultado Suavizado -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Resultado g (Filtro Suavizado)</span>
        <span style="font-size:10px;color:#2980b9;display:block;margin-bottom:10px;">Passe o mouse para inspecionar</span>
        <div id="sim_ep0306_grid_result" style="display:grid;gap:4px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Legenda -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Legenda:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fef5e7;border:1.5px dashed #b9770e;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Janela do Kernel</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#eafaf1;border:1.5px solid #27ae60;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Borda (Copiada)</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#ebf4fd;border:1.5px solid #2980b9;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Pixel Inspecionado</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0306_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:10px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;min-height:36px;line-height:1.5;">
      Passe o mouse sobre um pixel interno do resultado para ver o cálculo da média.
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0306(root){
    if (!root || root.dataset.simEp0306Init) return;
    root.dataset.simEp0306Init = "1";

    var ROWS = 7, COLS = 7, N = 49;
    var pixels = [], result = [], kSize = 3, radius = 1;

    var gridF   = root.querySelector('#sim_ep0306_grid_f');
    var gridRes = root.querySelector('#sim_ep0306_grid_result');
    var debug   = root.querySelector('#sim_ep0306_debug');
    var btnK3   = root.querySelector('#sim_ep0306_btn_k3');
    var btnK5   = root.querySelector('#sim_ep0306_btn_k5');
    var btnNew  = root.querySelector('#sim_ep0306_btnNew');

    function generatePixels() {
      pixels = Array.from({ length: N }, function(){
        var v = 60 + Math.floor(Math.random() * 60);
        if (Math.random() > 0.82) v = Math.random() > 0.5 ? 255 : 0;
        return v;
      });
    }

    function calculateFilter() {
      result = pixels.slice();
      radius = Math.floor(kSize / 2);
      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          if (r >= radius && r < ROWS - radius && c >= radius && c < COLS - radius) {
            var sum = 0, cnt = 0;
            for (var s = -radius; s <= radius; s++) {
              for (var t = -radius; t <= radius; t++) {
                sum += pixels[(r + s) * COLS + (c + t)];
                cnt++;
              }
            }
            result[r * COLS + c] = Math.round(sum / cnt);
          }
        }
      }
    }

    function textColor(g){ return g > 140 ? '#000000' : '#ffffff'; }

    function highlightKernel(tr, tc, on) {
      var cells = gridF.children;
      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          var idx = r * COLS + c;
          if (!cells[idx]) continue;
          var p = pixels[idx];
          if (on && Math.abs(r - tr) <= radius && Math.abs(c - tc) <= radius) {
            cells[idx].style.background = '#fef5e7';
            cells[idx].style.color = '#b9770e';
            cells[idx].style.boxShadow = '0 0 0 2px #b9770e inset';
          } else {
            cells[idx].style.background = 'rgb(' + p + ',' + p + ',' + p + ')';
            cells[idx].style.color = textColor(p);
            cells[idx].style.boxShadow = 'none';
          }
        }
      }
    }

    function render() {
      var cols = 'repeat(' + COLS + ', 42px)';
      gridF.style.gridTemplateColumns = cols;
      gridRes.style.gridTemplateColumns = cols;
      gridF.innerHTML = '';
      gridRes.innerHTML = '';

      for (var i = 0; i < N; i++) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var p = pixels[i], res = result[i];
        var isBorder = r < radius || r >= ROWS - radius || c < radius || c >= COLS - radius;

        // Célula Original f
        var cf = document.createElement('div');
        cf.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + textColor(p) + ';box-sizing:border-box;';
        cf.textContent = p;
        gridF.appendChild(cf);

        // Célula Resultado g
        var cr = document.createElement('div');
        cr.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;user-select:none;box-sizing:border-box;transition:all 0.15s ease;';

        if (isBorder) {
          cr.style.background = '#eafaf1';
          cr.style.border = '2px solid #27ae60';
          cr.style.color = '#27ae60';
          cr.textContent = res;

          (function(row, col, val){
            cr.addEventListener('mouseenter', function(){
              debug.innerHTML = 'Pixel (' + row + ',' + col + ') é <b>borda</b>: valor herdado do original sem cálculo &nbsp;→&nbsp; <b>' + val + '</b>';
            });
            cr.addEventListener('mouseleave', function(){ resetDebug(); });
          })(r, c, res);
        } else {
          cr.style.background = 'rgb(' + res + ',' + res + ',' + res + ')';
          cr.style.color = textColor(res);
          cr.style.border = '1px solid #e4dcc8';
          cr.style.cursor = 'pointer';
          cr.textContent = res;

          (function(row, col, val){
            cr.addEventListener('mouseenter', function(){
              highlightKernel(row, col, true);
              cr.style.transform = 'scale(1.12)';
              cr.style.boxShadow = '0 0 0 2px #2980b9 inset';
              cr.style.background = '#ebf4fd';
              cr.style.color = '#2980b9';

              var neighbors = [];
              for (var s = -radius; s <= radius; s++) {
                for (var t = -radius; t <= radius; t++) {
                  neighbors.push(pixels[(row + s) * COLS + (col + t)]);
                }
              }
              var kTotal = kSize * kSize;
              debug.innerHTML = 'Pixel (' + row + ',' + col + '): round( (' + neighbors.join(' + ') + ') / ' + kTotal + ' ) &nbsp;=&nbsp; <b>' + val + '</b>';
            });

            cr.addEventListener('mouseleave', function(){
              highlightKernel(row, col, false);
              cr.style.transform = 'scale(1)';
              cr.style.boxShadow = 'none';
              cr.style.background = 'rgb(' + val + ',' + val + ',' + val + ')';
              cr.style.color = textColor(val);
              resetDebug();
            });
          })(r, c, res);
        }
        gridRes.appendChild(cr);
      }
    }

    function resetDebug() {
      debug.textContent = 'Passe o mouse sobre um pixel interno do resultado para ver o cálculo da média.';
    }

    function setKernel(k) {
      kSize = k;
      if (k === 3) {
        btnK3.style.background = '#ebf4fd';
        btnK3.style.borderColor = '#2980b9';
        btnK3.style.color = '#2980b9';
        btnK3.style.fontWeight = '700';

        btnK5.style.background = '#f1ead7';
        btnK5.style.borderColor = '#e4dcc8';
        btnK5.style.color = '#5e5a4a';
        btnK5.style.fontWeight = '600';
      } else {
        btnK5.style.background = '#ebf4fd';
        btnK5.style.borderColor = '#2980b9';
        btnK5.style.color = '#2980b9';
        btnK5.style.fontWeight = '700';

        btnK3.style.background = '#f1ead7';
        btnK3.style.borderColor = '#e4dcc8';
        btnK3.style.color = '#5e5a4a';
        btnK3.style.fontWeight = '600';
      }
      calculateFilter();
      render();
    }

    btnK3.addEventListener('click', function(){ setKernel(3); });
    btnK5.addEventListener('click', function(){ setKernel(5); });

    btnNew.addEventListener('click', function(){
      generatePixels();
      calculateFilter();
      render();
    });

    generatePixels();
    calculateFilter();
    render();
  }

  function tryInitSimEP0306(){
    var root = document.getElementById('sim-ep0306-media');
    if (root) initSimEP0306(root); else setTimeout(tryInitSimEP0306, 200);
  }
  tryInitSimEP0306();
})();
</script>
</div>
""")

**Figura 3.31:** Simulador EP03_06: Filtro de Média com Kernel N×N


<figure id="fig-03-sim-ep0306-media">
  <img src="imagens/fig-03-sim-ep0306-media.png" alt=" Simulador EP03_06: Filtro de Média com Kernel N×N " style="max-width:80%" />
  <figcaption><strong>Figura 3.31:</strong>  Simulador EP03_06: Filtro de Média com Kernel N×N </figcaption>
</figure>

In [100]:
%%writefile EP03_06.cpp
// sua solução

Overwriting EP03_06.cpp


In [101]:
TestSuite("EP03_06.cpp").run()


### 3.0.7 EP03_07 🔍 Operador Laplaciano (w4) para Realce de Bordas

Em tomografias de alta resolução, a nitidez das bordas entre tecidos é crítica para diagnóstico. O **operador Laplaciano** é amplamente utilizado em *pipelines* de pré-processamento de imagens médicas para realçar automaticamente os contornos anatômicos antes da segmentação, evitando intervenção manual do radiologista.

Ver na [Figura 3.32](#fig-03-sim-ep0307-laplaciano) uma simulação deste EP.


#### 3.0.7.1 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Dados:** Ler a matriz de pixels $f$.
3. **Laplaciano (w4):** Para cada pixel **interno** $(i,j)$ com $1 \le i < L-1$, $1 \le j < C-1$, calcular:

$$\nabla^2 f(i,j) = f(i-1,j) + f(i+1,j) + f(i,j-1) + f(i,j+1) - 4 \cdot f(i,j)$$

4. **Realce:** Calcular a imagem realçada:

$$g(i,j) = \text{clip}(f(i,j) - \nabla^2 f(i,j))$$

5. **Borda:** Pixels na borda são copiados diretamente: $g(i,j) = f(i,j)$.
6. **Saída:** Exibir a matriz realçada $L \times C$.

#### 3.0.7.2 📌 Restrições Computacionais

* ***Kernel* w4:** $\begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{bmatrix}$ — apenas 4-vizinhos.
* **Saturação:** $\text{clip}(x) = \max(0, \min(255, x))$ aplicado ao resultado do realce.
* **Sem arredondamento:** O Laplaciano usa apenas somas/subtrações de inteiros.

#### 3.0.7.3 🧠 Fundamentação Teórica

| Região | $\nabla^2 f$ | Efeito do Realce |
|:------:|:------------:|:----------------:|
| Uniforme | $\approx 0$ | Sem alteração |
| Borda crescente | $< 0$ | Pixel clareado |
| Borda decrescente | $> 0$ | Pixel escurecido |

#### 3.0.7.4 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linhas seguintes: Elementos da matriz original.

**Saída:**

* Matriz realçada $L \times C$.

#### 3.0.7.5 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>0 0 0<br>0 100 0<br>0 0 0 | 0 0 0<br>0 255 0<br>0 0 0 | Pico isolado: lap=−400, g=100−(−400)=500 → clip=255 |
| 3<br>3<br>50 50 50<br>50 50 50<br>50 50 50 | 50 50 50<br>50 50 50<br>50 50 50 | Região uniforme: Laplaciano=0, sem alteração |

In [102]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0307-laplaciano" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">📐 Simulador EP03_07: Operador Laplaciano (w4)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f ∓ ∇²f</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Selecione a variante de realce e passe o mouse sobre os pixels internos do resultado para inspecionar a vizinhança de 4 pontos e a equação do Laplaciano.</p>

    <!-- Barra de Controles / Seleção de Variante -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;align-items:center;gap:10px;flex-wrap:wrap;justify-content:space-between;">
      <div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap;">
        <span style="font-size:11px;font-weight:700;color:#26241d;">Variante:</span>
        <button id="sim_ep0307_btn_v1" style="padding:5px 12px;font-size:11px;font-weight:700;border:1px solid #b9770e;background:#fef5e7;color:#b9770e;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">g = f − ∇²f (Realce Padrão)</button>
        <button id="sim_ep0307_btn_v2" style="padding:5px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">g = f + ∇²f (Inverte Sinal)</button>
      </div>
      <button id="sim_ep0307_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Novo Degrau</button>
    </div>

    <!-- Grid Principal de Comparação (2 colunas + setas) -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem Original f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">① Imagem Original f</span>
        <span style="font-size:10px;color:#8a8371;display:block;margin-bottom:10px;">Degrau com ruído leve</span>
        <div id="sim_ep0307_grid_f" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Laplaciano ∇²f -->
      <div style="background:#fafaf7;border:2px solid #b9770e;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#b9770e;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">② Laplaciano ∇²f</span>
        <span style="font-size:10px;color:#b9770e;display:block;margin-bottom:10px;">Bordas detectadas (±128 shift)</span>
        <div id="sim_ep0307_grid_l" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Linha de Resultado g e Kernel w4 -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Resultado g -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;" id="sim_ep0307_flabel">③ Resultado g = f − ∇²f</span>
        <span style="font-size:10px;color:#2980b9;display:block;margin-bottom:10px;">Passe o mouse para inspecionar</span>
        <div id="sim_ep0307_grid_res" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Estrutura do Kernel w4 -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#5e5a4a;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:6px;font-family:monospace;">Kernel w4 (4-Vizinhos)</span>
        <div style="display:inline-grid;grid-template-columns:repeat(3, 30px);gap:2px;margin-bottom:6px;justify-content:center;">
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">+1</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">+1</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#b9770e;border:1px solid #b9770e;border-radius:4px;color:#ffffff;">−4</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">+1</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">+1</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
        </div>
        <span style="font-size:10px;color:#8a8371;display:block;font-family:monospace;">∇²f = T + B + L + R − 4·f</span>
      </div>

    </div>

    <!-- Legenda -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Legenda:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fef5e7;border:1.5px dashed #b9770e;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">4-Vizinhos do Kernel</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#faece7;border:1.5px solid #c0392b;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Pixel Central</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#eafaf1;border:1.5px solid #27ae60;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Borda (Copiada)</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0307_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:10px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;min-height:36px;line-height:1.5;">
      Passe o mouse sobre um pixel interno do resultado para detalhar a equação.
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0307(root){
    if (!root || root.dataset.simEp0307Init) return;
    root.dataset.simEp0307Init = "1";

    var ROWS = 5, COLS = 5, N = 25;
    var pixels = [], lapV = [], lapA = [], result = [];
    var variant = 'subtract';

    var gridF   = root.querySelector('#sim_ep0307_grid_f');
    var gridL   = root.querySelector('#sim_ep0307_grid_l');
    var gridRes = root.querySelector('#sim_ep0307_grid_res');
    var debug   = root.querySelector('#sim_ep0307_debug');
    var fLabel  = root.querySelector('#sim_ep0307_flabel');
    var btnV1   = root.querySelector('#sim_ep0307_btn_v1');
    var btnV2   = root.querySelector('#sim_ep0307_btn_v2');
    var btnNew  = root.querySelector('#sim_ep0307_btnNew');

    function generate() {
      var sc = 2 + Math.floor(Math.random() * 2);
      var dark = 40 + Math.floor(Math.random() * 30);
      var light = 160 + Math.floor(Math.random() * 40);
      pixels = Array.from({ length: N }, function(_, i) {
        var c = i % COLS;
        var v = c < sc ? dark : light;
        v += Math.floor(Math.random() * 14) - 7;
        return Math.max(0, Math.min(255, v));
      });
      calculate();
    }

    function calculate() {
      lapV = new Array(N).fill(0);
      lapA = new Array(N).fill(0);
      result = pixels.slice();

      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          var i = r * COLS + c;
          if (r >= 1 && r < ROWS - 1 && c >= 1 && c < COLS - 1) {
            var t  = pixels[(r - 1) * COLS + c];
            var b  = pixels[(r + 1) * COLS + c];
            var l  = pixels[r * COLS + (c - 1)];
            var ri = pixels[r * COLS + (c + 1)];
            var f  = pixels[i];
            var lap = t + b + l + ri - 4 * f;

            lapV[i] = lap;
            lapA[i] = Math.max(0, Math.min(255, lap + 128));
            result[i] = Math.max(0, Math.min(255, variant === 'subtract' ? f - lap : f + lap));
          }
        }
      }
    }

    function textColor(g){ return g > 140 ? '#000000' : '#ffffff'; }

    function highlightCross(tr, tc, on) {
      var cells = gridF.children;
      for (var i = 0; i < N; i++) {
        if (!cells[i]) continue;
        var p = pixels[i];
        cells[i].style.background = 'rgb(' + p + ',' + p + ',' + p + ')';
        cells[i].style.color = textColor(p);
        cells[i].style.boxShadow = 'none';
      }
      if (on) {
        var ci = tr * COLS + tc;
        if (cells[ci]) {
          cells[ci].style.background = '#faece7';
          cells[ci].style.color = '#c0392b';
          cells[ci].style.boxShadow = '0 0 0 2px #c0392b inset';
        }
        var neighbors = [[tr - 1, tc], [tr + 1, tc], [tr, tc - 1], [tr, tc + 1]];
        neighbors.forEach(function(n) {
          var nr = n[0], nc = n[1];
          if (nr >= 0 && nr < ROWS && nc >= 0 && nc < COLS) {
            var ni = nr * COLS + nc;
            if (cells[ni]) {
              cells[ni].style.background = '#fef5e7';
              cells[ni].style.color = '#b9770e';
              cells[ni].style.boxShadow = '0 0 0 2px #b9770e inset';
            }
          }
        });
      }
    }

    function resetDebug() {
      debug.textContent = 'Passe o mouse sobre um pixel interno do resultado para detalhar a equação.';
    }

    function render() {
      gridF.innerHTML = '';
      gridL.innerHTML = '';
      gridRes.innerHTML = '';
      fLabel.textContent = variant === 'subtract' ? '③ Resultado g = f − ∇²f' : '③ Resultado g = f + ∇²f';

      for (var i = 0; i < N; i++) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var p = pixels[i], lv = lapV[i], la = lapA[i], res = result[i];
        var isBorder = (r === 0 || r === ROWS - 1 || c === 0 || c === COLS - 1);

        // Célula Original f
        var cf = document.createElement('div');
        cf.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + textColor(p) + ';box-sizing:border-box;';
        cf.textContent = p;
        gridF.appendChild(cf);

        // Célula Laplaciano l
        var cl = document.createElement('div');
        cl.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;';
        if (isBorder) {
          cl.style.background = '#fafaf7';
          cl.style.color = '#8a8371';
          cl.style.border = '1px solid #e4dcc8';
          cl.textContent = '—';
        } else {
          cl.style.background = 'rgb(' + la + ',' + la + ',' + la + ')';
          cl.style.color = textColor(la);
          cl.style.border = '1px solid #e4dcc8';
          cl.textContent = lv;
        }
        gridL.appendChild(cl);

        // Célula Resultado g
        var cr = document.createElement('div');
        cr.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;transition:all 0.15s ease;';

        if (isBorder) {
          cr.style.background = '#eafaf1';
          cr.style.border = '1.5px solid #27ae60';
          cr.style.color = '#27ae60';
          cr.textContent = res;

          (function(row, col, val){
            cr.addEventListener('mouseenter', function(){
              debug.innerHTML = 'Pixel borda (' + row + ',' + col + '): valor herdado sem cálculo &nbsp;→&nbsp; <b>' + val + '</b>';
            });
            cr.addEventListener('mouseleave', function(){ resetDebug(); });
          })(r, c, res);
        } else {
          cr.style.background = 'rgb(' + res + ',' + res + ',' + res + ')';
          cr.style.color = textColor(res);
          cr.style.border = '1px solid #e4dcc8';
          cr.style.cursor = 'pointer';
          cr.textContent = res;

          (function(row, col, val, lapVal, origVal){
            var tVal = pixels[(row - 1) * COLS + col];
            var bVal = pixels[(row + 1) * COLS + col];
            var lVal = pixels[row * COLS + (col - 1)];
            var rVal = pixels[row * COLS + (col + 1)];

            cr.addEventListener('mouseenter', function(){
              highlightCross(row, col, true);
              cr.style.transform = 'scale(1.12)';
              cr.style.boxShadow = '0 0 0 2px #2980b9 inset';
              cr.style.background = '#ebf4fd';
              cr.style.color = '#2980b9';

              var sign = variant === 'subtract' ? '−' : '+';
              debug.innerHTML = '∇²f = (' + tVal + ' + ' + bVal + ' + ' + lVal + ' + ' + rVal + ') − 4·' + origVal + ' = <b>' + lapVal + '</b> &nbsp;|&nbsp; g = clip(' + origVal + ' ' + sign + ' ' + lapVal + ') = <b>' + val + '</b>';
            });

            cr.addEventListener('mouseleave', function(){
              highlightCross(row, col, false);
              cr.style.transform = 'scale(1)';
              cr.style.boxShadow = 'none';
              cr.style.background = 'rgb(' + val + ',' + val + ',' + val + ')';
              cr.style.color = textColor(val);
              resetDebug();
            });
          })(r, c, res, lv, p);
        }
        gridRes.appendChild(cr);
      }
    }

    function setVariant(v) {
      variant = v;
      if (v === 'subtract') {
        btnV1.style.background = '#fef5e7';
        btnV1.style.borderColor = '#b9770e';
        btnV1.style.color = '#b9770e';
        btnV1.style.fontWeight = '700';

        btnV2.style.background = '#f1ead7';
        btnV2.style.borderColor = '#e4dcc8';
        btnV2.style.color = '#5e5a4a';
        btnV2.style.fontWeight = '600';
      } else {
        btnV2.style.background = '#fef5e7';
        btnV2.style.borderColor = '#b9770e';
        btnV2.style.color = '#b9770e';
        btnV2.style.fontWeight = '700';

        btnV1.style.background = '#f1ead7';
        btnV1.style.borderColor = '#e4dcc8';
        btnV1.style.color = '#5e5a4a';
        btnV1.style.fontWeight = '600';
      }
      calculate();
      render();
    }

    btnV1.addEventListener('click', function(){ setVariant('subtract'); });
    btnV2.addEventListener('click', function(){ setVariant('add'); });

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0307(){
    var root = document.getElementById('sim-ep0307-laplaciano');
    if (root) initSimEP0307(root); else setTimeout(tryInitSimEP0307, 200);
  }
  tryInitSimEP0307();
})();
</script>
</div>
""")

**Figura 3.32:** Simulador EP03_07: Operador Laplaciano (w4) para Realce de Bordas


<figure id="fig-03-sim-ep0307-laplaciano">
  <img src="imagens/fig-03-sim-ep0307-laplaciano.png" alt=" Simulador EP03_07: Operador Laplaciano (w4) para Realce de Bordas " style="max-width:80%" />
  <figcaption><strong>Figura 3.32:</strong>  Simulador EP03_07: Operador Laplaciano (w4) para Realce de Bordas </figcaption>
</figure>

In [103]:
%%writefile EP03_07.cpp
// sua solução

Overwriting EP03_07.cpp


In [104]:
TestSuite("EP03_07.cpp").run()


### 3.0.8 EP03_08 🧭 Gradiente de Sobel: Gx e Gy

Em robôs exploradores de Marte (como o Perseverance), a detecção de obstáculos é realizada em tempo real por câmeras estereoscópicas. O **operador de Sobel** calcula o gradiente direcional da cena e é utilizado no algoritmo de detecção de bordas para identificar rochas, fissuras e desníveis do terreno que possam comprometer a navegação.

Ver na [Figura 3.33](#fig-03-sim-ep0308-sobel) uma simulação deste EP.


#### 3.0.8.1 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Dados:** Ler a matriz $f$.
3. **Gx e Gy:** Para cada pixel **interno** $(i,j)$ com $1 \le i < L-1$, $1 \le j < C-1$:

$$G_x(i,j) = [f(i-1,j+1) + 2f(i,j+1) + f(i+1,j+1)] - [f(i-1,j-1) + 2f(i,j-1) + f(i+1,j-1)]$$

$$G_y(i,j) = [f(i+1,j-1) + 2f(i+1,j) + f(i+1,j+1)] - [f(i-1,j-1) + 2f(i-1,j) + f(i-1,j+1)]$$

4. **Magnitude:** $|\nabla f(i,j)| = \text{clip}(\text{round}(\sqrt{G_x^2 + G_y^2}))$.
5. **Borda:** Pixels de borda recebem magnitude 0.
6. **Saída:** Exibir a magnitude $L \times C$.

#### 3.0.8.2 📌 Restrições Computacionais

* **Arredondamento:** Aplicar `round` antes de converter para inteiro.
* **Saturação:** $\text{clip}(x) = \max(0, \min(255, x))$.
* **Raiz quadrada:** Usar $\sqrt{G_x^2 + G_y^2}$ (não a aproximação $|G_x| + |G_y|$).

#### 3.0.8.3 🧠 Fundamentação Teórica

| Operador | Detecta | Coeficientes diagonais |
|:--------:|:-------:|:----------------------:|
| $G_x$ | Bordas verticais | $\pm 1$ |
| $G_y$ | Bordas horizontais | $\pm 1$ |
| $|\nabla f|$ | Todas as bordas | Combinado |

#### 3.0.8.4 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linhas seguintes: Elementos da matriz.

**Saída:**

* Magnitude do gradiente, matriz $L \times C$.

#### 3.0.8.5 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>0 0 0<br>0 0 0<br>0 0 0 | 0 0 0<br>0 0 0<br>0 0 0 | Imagem nula: gradiente zero |
| 3<br>3<br>0 0 255<br>0 0 255<br>0 0 255 | 0 0 0<br>0 255 0<br>0 0 0 | Borda vertical central: Gx alto |

In [105]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0308-sobel" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧭 Simulador EP03_08: Gradiente de Sobel (Gx e Gy)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">|∇f| = √(Gx² + Gy²)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Analise a decomposição horizontal (Gx) e vertical (Gy) do operador de Sobel e passe o mouse sobre os pixels da magnitude para inspecionar a vizinhança 3×3.</p>

    <!-- Barra de Controles / Kernels Explicativos -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;align-items:center;gap:12px;flex-wrap:wrap;justify-content:space-between;">
      
      <div style="display:flex;align-items:center;gap:12px;flex-wrap:wrap;">
        <button id="sim_ep0308_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nova Cena</button>
        <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Kernels de Sobel:</span>
      </div>

      <!-- Representação Visual dos Kernels Gx e Gy -->
      <div style="display:flex;align-items:center;gap:16px;flex-wrap:wrap;">
        
        <!-- Kernel Gx -->
        <div style="display:flex;align-items:center;gap:6px;">
          <div style="display:inline-grid;grid-template-columns:repeat(3, 26px);gap:2px;background:#f1ead7;border-radius:6px;padding:4px;">
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#0c447c;color:#b5d4f4;border-radius:3px;">−1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#185fa5;color:#e6f1fb;border-radius:3px;">+1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#0c447c;color:#b5d4f4;border-radius:3px;">−2</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#185fa5;color:#e6f1fb;border-radius:3px;">+2</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#0c447c;color:#b5d4f4;border-radius:3px;">−1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#185fa5;color:#e6f1fb;border-radius:3px;">+1</div>
          </div>
          <span style="font-size:11px;font-weight:700;color:#2980b9;font-family:monospace;">Gx</span>
        </div>

        <!-- Kernel Gy -->
        <div style="display:flex;align-items:center;gap:6px;">
          <div style="display:inline-grid;grid-template-columns:repeat(3, 26px);gap:2px;background:#f1ead7;border-radius:6px;padding:4px;">
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#633806;color:#fac775;border-radius:3px;">−1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#854f0b;color:#fac775;border-radius:3px;">−2</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#633806;color:#fac775;border-radius:3px;">−1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ba7517;color:#faeeda;border-radius:3px;">+1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ef9f27;color:#412402;border-radius:3px;">+2</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ba7517;color:#faeeda;border-radius:3px;">+1</div>
          </div>
          <span style="font-size:11px;font-weight:700;color:#b9770e;font-family:monospace;">Gy</span>
        </div>

      </div>

    </div>

    <!-- Comparativo em 2 Linhas (Original, Magnitude, Gx, Gy) -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem Original f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Imagem Original f</span>
        <span style="font-size:10px;color:#8a8371;display:block;margin-bottom:10px;">Matriz 5×5 pixels</span>
        <div id="sim_ep0308_grid_f" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Magnitude do Gradiente |∇f| -->
      <div style="background:#fafaf7;border:2px solid #27ae60;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Magnitude |∇f|</span>
        <span style="font-size:10px;color:#27ae60;display:block;margin-bottom:10px;">√(Gx² + Gy²)</span>
        <div id="sim_ep0308_grid_m" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Gradiente Horizontal Gx -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Gx — Gradiente Horizontal</span>
        <span style="font-size:10px;color:#2980b9;display:block;margin-bottom:10px;">Azul = Negativo · Branco = Zero · Azul Vivo = Positivo</span>
        <div id="sim_ep0308_grid_gx" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Gradiente Vertical Gy -->
      <div style="background:#fafaf7;border:2px solid #b9770e;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#b9770e;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Gy — Gradiente Vertical</span>
        <span style="font-size:10px;color:#b9770e;display:block;margin-bottom:10px;">Âmbar = Negativo · Branco = Zero · Âmbar Vivo = Positivo</span>
        <div id="sim_ep0308_grid_gy" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Legenda -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Legenda:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fef5e7;border:1.5px dashed #b9770e;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Vizinhança 3×3 Inspecionada</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#ebf4fd;border:1.5px solid #2980b9;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Pixel Central</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#26241d;border:1.5px solid #8a8371;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Borda (Forçada a 0)</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0308_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:10px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;min-height:36px;line-height:1.5;">
      Passe o mouse sobre um pixel interno da magnitude para ver a decomposição Gx e Gy.
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0308(root){
    if (!root || root.dataset.simEp0308Init) return;
    root.dataset.simEp0308Init = "1";

    var ROWS = 5, COLS = 5, N = 25;
    var pixels = [], gxV = [], gyV = [], mgV = [];

    var gridF  = root.querySelector('#sim_ep0308_grid_f');
    var gridGx = root.querySelector('#sim_ep0308_grid_gx');
    var gridGy = root.querySelector('#sim_ep0308_grid_gy');
    var gridM  = root.querySelector('#sim_ep0308_grid_m');
    var debug  = root.querySelector('#sim_ep0308_debug');
    var btnNew = root.querySelector('#sim_ep0308_btnNew');

    function generate() {
      var block = Math.random() > 0.3;
      var bg = 30 + Math.floor(Math.random() * 30);
      var obj = 180 + Math.floor(Math.random() * 50);

      pixels = Array.from({ length: N }, function(_, i) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var v = bg;
        if (block && r >= 2 && r <= 3 && c >= 2 && c <= 3) v = obj;
        else if (!block && r + c >= 4) v = obj - 40;
        v += Math.floor(Math.random() * 10) - 5;
        return Math.max(0, Math.min(255, v));
      });
      calculate();
    }

    function calculate() {
      gxV = new Array(N).fill(0);
      gyV = new Array(N).fill(0);
      mgV = new Array(N).fill(0);

      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          var i = r * COLS + c;
          if (r >= 1 && r < ROWS - 1 && c >= 1 && c < COLS - 1) {
            var p = [
              pixels[(r - 1) * COLS + (c - 1)], pixels[(r - 1) * COLS + c], pixels[(r - 1) * COLS + (c + 1)],
              pixels[r * COLS + (c - 1)],       pixels[r * COLS + c],       pixels[r * COLS + (c + 1)],
              pixels[(r + 1) * COLS + (c - 1)], pixels[(r + 1) * COLS + c], pixels[(r + 1) * COLS + (c + 1)]
            ];
            var gx = (p[2] + 2 * p[5] + p[8]) - (p[0] + 2 * p[3] + p[6]);
            var gy = (p[6] + 2 * p[7] + p[8]) - (p[0] + 2 * p[1] + p[2]);

            gxV[i] = gx;
            gyV[i] = gy;
            mgV[i] = Math.min(255, Math.round(Math.sqrt(gx * gx + gy * gy)));
          }
        }
      }
    }

    function textColor(g){ return g > 150 ? '#000000' : '#ffffff'; }

    function colorGxBetter(v) {
      var n = Math.max(-400, Math.min(400, v));
      if (Math.abs(n) < 15) return '#fafaf7';
      if (n > 0) {
        var t = Math.min(1, n / 350);
        var r = Math.round(4 + t * 20);
        var g = Math.round(44 + t * 71);
        var b = Math.round(83 + t * 89);
        return 'rgb(' + r + ',' + g + ',' + b + ')';
      } else {
        var t = Math.min(1, -n / 350);
        return 'rgb(' + Math.round(12 + t * 0) + ',' + Math.round(68 - t * 24) + ',' + Math.round(165 - t * 82) + ')';
      }
    }

    function colorGyBetter(v) {
      var n = Math.max(-400, Math.min(400, v));
      if (Math.abs(n) < 15) return '#fafaf7';
      if (n > 0) {
        var t = Math.min(1, n / 350);
        return 'rgb(' + Math.round(186 + t * 63) + ',' + Math.round(117 + t * 70) + ',' + Math.round(23 - t * 18) + ')';
      } else {
        var t = Math.min(1, -n / 350);
        return 'rgb(' + Math.round(99 + t * 0) + ',' + Math.round(56 - t * 20) + ',' + Math.round(11 - t * 5) + ')';
      }
    }

    function textSignedColor(bg) {
      if (bg === '#fafaf7') return '#26241d';
      var m = bg.match(/rgb\((\d+),(\d+),(\d+)\)/);
      if (!m) return '#ffffff';
      var lum = 0.299 * m[1] + 0.587 * m[2] + 0.114 * m[3];
      return lum > 140 ? '#000000' : '#ffffff';
    }

    function highlight(tr, tc, on) {
      var cells = gridF.children;
      for (var i = 0; i < N; i++) {
        if (!cells[i]) continue;
        var p = pixels[i], r = Math.floor(i / COLS), c = i % COLS;
        if (on && Math.abs(r - tr) <= 1 && Math.abs(c - tc) <= 1) {
          if (r === tr && c === tc) {
            cells[i].style.background = '#ebf4fd';
            cells[i].style.color = '#2980b9';
            cells[i].style.boxShadow = '0 0 0 2px #2980b9 inset';
          } else {
            cells[i].style.background = '#fef5e7';
            cells[i].style.color = '#b9770e';
            cells[i].style.boxShadow = '0 0 0 2px #b9770e inset';
          }
        } else {
          cells[i].style.background = 'rgb(' + p + ',' + p + ',' + p + ')';
          cells[i].style.color = textColor(p);
          cells[i].style.boxShadow = 'none';
        }
      }
    }

    function resetDebug() {
      debug.textContent = 'Passe o mouse sobre um pixel interno da magnitude para ver a decomposição Gx e Gy.';
    }

    function render() {
      gridF.innerHTML = '';
      gridGx.innerHTML = '';
      gridGy.innerHTML = '';
      gridM.innerHTML = '';

      for (var i = 0; i < N; i++) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var p = pixels[i], gx = gxV[i], gy = gyV[i], mag = mgV[i];
        var isBorder = (r === 0 || r === ROWS - 1 || c === 0 || c === COLS - 1);

        // Célula Original f
        var cf = document.createElement('div');
        cf.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + textColor(p) + ';box-sizing:border-box;';
        cf.textContent = p;
        gridF.appendChild(cf);

        // Célula Gx
        var cgx = document.createElement('div');
        cgx.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;';
        if (isBorder) {
          cgx.style.background = '#fafaf7';
          cgx.style.color = '#8a8371';
          cgx.style.border = '1px solid #e4dcc8';
          cgx.textContent = '—';
        } else {
          var bgGx = colorGxBetter(gx);
          cgx.style.background = bgGx;
          cgx.style.color = textSignedColor(bgGx);
          cgx.style.border = '1px solid #e4dcc8';
          cgx.textContent = (gx > 0 ? '+' : '') + gx;
        }
        gridGx.appendChild(cgx);

        // Célula Gy
        var cgy = document.createElement('div');
        cgy.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;';
        if (isBorder) {
          cgy.style.background = '#fafaf7';
          cgy.style.color = '#8a8371';
          cgy.style.border = '1px solid #e4dcc8';
          cgy.textContent = '—';
        } else {
          var bgGy = colorGyBetter(gy);
          cgy.style.background = bgGy;
          cgy.style.color = textSignedColor(bgGy);
          cgy.style.border = '1px solid #e4dcc8';
          cgy.textContent = (gy > 0 ? '+' : '') + gy;
        }
        gridGy.appendChild(cgy);

        // Célula Magnitude m
        var cm = document.createElement('div');
        cm.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;transition:all 0.15s ease;';
        if (isBorder) {
          cm.style.background = '#26241d';
          cm.style.border = '1.5px solid #8a8371';
          cm.style.color = '#7ee7c6';
          cm.textContent = '0';

          (function(row, col){
            cm.addEventListener('mouseenter', function(){
              debug.innerHTML = 'Pixel borda (' + row + ',' + col + '): sem vizinhança completa → forçado para <b>0</b>';
            });
            cm.addEventListener('mouseleave', function(){ resetDebug(); });
          })(r, c);
        } else {
          cm.style.background = 'rgb(' + mag + ',' + mag + ',' + mag + ')';
          cm.style.color = textColor(mag);
          cm.style.border = '1px solid #e4dcc8';
          cm.style.cursor = 'pointer';
          cm.textContent = mag;

          (function(row, col, gxv, gyv, mgv){
            cm.addEventListener('mouseenter', function(){
              highlight(row, col, true);
              cm.style.transform = 'scale(1.12)';
              cm.style.boxShadow = '0 0 0 2px #27ae60 inset';
              cm.style.background = '#eafaf1';
              cm.style.color = '#27ae60';

              debug.innerHTML = 'Pixel (' + row + ',' + col + '): Gx = <b>' + gxv + '</b> &nbsp;|&nbsp; Gy = <b>' + gyv + '</b> &nbsp;|&nbsp; |∇f| = round(√(' + gxv + '² + ' + gyv + '²)) = <b>' + mgv + '</b>';
            });

            cm.addEventListener('mouseleave', function(){
              highlight(row, col, false);
              cm.style.transform = 'scale(1)';
              cm.style.boxShadow = 'none';
              cm.style.background = 'rgb(' + mgv + ',' + mgv + ',' + mgv + ')';
              cm.style.color = textColor(mgv);
              resetDebug();
            });
          })(r, c, gx, gy, mag);
        }
        gridM.appendChild(cm);
      }
    }

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0308(){
    var root = document.getElementById('sim-ep0308-sobel');
    if (root) initSimEP0308(root); else setTimeout(tryInitSimEP0308, 200);
  }
  tryInitSimEP0308();
})();
</script>
</div>
""")

**Figura 3.33:** Simulador EP03_08: Gradiente de Sobel (Gx e Gy)


<figure id="fig-03-sim-ep0308-sobel">
  <img src="imagens/fig-03-sim-ep0308-sobel.png" alt=" Simulador EP03_08: Gradiente de Sobel (Gx e Gy) " style="max-width:80%" />
  <figcaption><strong>Figura 3.33:</strong>  Simulador EP03_08: Gradiente de Sobel (Gx e Gy) </figcaption>
</figure>

In [106]:
%%writefile EP03_08.cpp
// sua solução

Overwriting EP03_08.cpp


In [107]:
TestSuite("EP03_08.cpp").run()


### 3.0.9 EP03_09 📡 Filtro da Mediana 3×3

Imagens de radar de abertura sintética (SAR) usadas em monitoramento ambiental e militar sofrem de um tipo específico de ruído chamado *speckle*, que possui características similares ao ruído sal e pimenta. O **filtro da mediana** é o método padrão de remoção desse ruído porque preserva as bordas das estruturas enquanto elimina os pontos espúrios.

Ver na [Figura 3.34](#fig-03-sim-ep0309-mediana) uma simulação deste EP.


#### 3.0.9.1 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Dados:** Ler a matriz de pixels $f$.
3. **Filtro da Mediana 3×3:** Para cada pixel **interno** $(i,j)$ com $1 \le i < L-1$, $1 \le j < C-1$:
   - Coletar os 9 pixels da vizinhança $3 \times 3$: $\{f(i+s, j+t) : s,t \in \{-1,0,1\}\}$.
   - Ordenar os 9 valores em ordem crescente.
   - Atribuir $g(i,j)$ ao valor central (posição índice 4, considerando índice 0).

$$g(i,j) = \text{mediana}\{f(i+s, j+t) : s,t \in \{-1,0,1\}\}$$

4. **Borda:** Copiar diretamente: $g(i,j) = f(i,j)$.
5. **Saída:** Exibir a matriz filtrada $L \times C$.

#### 3.0.9.2 📌 Restrições Computacionais

* **Janela:** Sempre $3 \times 3 = 9$ elementos.
* **Mediana:** O elemento central da sequência ordenada (índice 4 de 0 a 8).
* **Sem clipping:** A mediana de valores em $[0, 255]$ permanece em $[0, 255]$.
* **Não linear:** O filtro mediana não pode ser expresso como convolução linear.

#### 3.0.9.3 🧠 Fundamentação Teórica

| Ruído | Filtro de Média | Filtro de Mediana |
|:-----:|:---------------:|:-----------------:|
| Sal e pimenta (0 ou 255) | Espalha o ruído | Remove sem distorcer bordas |
| Gaussiano | Reduz eficazmente | Reduz parcialmente |

#### 3.0.9.4 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linhas seguintes: Elementos da matriz.

**Saída:**

* Matriz filtrada $L \times C$.

#### 3.0.9.5 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>100 100 100<br>100 0 100<br>100 100 100 | 100 100 100<br>100 100 100<br>100 100 100 | Ponto preto eliminado: mediana de 8×100+1×0 = 100 |
| 3<br>3<br>50 50 50<br>50 255 50<br>50 50 50 | 50 50 50<br>50 50 50<br>50 50 50 | Ponto branco (sal) eliminado |

In [108]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0309-mediana" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">📉 Simulador EP03_09: Filtro da Mediana 3×3</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = Mediana(Vizinhos)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Injete ruído impulsivo (sal e pimenta) e passe o mouse sobre os pixels internos do resultado para inspecionar a ordenação do vetor de vizinhança e a eliminação do ruído.</p>

    <!-- Barra de Controles / Ação -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;align-items:center;gap:12px;flex-wrap:wrap;justify-content:space-between;">
      <button id="sim_ep0309_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Injetar Ruído Impulsivo</button>
      <span style="font-size:11px;color:#8a8371;">Ruído sal (255) e pimenta (0) — ~30% dos pixels internos afetados</span>
    </div>

    <!-- Comparativo Lado a Lado: Original com Ruído vs Resultado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem f com Ruído -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Imagem f — Com Ruído</span>
        <span style="font-size:10px;color:#8a8371;display:block;margin-bottom:10px;">Sal (255) e pimenta (0) visíveis</span>
        <div id="sim_ep0309_grid_f" style="display:grid;grid-template-columns:repeat(5, 38px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Resultado g sem Ruído -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Resultado g — Sem Ruído</span>
        <span style="font-size:10px;color:#2980b9;display:block;margin-bottom:10px;">Passe o mouse para inspecionar</span>
        <div id="sim_ep0309_grid_result" style="display:grid;grid-template-columns:repeat(5, 38px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Vetor Ordenado -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:14px;text-align:center;">
      <span style="font-size:10px;font-weight:700;color:#5e5a4a;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:3px;">Vetor de Vizinhança 3×3 — Ordenado</span>
      <span style="font-size:10.5px;color:#8a8371;display:block;margin-bottom:8px;">Passe o mouse num píxel interno do resultado para visualizar</span>
      <div id="sim_ep0309_vector" style="display:flex;flex-wrap:wrap;justify-content:center;gap:3px;min-height:32px;padding:4px 0;align-items:center;">
        <span style="font-size:11px;color:#8a8371;font-style:italic;">—</span>
      </div>
    </div>

    <!-- Legenda -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Legenda:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fef5e7;border:1.5px dashed #b9770e;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Janela 3×3</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#ebf4fd;border:1.5px solid #2980b9;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Pixel Central</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#eafaf1;border:1.5px solid #27ae60;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Borda (Copiada)</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#eafaf1;border:1.5px solid #27ae60;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Mediana</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#26241d;border:1.5px solid #c0392b;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Ruído (Eliminado)</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0309_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:10px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;min-height:36px;line-height:1.5;">
      Passe o mouse sobre um píxel interno do resultado para ver o processo de ordenação.
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0309(root){
    if (!root || root.dataset.simEp0309Init) return;
    root.dataset.simEp0309Init = "1";

    var ROWS = 5, COLS = 5, N = 25, RADIUS = 1;
    var pixels = [], result = [];

    var gridF  = root.querySelector('#sim_ep0309_grid_f');
    var gridRes= root.querySelector('#sim_ep0309_grid_result');
    var vector = root.querySelector('#sim_ep0309_vector');
    var debug  = root.querySelector('#sim_ep0309_debug');
    var btnNew = root.querySelector('#sim_ep0309_btnNew');

    function generate() {
      pixels = Array.from({ length: N }, function(){
        var v = 110 + Math.floor(Math.random() * 30);
        var rnd = Math.random();
        if (rnd > 0.82) v = 255;
        else if (rnd < 0.18) v = 0;
        return v;
      });
      calculate();
    }

    function calculate() {
      result = pixels.slice();
      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          if (r >= RADIUS && r < ROWS - RADIUS && c >= RADIUS && c < COLS - RADIUS) {
            var win = [];
            for (var s = -RADIUS; s <= RADIUS; s++) {
              for (var t = -RADIUS; t <= RADIUS; t++) {
                win.push(pixels[(r + s) * COLS + (c + t)]);
              }
            }
            win.sort(function(a, b){ return a - b; });
            result[r * COLS + c] = win[4];
          }
        }
      }
    }

    function textColor(g){ return g > 140 ? '#000000' : '#ffffff'; }
    function isNoise(v){ return v === 0 || v === 255; }

    function highlight(tr, tc, on) {
      var cells = gridF.children;
      for (var i = 0; i < N; i++) {
        if (!cells[i]) continue;
        var p = pixels[i], r = Math.floor(i / COLS), c = i % COLS;
        if (on && Math.abs(r - tr) <= 1 && Math.abs(c - tc) <= 1) {
          if (r === tr && c === tc) {
            cells[i].style.background = '#ebf4fd';
            cells[i].style.color = '#2980b9';
            cells[i].style.boxShadow = '0 0 0 2px #2980b9 inset';
          } else {
            cells[i].style.background = '#fef5e7';
            cells[i].style.color = '#b9770e';
            cells[i].style.boxShadow = '0 0 0 2px #b9770e inset';
          }
        } else {
          if (isNoise(p)) {
            cells[i].style.background = p === 255 ? '#ffffff' : '#26241d';
            cells[i].style.color = p === 255 ? '#c0392b' : '#e74c3c';
            cells[i].style.border = '2px solid #c0392b';
          } else {
            cells[i].style.background = 'rgb(' + p + ',' + p + ',' + p + ')';
            cells[i].style.color = textColor(p);
            cells[i].style.border = '1px solid #e4dcc8';
          }
          cells[i].style.boxShadow = 'none';
        }
      }
    }

    function showVector(raw, sorted, median) {
      vector.innerHTML = '';

      var rawLabel = document.createElement('span');
      rawLabel.style.cssText = 'font-size:10px;color:#8a8371;margin-right:6px;font-family:monospace;';
      rawLabel.textContent = 'Bruto:';
      vector.appendChild(rawLabel);

      raw.forEach(function(v){
        var d = document.createElement('div');
        var noise = isNoise(v);
        d.style.cssText = 'display:inline-flex;align-items:center;justify-content:center;width:28px;height:24px;border-radius:4px;font-size:10px;font-weight:700;font-family:monospace;margin:1px;';
        d.style.background = noise ? '#26241d' : '#fafaf7';
        d.style.color = noise ? '#e74c3c' : '#26241d';
        d.style.border = noise ? '1.5px solid #c0392b' : '1px solid #e4dcc8';
        d.textContent = v;
        vector.appendChild(d);
      });

      var arr = document.createElement('span');
      arr.style.cssText = 'font-size:14px;margin:0 6px;color:#8a8371;font-weight:700;';
      arr.textContent = '→';
      vector.appendChild(arr);

      var sortLabel = document.createElement('span');
      sortLabel.style.cssText = 'font-size:10px;color:#8a8371;margin-right:6px;font-family:monospace;';
      sortLabel.textContent = 'Ordenado:';
      vector.appendChild(sortLabel);

      sorted.forEach(function(v, i){
        var d = document.createElement('div');
        var isMedian = (i === 4);
        var noise = isNoise(v);
        d.style.cssText = 'display:inline-flex;align-items:center;justify-content:center;width:28px;height:24px;border-radius:4px;font-size:10px;font-weight:700;font-family:monospace;margin:1px;';
        if (isMedian) {
          d.style.background = '#eafaf1';
          d.style.color = '#27ae60';
          d.style.border = '2px solid #27ae60';
        } else if (noise) {
          d.style.background = '#26241d';
          d.style.color = '#e74c3c';
          d.style.border = '1.5px solid #c0392b';
        } else {
          d.style.background = '#fafaf7';
          d.style.color = '#26241d';
          d.style.border = '1px solid #e4dcc8';
        }
        d.textContent = v;
        vector.appendChild(d);
      });

      var eq = document.createElement('span');
      eq.style.cssText = 'font-size:11px;margin-left:8px;font-family:monospace;color:#27ae60;font-weight:700;';
      eq.innerHTML = '→ mediana = <b>' + median + '</b>';
      vector.appendChild(eq);
    }

    function resetVector() {
      vector.innerHTML = '<span style="font-size:11px;color:#8a8371;font-style:italic;">—</span>';
    }

    function resetDebug() {
      debug.textContent = 'Passe o mouse sobre um píxel interno do resultado para ver o processo de ordenação.';
    }

    function render() {
      gridF.innerHTML = '';
      gridRes.innerHTML = '';

      for (var i = 0; i < N; i++) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var p = pixels[i], res = result[i];
        var isBorder = (r < RADIUS || r >= ROWS - RADIUS || c < RADIUS || c >= COLS - RADIUS);
        var noise = isNoise(p);

        // Célula F com Ruído
        var cf = document.createElement('div');
        if (noise) {
          cf.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;user-select:none;background:' + (p === 255 ? '#ffffff' : '#26241d') + ';color:' + (p === 255 ? '#c0392b' : '#e74c3c') + ';border:2px solid #c0392b;box-sizing:border-box;';
        } else {
          cf.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + textColor(p) + ';box-sizing:border-box;';
        }
        cf.textContent = p;
        gridF.appendChild(cf);

        // Célula Resultado g
        var cr = document.createElement('div');
        cr.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;transition:all 0.15s ease;';

        if (isBorder) {
          cr.style.background = '#eafaf1';
          cr.style.border = '1.5px solid #27ae60';
          cr.style.color = '#27ae60';
          cr.textContent = res;

          (function(row, col, val){
            cr.addEventListener('mouseenter', function(){
              debug.innerHTML = 'Pixel borda (' + row + ',' + col + '): copiado sem filtro → <b>' + val + '</b>';
              resetVector();
            });
            cr.addEventListener('mouseleave', function(){ resetDebug(); });
          })(r, c, res);
        } else {
          cr.style.background = 'rgb(' + res + ',' + res + ',' + res + ')';
          cr.style.color = textColor(res);
          cr.style.border = '1px solid #e4dcc8';
          cr.style.cursor = 'pointer';
          cr.textContent = res;

          (function(row, col, resVal){
            cr.addEventListener('mouseenter', function(){
              highlight(row, col, true);
              cr.style.transform = 'scale(1.12)';
              cr.style.boxShadow = '0 0 0 2px #2980b9 inset';
              cr.style.background = '#ebf4fd';
              cr.style.color = '#2980b9';

              var raw = [];
              for (var s = -RADIUS; s <= RADIUS; s++) {
                for (var t = -RADIUS; t <= RADIUS; t++) {
                  raw.push(pixels[(row + s) * COLS + (col + t)]);
                }
              }
              var sorted = raw.slice().sort(function(a, b){ return a - b; });
              showVector(raw, sorted, resVal);

              var noiseCount = raw.filter(isNoise).length;
              debug.innerHTML = 'Pixel (' + row + ',' + col + '): ' + noiseCount + ' vizinho(s) com ruído na janela &nbsp;→&nbsp; mediana = posição [4] = <b>' + resVal + '</b>';
            });

            cr.addEventListener('mouseleave', function(){
              highlight(row, col, false);
              cr.style.transform = 'scale(1)';
              cr.style.boxShadow = 'none';
              cr.style.background = 'rgb(' + resVal + ',' + resVal + ',' + resVal + ')';
              cr.style.color = textColor(resVal);
              resetDebug();
              resetVector();
            });
          })(r, c, res);
        }
        gridRes.appendChild(cr);
      }
    }

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0309(){
    var root = document.getElementById('sim-ep0309-mediana');
    if (root) initSimEP0309(root); else setTimeout(tryInitSimEP0309, 200);
  }
  tryInitSimEP0309();
})();
</script>
</div>
""")

**Figura 3.34:** Simulador EP03_09: Filtro da Mediana 3×3


<figure id="fig-03-sim-ep0309-mediana">
  <img src="imagens/fig-03-sim-ep0309-mediana.png" alt=" Simulador EP03_09: Filtro da Mediana 3×3 " style="max-width:80%" />
  <figcaption><strong>Figura 3.34:</strong>  Simulador EP03_09: Filtro da Mediana 3×3 </figcaption>
</figure>

In [109]:
%%writefile EP03_09.cpp
// sua solução

Overwriting EP03_09.cpp


In [110]:
TestSuite("EP03_09.cpp").run()


### 3.0.10 EP03_10 ✨ *Unsharp Masking* (USM)

Em sistemas de digitalização de documentos históricos e obras de arte, a nitidez das imagens é fundamental para leitura de textos manuscritos e detalhes ornamentais. O ***Unsharp Masking* (USM)** é o algoritmo de realce de nitidez padrão utilizado em *scanners* profissionais e softwares como Adobe Photoshop, controlado pelo parâmetro $k$ que determina a intensidade do realce.

Ver na [Figura 3.35](#fig-03-sim-ep0310-unsharp) uma simulação deste EP.


#### 3.0.10.1 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Parâmetro:** Ler o valor real $k$ (intensidade do realce, $k \ge 0$).
3. **Dados:** Ler a matriz de pixels $f$.
4. **Suavização:** Calcular $\bar{f}$ com filtro de média $3\times3$ (apenas pixels internos; bordas mantidas):

$$\bar{f}(i,j) = \frac{1}{9} \sum_{s=-1}^{1} \sum_{t=-1}^{1} f(i+s, j+t)$$

5. **Máscara de alta frequência:** $m(i,j) = f(i,j) - \bar{f}(i,j)$.
6. **Realce USM:** Para cada pixel interno:

$$g(i,j) = \text{clip}\left(\text{round}\left(f(i,j) + k \cdot m(i,j)\right)\right)$$

7. **Borda:** $g(i,j) = f(i,j)$ (cópia direta).
8. **Saída:** Exibir a matriz realçada $L \times C$.

#### 3.0.10.2 📌 Restrições Computacionais

* **Arredondamento:** Aplicar `round` antes do clipping.
* **Saturação:** $\text{clip}(x) = \max(0, \min(255, x))$.
* **Operações em float:** Calcular $\bar{f}$ e $m$ em ponto flutuante antes de arredondar o resultado final.
* **$k = 0$:** Sem realce — a saída é idêntica à entrada (exceto pelas bordas).

#### 3.0.10.3 🧠 Fundamentação Teórica

| Etapa | Operação | Descrição |
|:-----:|:---------|:----------|
| 1 | $\bar{f} = f * \frac{1}{9}\mathbf{1}_{3\times3}$ | Suavização (baixas frequências) |
| 2 | $m = f - \bar{f}$ | Máscara (altas frequências) |
| 3 | $g = \text{clip}(\text{round}(f + k \cdot m))$ | Realce ponderado |

#### 3.0.10.4 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Real $k$.
* Linhas seguintes: Elementos da matriz original.

**Saída:**

* Matriz realçada $L \times C$.

#### 3.0.10.5 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>0.0<br>100 100 100<br>100 100 100<br>100 100 100 | 100 100 100<br>100 100 100<br>100 100 100 | k=0: sem realce |
| 3<br>3<br>1.0<br>50 50 50<br>50 200 50<br>50 50 50 | 50 50 50<br>50 255 50<br>50 50 50 | k=1: pixel central realçado e saturado |

In [111]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0310-unsharp" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">✨ Simulador EP03_10: Unsharp Masking (USM)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f + k · m</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Ajuste o fator de ganho k, observe o pipeline completo de realce (desfocagem, máscara de alta frequência) e passe o mouse sobre o resultado.</p>

    <!-- Barra de Controles / Slider de Ganho k -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;align-items:center;gap:12px;flex-wrap:wrap;justify-content:space-between;">
      
      <div style="display:flex;align-items:center;gap:10px;flex:1;min-width:240px;">
        <span style="font-size:11px;font-weight:700;color:#26241d;">Fator de ganho k:</span>
        <input id="sim_ep0310_k" type="range" min="0.0" max="3.0" step="0.5" value="1.0" style="flex:1;cursor:pointer;accent-color:#2980b9;">
        <span id="sim_ep0310_klabel" style="font-family:monospace;font-size:11px;font-weight:700;background:#26241d;color:#7ee7c6;padding:3px 8px;border-radius:6px;">k = 1.0</span>
      </div>

      <button id="sim_ep0310_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nova Imagem</button>
    </div>

    <!-- Comparativo do Pipeline USM (Grid de 4 Cards) -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- ① Imagem Original f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">① Imagem Original f</span>
        <span style="font-size:10px;color:#8a8371;display:block;margin-bottom:10px;">Matriz 5×5 pixels</span>
        <div id="sim_ep0310_grid_f" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- ② Desfocado f_barra -->
      <div style="background:#fafaf7;border:2px solid #b9770e;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#b9770e;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">② Desfocado f̄</span>
        <span style="font-size:10px;color:#b9770e;display:block;margin-bottom:10px;">Média 3 × 3</span>
        <div id="sim_ep0310_grid_b" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- ③ Máscara m -->
      <div style="background:#fafaf7;border:2px solid #c0392b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">③ Máscara m</span>
        <span style="font-size:10px;color:#c0392b;display:block;margin-bottom:10px;">m = f − f̄ (Altas Frequências)</span>
        <div id="sim_ep0310_grid_m" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- ④ Resultado g -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;" id="sim_ep0310_flabel">④ Resultado g = f + 1.0·m</span>
        <span style="font-size:10px;color:#2980b9;display:block;margin-bottom:10px;">Passe o mouse para inspecionar</span>
        <div id="sim_ep0310_grid_res" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Legenda -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Legenda:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fef5e7;border:1.5px dashed #b9770e;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Vizinhança 3×3</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#ebf4fd;border:1.5px solid #2980b9;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Pixel Central</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#eafaf1;border:1.5px solid #27ae60;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Borda (Copiada)</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fbeaf0;border:1.5px solid #c0392b;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Máscara Positiva/Negativa</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0310_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:10px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;min-height:36px;line-height:1.5;">
      Passe o mouse sobre um píxel interno do resultado para rastrear o pipeline completo.
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0310(root){
    if (!root || root.dataset.simEp0310Init) return;
    root.dataset.simEp0310Init = "1";

    var ROWS = 5, COLS = 5, N = 25, RADIUS = 1;
    var pixels = [], blurred = [], mask = [], result = [];
    var gainK = 1.0;

    var gridF  = root.querySelector('#sim_ep0310_grid_f');
    var gridB  = root.querySelector('#sim_ep0310_grid_b');
    var gridM  = root.querySelector('#sim_ep0310_grid_m');
    var gridRes= root.querySelector('#sim_ep0310_grid_res');
    var debug  = root.querySelector('#sim_ep0310_debug');
    var fLabel = root.querySelector('#sim_ep0310_flabel');
    var sliderK= root.querySelector('#sim_ep0310_k');
    var kLabel = root.querySelector('#sim_ep0310_klabel');
    var btnNew = root.querySelector('#sim_ep0310_btnNew');

    function textColor(g){ return g > 140 ? '#000000' : '#ffffff'; }

    function colorMask(v) {
      if (Math.abs(v) < 2) return '#fafaf7';
      if (v > 0) {
        var t = Math.min(1, v / 80);
        return 'rgb(' + Math.round(230 + t * 20) + ',' + Math.round(210 - t * 60) + ',' + Math.round(220 - t * 60) + ')';
      }
      var t = Math.min(1, -v / 80);
      return 'rgb(' + Math.round(210 - t * 60) + ',' + Math.round(220 - t * 40) + ',' + Math.round(230 + t * 20) + ')';
    }

    function generate() {
      pixels = Array.from({ length: N }, function(_, i) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var v = (r >= 2 && c >= 2) ? 170 : 70;
        v += Math.floor(Math.random() * 16) - 8;
        return Math.max(0, Math.min(255, v));
      });
      calculate();
    }

    function calculate() {
      blurred = new Array(N).fill(0);
      mask = new Array(N).fill(0);
      result = new Array(N).fill(0);

      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          var i = r * COLS + c;
          if (r >= RADIUS && r < ROWS - RADIUS && c >= RADIUS && c < COLS - RADIUS) {
            var sum = 0;
            for (var dr = -RADIUS; dr <= RADIUS; dr++) {
              for (var dc = -RADIUS; dc <= RADIUS; dc++) {
                sum += pixels[(r + dr) * COLS + (c + dc)];
              }
            }
            var b = sum / 9;
            var m = pixels[i] - b;
            blurred[i] = b;
            mask[i] = m;
            result[i] = Math.max(0, Math.min(255, Math.round(pixels[i] + gainK * m)));
          } else {
            blurred[i] = pixels[i];
            mask[i] = 0;
            result[i] = pixels[i];
          }
        }
      }
    }

    function highlight(tr, tc, on) {
      var cells = gridF.children;
      for (var i = 0; i < N; i++) {
        if (!cells[i]) continue;
        var p = pixels[i], r = Math.floor(i / COLS), c = i % COLS;
        if (on && Math.abs(r - tr) <= 1 && Math.abs(c - tc) <= 1) {
          if (r === tr && c === tc) {
            cells[i].style.background = '#ebf4fd';
            cells[i].style.color = '#2980b9';
            cells[i].style.boxShadow = '0 0 0 2px #2980b9 inset';
          } else {
            cells[i].style.background = '#fef5e7';
            cells[i].style.color = '#b9770e';
            cells[i].style.boxShadow = '0 0 0 2px #b9770e inset';
          }
        } else {
          cells[i].style.background = 'rgb(' + p + ',' + p + ',' + p + ')';
          cells[i].style.color = textColor(p);
          cells[i].style.boxShadow = 'none';
        }
      }
    }

    function resetDebug() {
      debug.textContent = 'Passe o mouse sobre um píxel interno do resultado para rastrear o pipeline completo.';
    }

    function render() {
      gridF.innerHTML = '';
      gridB.innerHTML = '';
      gridM.innerHTML = '';
      gridRes.innerHTML = '';
      fLabel.textContent = '④ Resultado g = f + ' + gainK.toFixed(1) + '·m';

      for (var i = 0; i < N; i++) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var p = pixels[i], b = blurred[i], m = mask[i], res = result[i];
        var isBorder = (r < RADIUS || r >= ROWS - RADIUS || c < RADIUS || c >= COLS - RADIUS);

        // Célula F
        var cf = document.createElement('div');
        cf.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + textColor(p) + ';box-sizing:border-box;';
        cf.textContent = p;
        gridF.appendChild(cf);

        // Célula Desfocado B
        var cb = document.createElement('div');
        cb.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;';
        if (isBorder) {
          cb.style.background = '#fafaf7';
          cb.style.color = '#8a8371';
          cb.style.border = '1px solid #e4dcc8';
          cb.textContent = '—';
        } else {
          var bv = Math.round(b);
          cb.style.background = 'rgb(' + bv + ',' + bv + ',' + bv + ')';
          cb.style.color = textColor(bv);
          cb.style.border = '1px solid #e4dcc8';
          cb.textContent = b.toFixed(1);
        }
        gridB.appendChild(cb);

        // Célula Máscara M
        var cm = document.createElement('div');
        cm.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;';
        if (isBorder) {
          cm.style.background = '#fafaf7';
          cm.style.color = '#8a8371';
          cm.style.border = '1px solid #e4dcc8';
          cm.textContent = '0';
        } else {
          var sg = m >= 0 ? '+' : '';
          cm.style.background = colorMask(m);
          cm.style.color = '#26241d';
          cm.style.border = '1px solid #e4dcc8';
          cm.textContent = sg + m.toFixed(1);
        }
        gridM.appendChild(cm);

        // Célula Resultado g
        var cr = document.createElement('div');
        cr.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;transition:all 0.15s ease;';

        if (isBorder) {
          cr.style.background = '#eafaf1';
          cr.style.border = '1.5px solid #27ae60';
          cr.style.color = '#27ae60';
          cr.textContent = res;

          (function(row, col, val){
            cr.addEventListener('mouseenter', function(){
              debug.innerHTML = 'Pixel borda (' + row + ',' + col + '): copiado sem alteração &nbsp;→&nbsp; <b>' + val + '</b>';
            });
            cr.addEventListener('mouseleave', function(){ resetDebug(); });
          })(r, c, res);
        } else {
          cr.style.background = 'rgb(' + res + ',' + res + ',' + res + ')';
          cr.style.color = textColor(res);
          cr.style.border = '1px solid #e4dcc8';
          cr.style.cursor = 'pointer';
          cr.textContent = res;

          (function(row, col, pVal, bVal, mVal, resVal){
            cr.addEventListener('mouseenter', function(){
              highlight(row, col, true);
              cr.style.transform = 'scale(1.12)';
              cr.style.boxShadow = '0 0 0 2px #2980b9 inset';
              cr.style.background = '#ebf4fd';
              cr.style.color = '#2980b9';

              var sg = mVal >= 0 ? '+' : '';
              var raw = pVal + gainK * mVal;
              debug.innerHTML = '① f=' + pVal + ' &nbsp;→&nbsp; ② f̄=' + bVal.toFixed(1) + ' &nbsp;→&nbsp; ③ m=f−f̄=' + sg + mVal.toFixed(1) + ' &nbsp;→&nbsp; ④ g = clip(' + pVal + ' + ' + gainK.toFixed(1) + '·(' + sg + mVal.toFixed(1) + ')) = clip(' + raw.toFixed(1) + ') = <b>' + resVal + '</b>';
            });

            cr.addEventListener('mouseleave', function(){
              highlight(row, col, false);
              cr.style.transform = 'scale(1)';
              cr.style.boxShadow = 'none';
              cr.style.background = 'rgb(' + resVal + ',' + resVal + ',' + resVal + ')';
              cr.style.color = textColor(resVal);
              resetDebug();
            });
          })(r, c, p, b, m, res);
        }
        gridRes.appendChild(cr);
      }
    }

    sliderK.addEventListener('input', function(e){
      gainK = parseFloat(e.target.value) || 0;
      kLabel.textContent = 'k = ' + gainK.toFixed(1);
      calculate();
      render();
    });

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0310(){
    var root = document.getElementById('sim-ep0310-unsharp');
    if (root) initSimEP0310(root); else setTimeout(tryInitSimEP0310, 200);
  }
  tryInitSimEP0310();
})();
</script>
</div>
""")

**Figura 3.35:** Simulador EP03_10: *Unsharp Masking* (USM)


<figure id="fig-03-sim-ep0310-unsharp">
  <img src="imagens/fig-03-sim-ep0310-unsharp.png" alt=" Simulador EP03_10: *Unsharp Masking* (USM) " style="max-width:80%" />
  <figcaption><strong>Figura 3.35:</strong>  Simulador EP03_10: *Unsharp Masking* (USM) </figcaption>
</figure>

In [112]:
%%writefile EP03_10.cpp
// sua solução

Overwriting EP03_10.cpp


In [113]:
TestSuite("EP03_10.cpp").run()

## Referências do Capítulo


A fundamentação teórica deste capítulo baseia-se nas seguintes obras:
* Gonzalez (2018) para os conceitos de operações de intensidade, histograma, convolução e filtragem espacial.
* Szeliski (2022) para a visão computacional e aplicações práticas de filtragem.
* Bradski (2008) para a implementação prática com OpenCV e `morph.py`.

BRADSKI, Gary; KAEHLER, Adrian. **Learning OpenCV: Computer vision with the OpenCV library**. " O'Reilly Media, Inc.", 2008.

GONZALEZ, R. C.; WOODS, R. E. **Digital Image Processing**. New York, Pearson, 2018.

SZELISKI, Richard. **Computer Vision: Algorithms and Applications**. Springer, 2022.